In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import json
import time
import random
import requests
import pandas as pd

from bs4 import BeautifulSoup
from pathlib import Path
from datetime import datetime
from urllib.parse import quote_plus, urljoin

BASE_DIR = Path("/content/drive/MyDrive/Calorify")

PHASE2_DIR = BASE_DIR / "phase2_text_calorie"
DATA_DIR = PHASE2_DIR / "data"
SCRAPED_RAW_DIR = DATA_DIR / "scraped_raw"
CLEAN_DIR = DATA_DIR / "cleaned"
RESULTS_DIR = PHASE2_DIR / "results"
MODELS_DIR = PHASE2_DIR / "models"
LOGS_DIR = PHASE2_DIR / "logs"

folders = [
    PHASE2_DIR,
    DATA_DIR,
    SCRAPED_RAW_DIR,
    CLEAN_DIR,
    RESULTS_DIR,
    MODELS_DIR,
    LOGS_DIR
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Phase 2 scraping folders created successfully.")
for folder in folders:
    print(folder)

Mounted at /content/drive
Phase 2 scraping folders created successfully.
/content/drive/MyDrive/Calorify/phase2_text_calorie
/content/drive/MyDrive/Calorify/phase2_text_calorie/data
/content/drive/MyDrive/Calorify/phase2_text_calorie/data/scraped_raw
/content/drive/MyDrive/Calorify/phase2_text_calorie/data/cleaned
/content/drive/MyDrive/Calorify/phase2_text_calorie/results
/content/drive/MyDrive/Calorify/phase2_text_calorie/models
/content/drive/MyDrive/Calorify/phase2_text_calorie/logs


In [2]:
# Cell 2 — Define food classes, search queries, and scraping source configuration

FOOD_CLASSES = [
    "burger",
    "pizza",
    "fries",
    "pasta",
    "rice",
    "chicken",
    "salad",
    "sandwich",
    "sushi",
    "steak"
]

SEARCH_QUERIES = {
    "burger": [
        "burger",
        "cheeseburger",
        "beef burger",
        "chicken burger",
        "turkey burger"
    ],
    "pizza": [
        "pizza",
        "pepperoni pizza",
        "cheese pizza",
        "margherita pizza",
        "vegetable pizza"
    ],
    "fries": [
        "fries",
        "french fries",
        "potato fries",
        "sweet potato fries",
        "loaded fries"
    ],
    "pasta": [
        "pasta",
        "chicken pasta",
        "spaghetti",
        "alfredo pasta",
        "tomato pasta"
    ],
    "rice": [
        "rice",
        "fried rice",
        "chicken rice",
        "rice bowl",
        "vegetable rice"
    ],
    "chicken": [
        "chicken",
        "grilled chicken",
        "fried chicken",
        "chicken breast",
        "roasted chicken"
    ],
    "salad": [
        "salad",
        "caesar salad",
        "chicken salad",
        "green salad",
        "greek salad"
    ],
    "sandwich": [
        "sandwich",
        "club sandwich",
        "chicken sandwich",
        "grilled cheese sandwich",
        "turkey sandwich"
    ],
    "sushi": [
        "sushi",
        "sushi roll",
        "california roll",
        "tuna roll",
        "salmon roll"
    ],
    "steak": [
        "steak",
        "grilled steak",
        "beef steak",
        "sirloin steak",
        "steak dinner"
    ]
}

SCRAPING_SOURCES = {
    "allrecipes": {
        "base_url": "https://www.allrecipes.com",
        "search_url_template": "https://www.allrecipes.com/search?q={query}"
    }
}

TARGET_RECORDS_PER_CLASS = 50

print("Food classes:", len(FOOD_CLASSES))
print("Target records per class:", TARGET_RECORDS_PER_CLASS)

print("\nClasses and number of queries:")
for food_class in FOOD_CLASSES:
    print(food_class, ":", len(SEARCH_QUERIES[food_class]))

print("\nScraping sources:")
for source_name, config in SCRAPING_SOURCES.items():
    print(source_name, ":", config["base_url"])

Food classes: 10
Target records per class: 50

Classes and number of queries:
burger : 5
pizza : 5
fries : 5
pasta : 5
rice : 5
chicken : 5
salad : 5
sandwich : 5
sushi : 5
steak : 5

Scraping sources:
allrecipes : https://www.allrecipes.com


In [3]:
# Cell 3 — Safe request helper and search page test

def safe_get(url, sleep_min=1, sleep_max=2, timeout=20):
    """
    Send a GET request with browser-like headers and simple delay.
    Returns the response object or None if the request fails.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }

    try:
        time.sleep(random.uniform(sleep_min, sleep_max))
        response = requests.get(url, headers=headers, timeout=timeout)
        return response

    except Exception as e:
        print("Request error:", e)
        return None


# Test one Allrecipes search page
test_query = "burger"
encoded_query = quote_plus(test_query)

test_search_url = SCRAPING_SOURCES["allrecipes"]["search_url_template"].format(
    query=encoded_query
)

print("Test search URL:")
print(test_search_url)

response = safe_get(test_search_url)

if response is None:
    print("No response returned.")
else:
    print("Status code:", response.status_code)
    print("Final URL:", response.url)
    print("HTML length:", len(response.text))
    print("First 300 characters:")
    print(response.text[:300])

Test search URL:
https://www.allrecipes.com/search?q=burger
Status code: 402
Final URL: https://www.allrecipes.com/search?q=burger
HTML length: 612
First 300 characters:
<p>
  If you are a reader experiencing an access issue, please contact
  <a href="mailto:support@people.inc">support@people.inc</a>. <br>
  To help us troubleshoot more quickly, you may include your IP address. <br>
  You can find it by visiting
  <a href="https://icanhazip.com" target="_blank" rel=


In [4]:
# Cell 4 — Test alternative recipe scraping sources

CANDIDATE_SOURCES = {
    "bbc_good_food": {
        "base_url": "https://www.bbcgoodfood.com",
        "search_url_template": "https://www.bbcgoodfood.com/search?q={query}"
    },
    "skinnytaste": {
        "base_url": "https://www.skinnytaste.com",
        "search_url_template": "https://www.skinnytaste.com/?s={query}"
    },
    "taste_of_home": {
        "base_url": "https://www.tasteofhome.com",
        "search_url_template": "https://www.tasteofhome.com/?s={query}"
    },
    "food_network": {
        "base_url": "https://www.foodnetwork.com",
        "search_url_template": "https://www.foodnetwork.com/search/{query}-"
    }
}

test_query = "burger"
encoded_query = quote_plus(test_query)

source_test_results = []

for source_name, config in CANDIDATE_SOURCES.items():
    search_url = config["search_url_template"].format(query=encoded_query)

    print("\nTesting source:", source_name)
    print("URL:", search_url)

    response = safe_get(search_url, sleep_min=1, sleep_max=2)

    if response is None:
        result = {
            "source_name": source_name,
            "status_code": None,
            "html_length": 0,
            "final_url": None,
            "works": False
        }
        print("No response.")
    else:
        result = {
            "source_name": source_name,
            "status_code": response.status_code,
            "html_length": len(response.text),
            "final_url": response.url,
            "works": response.status_code == 200 and len(response.text) > 5000
        }

        print("Status code:", response.status_code)
        print("HTML length:", len(response.text))
        print("Final URL:", response.url)
        print("Works:", result["works"])

    source_test_results.append(result)

source_test_df = pd.DataFrame(source_test_results)

print("\nSource test summary:")
display(source_test_df)# Cell 4 — Test alternative recipe scraping sources

CANDIDATE_SOURCES = {
    "bbc_good_food": {
        "base_url": "https://www.bbcgoodfood.com",
        "search_url_template": "https://www.bbcgoodfood.com/search?q={query}"
    },
    "skinnytaste": {
        "base_url": "https://www.skinnytaste.com",
        "search_url_template": "https://www.skinnytaste.com/?s={query}"
    },
    "taste_of_home": {
        "base_url": "https://www.tasteofhome.com",
        "search_url_template": "https://www.tasteofhome.com/?s={query}"
    },
    "food_network": {
        "base_url": "https://www.foodnetwork.com",
        "search_url_template": "https://www.foodnetwork.com/search/{query}-"
    }
}

test_query = "burger"
encoded_query = quote_plus(test_query)

source_test_results = []

for source_name, config in CANDIDATE_SOURCES.items():
    search_url = config["search_url_template"].format(query=encoded_query)

    print("\nTesting source:", source_name)
    print("URL:", search_url)

    response = safe_get(search_url, sleep_min=1, sleep_max=2)

    if response is None:
        result = {
            "source_name": source_name,
            "status_code": None,
            "html_length": 0,
            "final_url": None,
            "works": False
        }
        print("No response.")
    else:
        result = {
            "source_name": source_name,
            "status_code": response.status_code,
            "html_length": len(response.text),
            "final_url": response.url,
            "works": response.status_code == 200 and len(response.text) > 5000
        }

        print("Status code:", response.status_code)
        print("HTML length:", len(response.text))
        print("Final URL:", response.url)
        print("Works:", result["works"])

    source_test_results.append(result)

source_test_df = pd.DataFrame(source_test_results)

print("\nSource test summary:")
display(source_test_df)


Testing source: bbc_good_food
URL: https://www.bbcgoodfood.com/search?q=burger
Status code: 200
HTML length: 962908
Final URL: https://www.bbcgoodfood.com/search?q=burger
Works: True

Testing source: skinnytaste
URL: https://www.skinnytaste.com/?s=burger
Status code: 200
HTML length: 585691
Final URL: https://www.skinnytaste.com/?s=burger
Works: True

Testing source: taste_of_home
URL: https://www.tasteofhome.com/?s=burger
Status code: 200
HTML length: 168249
Final URL: https://www.tasteofhome.com/?s=burger
Works: True

Testing source: food_network
URL: https://www.foodnetwork.com/search/burger-
Status code: 403
HTML length: 395
Final URL: https://www.foodnetwork.com/search/burger-
Works: False

Source test summary:


,source_name,status_code,html_length,final_url,works
0,bbc_good_food,200,962908,https://www.bbcgoodfood.com/search?q=burger,True
1,skinnytaste,200,585691,https://www.skinnytaste.com/?s=burger,True
2,taste_of_home,200,168249,https://www.tasteofhome.com/?s=burger,True
3,food_network,403,395,https://www.foodnetwork.com/search/burger-,False



Testing source: bbc_good_food
URL: https://www.bbcgoodfood.com/search?q=burger
Status code: 200
HTML length: 962908
Final URL: https://www.bbcgoodfood.com/search?q=burger
Works: True

Testing source: skinnytaste
URL: https://www.skinnytaste.com/?s=burger
Status code: 200
HTML length: 585691
Final URL: https://www.skinnytaste.com/?s=burger
Works: True

Testing source: taste_of_home
URL: https://www.tasteofhome.com/?s=burger
Status code: 200
HTML length: 168249
Final URL: https://www.tasteofhome.com/?s=burger
Works: True

Testing source: food_network
URL: https://www.foodnetwork.com/search/burger-
Status code: 403
HTML length: 395
Final URL: https://www.foodnetwork.com/search/burger-
Works: False

Source test summary:


,source_name,status_code,html_length,final_url,works
0,bbc_good_food,200,962908,https://www.bbcgoodfood.com/search?q=burger,True
1,skinnytaste,200,585691,https://www.skinnytaste.com/?s=burger,True
2,taste_of_home,200,168249,https://www.tasteofhome.com/?s=burger,True
3,food_network,403,395,https://www.foodnetwork.com/search/burger-,False


In [5]:
# Cell 5 — Extract candidate recipe links from working search pages

def extract_candidate_links_from_search(source_name, query, max_links=30):
    """
    Extract candidate recipe links from a search results page.
    This is an inspection function before building the final scraper.
    """
    config = CANDIDATE_SOURCES[source_name]
    encoded_query = quote_plus(query)
    search_url = config["search_url_template"].format(query=encoded_query)

    response = safe_get(search_url, sleep_min=1, sleep_max=2)

    if response is None:
        print("No response returned.")
        return []

    print("Source:", source_name)
    print("Search URL:", search_url)
    print("Status code:", response.status_code)
    print("HTML length:", len(response.text))

    soup = BeautifulSoup(response.text, "html.parser")
    links = []

    for a_tag in soup.find_all("a", href=True):
        href = a_tag.get("href")
        text = a_tag.get_text(" ", strip=True)

        if not href:
            continue

        full_url = urljoin(config["base_url"], href)

        if source_name == "bbc_good_food":
            is_recipe_link = "/recipes/" in full_url

        elif source_name == "skinnytaste":
            is_recipe_link = (
                "skinnytaste.com" in full_url
                and "/?s=" not in full_url
                and "#" not in full_url
                and len(full_url.split("/")) >= 4
            )

        else:
            is_recipe_link = False

        if is_recipe_link:
            links.append({
                "source_name": source_name,
                "query": query,
                "title_text": text,
                "url": full_url
            })

    links_df = pd.DataFrame(links).drop_duplicates(subset=["url"])

    print("Candidate recipe links found:", len(links_df))

    if not links_df.empty:
        display(links_df.head(max_links))

    return links_df


bbc_burger_links_df = extract_candidate_links_from_search(
    source_name="bbc_good_food",
    query="burger",
    max_links=20
)

skinnytaste_burger_links_df = extract_candidate_links_from_search(
    source_name="skinnytaste",
    query="burger",
    max_links=20
)

Source: bbc_good_food
Search URL: https://www.bbcgoodfood.com/search?q=burger
Status code: 200
HTML length: 962908
Candidate recipe links found: 86


,source_name,query,title_text,url
0,bbc_good_food,burger,Easy dinner ideas,https://www.bbcgoodfood.com/recipes/collection...
1,bbc_good_food,burger,High protein recipes,https://www.bbcgoodfood.com/recipes/collection...
2,bbc_good_food,burger,Chocolate bakes,https://www.bbcgoodfood.com/recipes/collection...
3,bbc_good_food,burger,Quick pastas,https://www.bbcgoodfood.com/recipes/collection...
4,bbc_good_food,burger,May recipes,https://www.bbcgoodfood.com/recipes/collection...
5,bbc_good_food,burger,Barbecue ideas,https://www.bbcgoodfood.com/recipes/collection...
6,bbc_good_food,burger,Quick and easy,https://www.bbcgoodfood.com/recipes/collection...
8,bbc_good_food,burger,Family recipes,https://www.bbcgoodfood.com/recipes/collection...
9,bbc_good_food,burger,One-pots,https://www.bbcgoodfood.com/recipes/collection...
10,bbc_good_food,burger,Slow cooker recipes,https://www.bbcgoodfood.com/recipes/collection...


Source: skinnytaste
Search URL: https://www.skinnytaste.com/?s=burger
Status code: 200
HTML length: 585691
Candidate recipe links found: 86


,source_name,query,title_text,url
0,skinnytaste,burger,About Gina,https://www.skinnytaste.com/about/
1,skinnytaste,burger,Skinnytaste Community,https://skinnytaste.raptive.com/?utm_campaign=...
2,skinnytaste,burger,About,https://www.skinnytaste.com/join-skinnytaste-c...
3,skinnytaste,burger,Contact,https://www.skinnytaste.com/contact-me/
4,skinnytaste,burger,Book Tour,https://www.skinnytaste.com/book-tour/
5,skinnytaste,burger,My Shop,https://www.skinnytaste.com/shop/
6,skinnytaste,burger,,https://www.skinnytaste.com/
8,skinnytaste,burger,Recipes,https://www.skinnytaste.com/recipe-index/
10,skinnytaste,burger,Category Index,https://www.skinnytaste.com/category-index/
11,skinnytaste,burger,High Fiber,https://www.skinnytaste.com/high-fiber-recipes/


In [6]:
# Cell 6 — Refine candidate recipe link filtering

from urllib.parse import urlparse

def is_valid_bbc_good_food_recipe_url(url):
    """
    BBC Good Food recipe URLs usually look like:
    https://www.bbcgoodfood.com/recipes/recipe-slug

    We exclude:
    /recipes/collection/...
    /recipes/category/...
    /recipes/...
    """
    parsed = urlparse(url)
    path_parts = [part for part in parsed.path.split("/") if part]

    if len(path_parts) != 2:
        return False

    if path_parts[0] != "recipes":
        return False

    blocked_second_parts = {"collection", "category"}
    if path_parts[1] in blocked_second_parts:
        return False

    return True


def is_valid_skinnytaste_recipe_url(url, title_text, query):
    """
    Skinnytaste recipe pages are usually post URLs.
    This filter removes navigation/category/about pages and keeps URLs
    likely related to the query.
    """
    parsed = urlparse(url)
    path = parsed.path.strip("/").lower()
    title = str(title_text).strip().lower()
    query_terms = str(query).lower().split()

    if not path:
        return False

    blocked_paths = [
        "about",
        "contact-me",
        "shop",
        "book-tour",
        "recipe-index",
        "category-index",
        "recipes",
        "category",
        "smart-points",
        "high-fiber-recipes",
        "high-protein",
        "anti-inflammatory-recipes",
        "low-sodium",
        "join-skinnytaste-community"
    ]

    for blocked in blocked_paths:
        if path == blocked or path.startswith(blocked + "/"):
            return False

    if "skinnytaste.com" not in parsed.netloc:
        return False

    if len(title) < 5:
        return False

    has_query_term = any(term in title for term in query_terms)
    has_query_term_in_url = any(term in path for term in query_terms)

    return has_query_term or has_query_term_in_url


def extract_refined_recipe_links_from_search(source_name, query, max_links=30):
    """
    Extract more accurate recipe links from a search results page.
    """
    config = CANDIDATE_SOURCES[source_name]
    encoded_query = quote_plus(query)
    search_url = config["search_url_template"].format(query=encoded_query)

    response = safe_get(search_url, sleep_min=1, sleep_max=2)

    if response is None:
        print("No response returned.")
        return pd.DataFrame()

    print("Source:", source_name)
    print("Search URL:", search_url)
    print("Status code:", response.status_code)
    print("HTML length:", len(response.text))

    soup = BeautifulSoup(response.text, "html.parser")
    links = []

    for a_tag in soup.find_all("a", href=True):
        href = a_tag.get("href")
        title_text = a_tag.get_text(" ", strip=True)

        if not href:
            continue

        full_url = urljoin(config["base_url"], href)

        if source_name == "bbc_good_food":
            is_recipe_link = is_valid_bbc_good_food_recipe_url(full_url)

        elif source_name == "skinnytaste":
            is_recipe_link = is_valid_skinnytaste_recipe_url(
                url=full_url,
                title_text=title_text,
                query=query
            )

        else:
            is_recipe_link = False

        if is_recipe_link:
            links.append({
                "source_name": source_name,
                "query": query,
                "title_text": title_text,
                "url": full_url
            })

    links_df = pd.DataFrame(links)

    if not links_df.empty:
        links_df = links_df.drop_duplicates(subset=["url"]).reset_index(drop=True)

    print("Refined recipe links found:", len(links_df))

    if not links_df.empty:
        display(links_df.head(max_links))

    return links_df


bbc_burger_refined_df = extract_refined_recipe_links_from_search(
    source_name="bbc_good_food",
    query="burger",
    max_links=20
)

skinnytaste_burger_refined_df = extract_refined_recipe_links_from_search(
    source_name="skinnytaste",
    query="burger",
    max_links=20
)

Source: bbc_good_food
Search URL: https://www.bbcgoodfood.com/search?q=burger
Status code: 200
HTML length: 962908
Refined recipe links found: 24


,source_name,query,title_text,url
0,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/burger-bowl
1,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/falafel-bu...
2,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/15-minute-...
3,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/the-big-do...
4,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/next-level...
5,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/superhealt...
6,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/smash-burgers
7,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/mexican-ch...
8,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/soft-burge...
9,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/spiced-hal...


Source: skinnytaste
Search URL: https://www.skinnytaste.com/?s=burger
Status code: 200
HTML length: 585691
Refined recipe links found: 10


,source_name,query,title_text,url
0,skinnytaste,burger,Black Bean Burgers with Chipotle Mayonnaise,https://www.skinnytaste.com/spicy-black-bean-b...
1,skinnytaste,burger,Turkey Burger Recipe,https://www.skinnytaste.com/turkey-burger-recipe/
2,skinnytaste,burger,Cheeseburger Casserole,https://www.skinnytaste.com/cheeseburger-casse...
3,skinnytaste,burger,Homemade Hamburger Helper,https://www.skinnytaste.com/homemade-hamburger...
4,skinnytaste,burger,The 4 Best Burger Presses of 2024,https://www.skinnytaste.com/best-burger-presses/
5,skinnytaste,burger,Juicy Lucy (Stuffed Turkey Cheeseburger),https://www.skinnytaste.com/turkey-burgers-stu...
6,skinnytaste,burger,20 Best Burger Recipes,https://www.skinnytaste.com/burger-recipes/
7,skinnytaste,burger,Chicken Burger,https://www.skinnytaste.com/chicken-burger/
8,skinnytaste,burger,Juicy Turkey Burgers with Zucchini,https://www.skinnytaste.com/turkey-burgers-wit...
9,skinnytaste,burger,Cheeseburger Crunch Wrap,https://www.skinnytaste.com/cheeseburger-crunc...


In [7]:
# Cell 7 — Scrape one BBC Good Food recipe page

def extract_number(value):
    """
    Extract the first numeric value from a text field.
    Examples:
    "709 kcal" -> 709
    "23.5g" -> 23.5
    """
    if value is None:
        return None

    text = str(value)
    match = re.search(r"(\d+(?:\.\d+)?)", text)

    if match:
        return float(match.group(1))

    return None


def is_recipe_type(item):
    """
    Check whether a JSON-LD item is a Recipe object.
    """
    item_type = item.get("@type")

    if isinstance(item_type, list):
        return "Recipe" in item_type

    return item_type == "Recipe"


def find_recipe_json_ld(soup):
    """
    Find Recipe JSON-LD data inside a recipe page.
    """
    scripts = soup.find_all("script", type="application/ld+json")

    for script in scripts:
        raw_json = script.string or script.get_text(strip=True)

        if not raw_json:
            continue

        try:
            data = json.loads(raw_json)
        except Exception:
            continue

        items_to_check = []

        if isinstance(data, dict):
            items_to_check.append(data)

            if "@graph" in data and isinstance(data["@graph"], list):
                items_to_check.extend(data["@graph"])

        elif isinstance(data, list):
            items_to_check.extend(data)

        for item in items_to_check:
            if isinstance(item, dict) and is_recipe_type(item):
                return item

    return None


def scrape_bbc_good_food_recipe(url, food_class=None, query=None):
    """
    Scrape one BBC Good Food recipe page.
    """
    response = safe_get(url, sleep_min=1, sleep_max=2)

    if response is None:
        return None

    if response.status_code != 200:
        print("Failed URL:", url)
        print("Status code:", response.status_code)
        return None

    soup = BeautifulSoup(response.text, "html.parser")
    recipe_json = find_recipe_json_ld(soup)

    if recipe_json is None:
        print("No Recipe JSON-LD found for:", url)
        return None

    recipe_name = recipe_json.get("name")

    ingredients = recipe_json.get("recipeIngredient", [])
    if isinstance(ingredients, list):
        ingredients_text = " ".join([str(item) for item in ingredients])
    else:
        ingredients_text = str(ingredients)

    nutrition = recipe_json.get("nutrition", {})
    if not isinstance(nutrition, dict):
        nutrition = {}

    calories = extract_number(nutrition.get("calories"))
    protein = extract_number(nutrition.get("proteinContent"))
    fat = extract_number(nutrition.get("fatContent"))
    carbs = extract_number(nutrition.get("carbohydrateContent"))

    record = {
        "source_site": "bbc_good_food",
        "food_class": food_class,
        "query": query,
        "recipe_name": recipe_name,
        "ingredients_text": ingredients_text,
        "calories": calories,
        "protein": protein,
        "fat": fat,
        "carbs": carbs,
        "source_url": url,
        "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return record


# Test on first BBC burger recipe link
test_bbc_url = bbc_burger_refined_df.iloc[0]["url"]

print("Testing BBC recipe URL:")
print(test_bbc_url)

test_bbc_record = scrape_bbc_good_food_recipe(
    url=test_bbc_url,
    food_class="burger",
    query="burger"
)

print("\nScraped record:")
print(test_bbc_record)

if test_bbc_record is not None:
    print("\nIngredients length:", len(test_bbc_record["ingredients_text"]))
    print("Calories:", test_bbc_record["calories"])
    print("Protein:", test_bbc_record["protein"])
    print("Fat:", test_bbc_record["fat"])
    print("Carbs:", test_bbc_record["carbs"])

Testing BBC recipe URL:
https://www.bbcgoodfood.com/recipes/burger-bowl

Scraped record:
{'source_site': 'bbc_good_food', 'food_class': 'burger', 'query': 'burger', 'recipe_name': 'Burger bowl', 'ingredients_text': '2 large potatoes cut into rough chunks 2 tbsp vegetable oil 2 tbsp Cajun spice mix 1 tsp garlic granules (optional) 250g lean beef mince 25g crinkle cut pickles plus 1 tbsp brine from the jar 1 Iceberg lettuce shredded 1 red onion onion, finely sliced into rings and rinsed in cold water, if you like (this will reduce the strong raw onion flavour) 150g cherry tomatoes halved 50g cheddar grated 3 tbsp low-sugar ketchup 2 tbsp light mayonnaise ½ tbsp American mustard', 'calories': 207.0, 'protein': 18.0, 'fat': 13.0, 'carbs': 19.0, 'source_url': 'https://www.bbcgoodfood.com/recipes/burger-bowl', 'scraped_at': '2026-05-11 23:58:00'}

Ingredients length: 468
Calories: 207.0
Protein: 18.0
Fat: 13.0
Carbs: 19.0


In [8]:
# Cell 8 — Test scraping multiple BBC Good Food recipe pages

def scrape_multiple_bbc_links(links_df, food_class, max_pages=10):
    """
    Scrape multiple BBC Good Food recipe pages from a links dataframe.
    This is a small test before building the full crawler.
    """
    records = []
    failed_urls = []

    selected_links = links_df.head(max_pages)

    print("Testing multiple BBC recipe pages")
    print("Food class:", food_class)
    print("Pages to test:", len(selected_links))

    for idx, row in selected_links.iterrows():
        url = row["url"]
        query = row["query"]

        print("\nScraping page:", idx)
        print("URL:", url)

        record = scrape_bbc_good_food_recipe(
            url=url,
            food_class=food_class,
            query=query
        )

        if record is None:
            failed_urls.append(url)
            print("Result: failed")
            continue

        has_required_fields = (
            record["recipe_name"] is not None
            and record["ingredients_text"] is not None
            and len(record["ingredients_text"]) > 0
            and record["calories"] is not None
        )

        if has_required_fields:
            records.append(record)
            print("Result: success")
            print("Recipe:", record["recipe_name"])
            print("Calories:", record["calories"])
        else:
            failed_urls.append(url)
            print("Result: missing required fields")
            print("Recipe:", record["recipe_name"])
            print("Calories:", record["calories"])
            print("Ingredients length:", len(record["ingredients_text"]) if record["ingredients_text"] else 0)

    records_df = pd.DataFrame(records)

    print("\nMultiple page scraping test finished.")
    print("Successful records:", len(records))
    print("Failed pages:", len(failed_urls))

    if not records_df.empty:
        display(records_df[[
            "food_class",
            "query",
            "recipe_name",
            "calories",
            "protein",
            "fat",
            "carbs",
            "source_url"
        ]])

    return records_df, failed_urls


bbc_burger_test_records_df, bbc_burger_failed_urls = scrape_multiple_bbc_links(
    links_df=bbc_burger_refined_df,
    food_class="burger",
    max_pages=10
)

Testing multiple BBC recipe pages
Food class: burger
Pages to test: 10

Scraping page: 0
URL: https://www.bbcgoodfood.com/recipes/burger-bowl
Result: success
Recipe: Burger bowl
Calories: 207.0

Scraping page: 1
URL: https://www.bbcgoodfood.com/recipes/falafel-burgers-0
Result: success
Recipe: Falafel burgers
Calories: 175.0

Scraping page: 2
URL: https://www.bbcgoodfood.com/recipes/15-minute-chicken-halloumi-burgers
Result: success
Recipe: 15-minute chicken & halloumi burgers
Calories: 737.0

Scraping page: 3
URL: https://www.bbcgoodfood.com/recipes/the-big-double-cheeseburger-secret-sauce
Result: success
Recipe: The ultimate beef burger
Calories: 893.0

Scraping page: 4
URL: https://www.bbcgoodfood.com/recipes/next-level-chicken-burgers
Result: success
Recipe: Next level chicken burgers
Calories: 794.0

Scraping page: 5
URL: https://www.bbcgoodfood.com/recipes/superhealthy-salmon-burgers
Result: success
Recipe: Superhealthy salmon burgers
Calories: 292.0

Scraping page: 6
URL: https:

,food_class,query,recipe_name,calories,protein,fat,carbs,source_url
0,burger,burger,Burger bowl,207.0,18.0,13.0,19.0,https://www.bbcgoodfood.com/recipes/burger-bowl
1,burger,burger,Falafel burgers,175.0,6.0,8.0,18.0,https://www.bbcgoodfood.com/recipes/falafel-bu...
2,burger,burger,15-minute chicken & halloumi burgers,737.0,39.0,42.0,49.0,https://www.bbcgoodfood.com/recipes/15-minute-...
3,burger,burger,The ultimate beef burger,893.0,56.0,37.0,82.0,https://www.bbcgoodfood.com/recipes/the-big-do...
4,burger,burger,Next level chicken burgers,794.0,33.0,33.0,90.0,https://www.bbcgoodfood.com/recipes/next-level...
5,burger,burger,Superhealthy salmon burgers,292.0,29.0,17.0,7.0,https://www.bbcgoodfood.com/recipes/superhealt...
6,burger,burger,Smash burgers,666.0,37.0,39.0,41.0,https://www.bbcgoodfood.com/recipes/smash-burgers
7,burger,burger,Mexican chicken burger,709.0,46.0,34.0,52.0,https://www.bbcgoodfood.com/recipes/mexican-ch...
8,burger,burger,Soft burger buns,210.0,6.0,5.0,35.0,https://www.bbcgoodfood.com/recipes/soft-burge...
9,burger,burger,Spiced halloumi & pineapple burger with zingy ...,264.0,11.0,14.0,19.0,https://www.bbcgoodfood.com/recipes/spiced-hal...


In [9]:
# Cell 9 — Test BBC Good Food recipe link extraction for all classes

bbc_link_test_rows = []
bbc_all_class_links = {}

for food_class in FOOD_CLASSES:
    print("\n====================================")
    print("Testing BBC links for class:", food_class)
    print("====================================")

    class_links = []

    for query in SEARCH_QUERIES[food_class]:
        print("\nQuery:", query)

        links_df = extract_refined_recipe_links_from_search(
            source_name="bbc_good_food",
            query=query,
            max_links=10
        )

        if links_df is not None and not links_df.empty:
            class_links.append(links_df)

        time.sleep(1)

    if class_links:
        class_links_df = pd.concat(class_links, ignore_index=True)
        class_links_df = class_links_df.drop_duplicates(subset=["url"]).reset_index(drop=True)
    else:
        class_links_df = pd.DataFrame(columns=["source_name", "query", "title_text", "url"])

    bbc_all_class_links[food_class] = class_links_df

    link_count = len(class_links_df)

    bbc_link_test_rows.append({
        "food_class": food_class,
        "unique_links_found": link_count
    })

    print("\nFinished class:", food_class)
    print("Unique BBC recipe links found:", link_count)

bbc_link_test_df = pd.DataFrame(bbc_link_test_rows)

print("\nBBC link extraction test summary:")
display(bbc_link_test_df)

print("\nClasses with fewer than 50 links:")
display(bbc_link_test_df[bbc_link_test_df["unique_links_found"] < TARGET_RECORDS_PER_CLASS])


Testing BBC links for class: burger

Query: burger
Source: bbc_good_food
Search URL: https://www.bbcgoodfood.com/search?q=burger
Status code: 200
HTML length: 962908
Refined recipe links found: 24


,source_name,query,title_text,url
0,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/burger-bowl
1,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/falafel-bu...
2,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/15-minute-...
3,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/the-big-do...
4,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/next-level...
5,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/superhealt...
6,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/smash-burgers
7,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/mexican-ch...
8,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/soft-burge...
9,bbc_good_food,burger,,https://www.bbcgoodfood.com/recipes/spiced-hal...



Query: cheeseburger
Source: bbc_good_food
Search URL: https://www.bbcgoodfood.com/search?q=cheeseburger
Status code: 200
HTML length: 660375
Refined recipe links found: 13


,source_name,query,title_text,url
0,bbc_good_food,cheeseburger,,https://www.bbcgoodfood.com/recipes/one-pot-ch...
1,bbc_good_food,cheeseburger,,https://www.bbcgoodfood.com/recipes/nacho-chee...
2,bbc_good_food,cheeseburger,,https://www.bbcgoodfood.com/recipes/cheeseburg...
3,bbc_good_food,cheeseburger,,https://www.bbcgoodfood.com/recipes/cheeseburg...
4,bbc_good_food,cheeseburger,,https://www.bbcgoodfood.com/recipes/veg-packed...
5,bbc_good_food,cheeseburger,,https://www.bbcgoodfood.com/recipes/cheeseburg...
6,bbc_good_food,cheeseburger,,https://www.bbcgoodfood.com/recipes/french-oni...
7,bbc_good_food,cheeseburger,,https://www.bbcgoodfood.com/recipes/cheeseburgers
8,bbc_good_food,cheeseburger,,https://www.bbcgoodfood.com/recipes/pork-thyme...
9,bbc_good_food,cheeseburger,,https://www.bbcgoodfood.com/recipes/the-big-do...



Query: beef burger
Source: bbc_good_food
Search URL: https://www.bbcgoodfood.com/search?q=beef+burger
Status code: 200
HTML length: 955511
Refined recipe links found: 28


...

,source_name,query,title_text,url
0,bbc_good_food,steak dinner,,https://www.bbcgoodfood.com/recipes/easy-steak...
1,bbc_good_food,steak dinner,,https://www.bbcgoodfood.com/recipes/steak-ale-pie
2,bbc_good_food,steak dinner,,https://www.bbcgoodfood.com/recipes/sesame-ste...
3,bbc_good_food,steak dinner,,https://www.bbcgoodfood.com/recipes/next-level...
4,bbc_good_food,steak dinner,,https://www.bbcgoodfood.com/recipes/one-pan-si...
5,bbc_good_food,steak dinner,,https://www.bbcgoodfood.com/recipes/cauliflowe...
6,bbc_good_food,steak dinner,,https://www.bbcgoodfood.com/recipes/air-fryer-...
7,bbc_good_food,steak dinner,,https://www.bbcgoodfood.com/recipes/smoky-stea...
8,bbc_good_food,steak dinner,,https://www.bbcgoodfood.com/recipes/rump-steak
9,bbc_good_food,steak dinner,,https://www.bbcgoodfood.com/recipes/proper-bee...



Finished class: steak
Unique BBC recipe links found: 64

BBC link extraction test summary:


,food_class,unique_links_found
0,burger,90
1,pizza,69
2,fries,72
3,pasta,101
4,rice,81
5,chicken,110
6,salad,101
7,sandwich,107
8,sushi,92
9,steak,64



Classes with fewer than 50 links:


,food_class,unique_links_found


In [10]:
# Cell 10 — Scrape BBC pilot records for all classes

def scrape_bbc_records_for_class(food_class, links_df, target_records=10):
    """
    Scrape valid BBC Good Food recipe records for one class.
    """
    records = []
    failed_urls = []

    print("Scraping class:", food_class)
    print("Available links:", len(links_df))
    print("Target records:", target_records)

    for idx, row in links_df.iterrows():
        if len(records) >= target_records:
            break

        url = row["url"]
        query = row["query"]

        print("\nScraping URL:")
        print(url)

        record = scrape_bbc_good_food_recipe(
            url=url,
            food_class=food_class,
            query=query
        )

        if record is None:
            failed_urls.append(url)
            print("Result: failed")
            continue

        ingredients_text = record.get("ingredients_text")
        calories = record.get("calories")

        has_required_fields = (
            record.get("recipe_name") is not None
            and ingredients_text is not None
            and len(ingredients_text) > 0
            and calories is not None
        )

        if has_required_fields:
            records.append(record)
            print("Result: success")
            print("Recipe:", record["recipe_name"])
            print("Calories:", record["calories"])
        else:
            failed_urls.append(url)
            print("Result: missing required fields")
            print("Recipe:", record.get("recipe_name"))
            print("Calories:", record.get("calories"))

    return records, failed_urls


all_bbc_pilot_records = []
all_bbc_pilot_failed_urls = []

for food_class in FOOD_CLASSES:
    print("\n====================================")
    print("Pilot scraping for class:", food_class)
    print("====================================")

    links_df = bbc_all_class_links[food_class]

    class_records, class_failed_urls = scrape_bbc_records_for_class(
        food_class=food_class,
        links_df=links_df,
        target_records=10
    )

    all_bbc_pilot_records.extend(class_records)

    for failed_url in class_failed_urls:
        all_bbc_pilot_failed_urls.append({
            "food_class": food_class,
            "failed_url": failed_url
        })

bbc_pilot_df = pd.DataFrame(all_bbc_pilot_records)
bbc_pilot_failed_df = pd.DataFrame(all_bbc_pilot_failed_urls)

print("\nBBC pilot scraping finished.")
print("Pilot dataset shape:", bbc_pilot_df.shape)

print("\nRecords per class:")
print(bbc_pilot_df["food_class"].value_counts())

print("\nFailed URLs:", len(bbc_pilot_failed_df))

bbc_pilot_output_path = SCRAPED_RAW_DIR / "bbc_text_calorie_pilot_raw.csv"
bbc_pilot_failed_path = LOGS_DIR / "bbc_pilot_failed_urls.csv"

bbc_pilot_df.to_csv(bbc_pilot_output_path, index=False)
bbc_pilot_failed_df.to_csv(bbc_pilot_failed_path, index=False)

print("\nPilot raw dataset saved to:")
print(bbc_pilot_output_path)

print("\nFailed URLs log saved to:")
print(bbc_pilot_failed_path)

display(bbc_pilot_df.head(10))


Pilot scraping for class: burger
Scraping class: burger
Available links: 90
Target records: 10

Scraping URL:
https://www.bbcgoodfood.com/recipes/burger-bowl
Result: success
Recipe: Burger bowl
Calories: 207.0

Scraping URL:
https://www.bbcgoodfood.com/recipes/falafel-burgers-0
Result: success
Recipe: Falafel burgers
Calories: 175.0

Scraping URL:
https://www.bbcgoodfood.com/recipes/15-minute-chicken-halloumi-burgers
Result: success
Recipe: 15-minute chicken & halloumi burgers
Calories: 737.0

Scraping URL:
https://www.bbcgoodfood.com/recipes/the-big-double-cheeseburger-secret-sauce
Result: success
Recipe: The ultimate beef burger
Calories: 893.0

Scraping URL:
https://www.bbcgoodfood.com/recipes/next-level-chicken-burgers
Result: success
Recipe: Next level chicken burgers
Calories: 794.0

Scraping URL:
https://www.bbcgoodfood.com/recipes/superhealthy-salmon-burgers
Result: success
Recipe: Superhealthy salmon burgers
Calories: 292.0

Scraping URL:
https://www.bbcgoodfood.com/recipes/s

,source_site,food_class,query,recipe_name,ingredients_text,calories,protein,fat,carbs,source_url,scraped_at
0,bbc_good_food,burger,burger,Burger bowl,2 large potatoes cut into rough chunks 2 tbsp ...,207.0,18.0,13.0,19.0,https://www.bbcgoodfood.com/recipes/burger-bowl,2026-05-12 00:11:07
1,bbc_good_food,burger,burger,Falafel burgers,400g can chickpeas rinsed and drained 1 small...,175.0,6.0,8.0,18.0,https://www.bbcgoodfood.com/recipes/falafel-bu...,2026-05-12 00:11:08
2,bbc_good_food,burger,burger,15-minute chicken & halloumi burgers,2 chicken breast fillets 1 tbsp oil plus extra...,737.0,39.0,42.0,49.0,https://www.bbcgoodfood.com/recipes/15-minute-...,2026-05-12 00:11:10
3,bbc_good_food,burger,burger,The ultimate beef burger,1 small onion finely chopped 4 sesame-topped ...,893.0,56.0,37.0,82.0,https://www.bbcgoodfood.com/recipes/the-big-do...,2026-05-12 00:11:11
4,bbc_good_food,burger,burger,Next level chicken burgers,2 large chicken breasts sunflower oil for deep...,794.0,33.0,33.0,90.0,https://www.bbcgoodfood.com/recipes/next-level...,2026-05-12 00:11:13
5,bbc_good_food,burger,burger,Superhealthy salmon burgers,"4 boneless, skinless salmon fillets about 550g...",292.0,29.0,17.0,7.0,https://www.bbcgoodfood.com/recipes/superhealt...,2026-05-12 00:11:15
6,bbc_good_food,burger,burger,Smash burgers,"4 burger buns sesame topped or brioche, whiche...",666.0,37.0,39.0,41.0,https://www.bbcgoodfood.com/recipes/smash-burgers,2026-05-12 00:11:17
7,bbc_good_food,burger,burger,Mexican chicken burger,1 chicken breast 1 tsp chipotle paste 1 lime j...,709.0,46.0,34.0,52.0,https://www.bbcgoodfood.com/recipes/mexican-ch...,2026-05-12 00:11:19
8,bbc_good_food,burger,burger,Soft burger buns,200ml whole milk plus extra for brushing 50g u...,210.0,6.0,5.0,35.0,https://www.bbcgoodfood.com/recipes/soft-burge...,2026-05-12 00:11:20
9,bbc_good_food,burger,burger,Spiced halloumi & pineapple burger with zingy ...,½ red cabbage grated 2 carrots grated 100g rad...,264.0,11.0,14.0,19.0,https://www.bbcgoodfood.com/recipes/spiced-hal...,2026-05-12 00:11:22


In [11]:
# Cell 11 — Relevance filtering rules for scraped recipe records

CLASS_KEYWORDS = {
    "burger": [
        "burger", "cheeseburger", "hamburger"
    ],
    "pizza": [
        "pizza", "pizzas"
    ],
    "fries": [
        "fries", "french fries", "chips", "loaded fries", "sweet potato fries"
    ],
    "pasta": [
        "pasta", "spaghetti", "alfredo", "penne", "lasagne", "lasagna",
        "tagliatelle", "fettuccine", "macaroni", "orzo"
    ],
    "rice": [
        "rice", "risotto", "paella", "pilau", "biryani", "jambalaya"
    ],
    "chicken": [
        "chicken"
    ],
    "salad": [
        "salad", "slaw"
    ],
    "sandwich": [
        "sandwich", "club sandwich", "toastie", "wrap", "bagel", "panini"
    ],
    "sushi": [
        "sushi", "maki", "nigiri", "sashimi", "sushi bowl", "sushi rice"
    ],
    "steak": [
        "steak", "sirloin", "ribeye", "beef steak"
    ]
}

CLASS_NEGATIVE_KEYWORDS = {
    "fries": [
        "fried rice", "fried chicken", "pan-fried", "stir-fried", "egg-fried rice"
    ],
    "sandwich": [
        "victoria sandwich", "sponge sandwich", "cake"
    ],
    "pizza": [
        "pizza sauce", "pizza dough"
    ],
    "steak": [
        "cauliflower steak", "cauliflower steaks"
    ],
    "sushi": [
        "bread roll", "sausage roll", "cinnamon roll"
    ]
}


def normalize_text_for_filtering(text):
    """
    Normalize text before relevance filtering.
    """
    if text is None:
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def is_relevant_record(record):
    """
    Decide whether a scraped record is relevant to its assigned food_class.
    Uses recipe name, source URL, and query text.
    """
    food_class = record.get("food_class")
    recipe_name = normalize_text_for_filtering(record.get("recipe_name"))
    source_url = normalize_text_for_filtering(record.get("source_url"))
    query = normalize_text_for_filtering(record.get("query"))

    combined_text = " ".join([recipe_name, source_url, query])

    positive_keywords = CLASS_KEYWORDS.get(food_class, [])
    negative_keywords = CLASS_NEGATIVE_KEYWORDS.get(food_class, [])

    for neg_kw in negative_keywords:
        if neg_kw in combined_text:
            return False

    for pos_kw in positive_keywords:
        if pos_kw in combined_text:
            return True

    return False


# Test relevance filtering on pilot dataset
bbc_pilot_df["is_relevant"] = bbc_pilot_df.apply(
    lambda row: is_relevant_record(row.to_dict()),
    axis=1
)

print("Pilot relevance summary:")
print(bbc_pilot_df.groupby("food_class")["is_relevant"].value_counts())

print("\nRejected pilot records:")
rejected_pilot_df = bbc_pilot_df[bbc_pilot_df["is_relevant"] == False]
display(rejected_pilot_df[[
    "food_class",
    "query",
    "recipe_name",
    "calories",
    "source_url"
]])

Pilot relevance summary:
food_class  is_relevant
burger      True           10
chicken     True           10
fries       False          10
pasta       True           10
pizza       True            6
            False           4
rice        True           10
salad       True           10
sandwich    True            8
            False           2
steak       True            9
            False           1
sushi       True           10
Name: count, dtype: int64

Rejected pilot records:


,food_class,query,recipe_name,calories,source_url
10,pizza,pizza,Pizza sauce,120.0,https://www.bbcgoodfood.com/recipes/pizza-sauce
11,pizza,pizza,Quick pizza dough,656.0,https://www.bbcgoodfood.com/recipes/quick-pizz...
13,pizza,pizza,Pizza dough,504.0,https://www.bbcgoodfood.com/recipes/basic-pizz...
14,pizza,pizza,No yeast pizza dough,352.0,https://www.bbcgoodfood.com/recipes/no-yeast-p...
20,fries,fries,Korean fried chicken,487.0,https://www.bbcgoodfood.com/recipes/korean-fri...
21,fries,fries,Easy egg-fried rice,387.0,https://www.bbcgoodfood.com/recipes/egg-fried-...
22,fries,fries,Next level fried chicken,556.0,https://www.bbcgoodfood.com/recipes/next-level...
23,fries,fries,Prawn fried rice,418.0,https://www.bbcgoodfood.com/recipes/prawn-frie...
24,fries,fries,Chorizo egg-fried rice,325.0,https://www.bbcgoodfood.com/recipes/chorizo-eg...
25,fries,fries,Chicken & ginger fried rice,535.0,https://www.bbcgoodfood.com/recipes/chicken-gi...


In [12]:
# Cell 12 — Improve relevance filtering and fries search queries

# Replace weak fries queries with more precise UK/US terms
SEARCH_QUERIES["fries"] = [
    "french fries",
    "loaded fries",
    "sweet potato fries",
    "air fryer chips",
    "homemade chips",
    "potato wedges",
    "oven chips",
    "skinny fries"
]

# Update positive keywords for fries
CLASS_KEYWORDS["fries"] = [
    "fries",
    "french fries",
    "loaded fries",
    "sweet potato fries",
    "chips",
    "oven chips",
    "air fryer chips",
    "potato wedges",
    "wedges"
]

# Strengthen negative keywords
CLASS_NEGATIVE_KEYWORDS["fries"] = [
    "fried rice",
    "egg-fried rice",
    "prawn fried rice",
    "chorizo egg-fried rice",
    "chicken ginger fried rice",
    "pineapple fried rice",
    "salmon egg-fried rice",
    "fried chicken",
    "korean fried chicken",
    "next level fried chicken",
    "pan-fried",
    "stir-fried"
]

CLASS_NEGATIVE_KEYWORDS["pizza"] = [
    "pizza sauce",
    "pizza dough",
    "no yeast pizza dough",
    "basic pizza dough",
    "quick pizza dough"
]

CLASS_NEGATIVE_KEYWORDS["sandwich"] = [
    "victoria sandwich",
    "sponge sandwich",
    "cake"
]

CLASS_NEGATIVE_KEYWORDS["steak"] = [
    "cauliflower steak",
    "cauliflower steaks"
]


def is_relevant_record_v2(record):
    """
    Decide whether a scraped record is relevant to its assigned food_class.

    Important:
    This version does NOT use the query text for positive matching,
    because the query can make irrelevant results look relevant.
    """
    food_class = record.get("food_class")
    recipe_name = normalize_text_for_filtering(record.get("recipe_name"))
    source_url = normalize_text_for_filtering(record.get("source_url"))

    combined_text = " ".join([recipe_name, source_url])

    positive_keywords = CLASS_KEYWORDS.get(food_class, [])
    negative_keywords = CLASS_NEGATIVE_KEYWORDS.get(food_class, [])

    for neg_kw in negative_keywords:
        if neg_kw in combined_text:
            return False

    for pos_kw in positive_keywords:
        if pos_kw in combined_text:
            return True

    return False


# Apply improved filter on pilot dataset
bbc_pilot_df["is_relevant_v2"] = bbc_pilot_df.apply(
    lambda row: is_relevant_record_v2(row.to_dict()),
    axis=1
)

print("Pilot relevance summary with v2:")
print(bbc_pilot_df.groupby("food_class")["is_relevant_v2"].value_counts())

print("\nRejected pilot records with v2:")
rejected_pilot_v2_df = bbc_pilot_df[bbc_pilot_df["is_relevant_v2"] == False]
display(rejected_pilot_v2_df[[
    "food_class",
    "query",
    "recipe_name",
    "calories",
    "source_url"
]])

print("\nUpdated fries queries:")
for query in SEARCH_QUERIES["fries"]:
    print("-", query)

Pilot relevance summary with v2:
food_class  is_relevant_v2
burger      True              10
chicken     True              10
fries       False             10
pasta       True              10
pizza       True               6
            False              4
rice        True              10
salad       True              10
sandwich    True               8
            False              2
steak       True               9
            False              1
sushi       True              10
Name: count, dtype: int64

Rejected pilot records with v2:


,food_class,query,recipe_name,calories,source_url
10,pizza,pizza,Pizza sauce,120.0,https://www.bbcgoodfood.com/recipes/pizza-sauce
11,pizza,pizza,Quick pizza dough,656.0,https://www.bbcgoodfood.com/recipes/quick-pizz...
13,pizza,pizza,Pizza dough,504.0,https://www.bbcgoodfood.com/recipes/basic-pizz...
14,pizza,pizza,No yeast pizza dough,352.0,https://www.bbcgoodfood.com/recipes/no-yeast-p...
20,fries,fries,Korean fried chicken,487.0,https://www.bbcgoodfood.com/recipes/korean-fri...
21,fries,fries,Easy egg-fried rice,387.0,https://www.bbcgoodfood.com/recipes/egg-fried-...
22,fries,fries,Next level fried chicken,556.0,https://www.bbcgoodfood.com/recipes/next-level...
23,fries,fries,Prawn fried rice,418.0,https://www.bbcgoodfood.com/recipes/prawn-frie...
24,fries,fries,Chorizo egg-fried rice,325.0,https://www.bbcgoodfood.com/recipes/chorizo-eg...
25,fries,fries,Chicken & ginger fried rice,535.0,https://www.bbcgoodfood.com/recipes/chicken-gi...



Updated fries queries:
- french fries
- loaded fries
- sweet potato fries
- air fryer chips
- homemade chips
- potato wedges
- oven chips
- skinny fries


In [14]:
# Cell 13 — Re-test BBC Good Food links for fries only using improved queries

print("Testing updated BBC links for fries")

fries_links_list = []

for query in SEARCH_QUERIES["fries"]:
    print("\nQuery:", query)

    links_df = extract_refined_recipe_links_from_search(
        source_name="bbc_good_food",
        query=query,
        max_links=15
    )

    if links_df is not None and not links_df.empty:
        fries_links_list.append(links_df)

    time.sleep(1)

if fries_links_list:
    fries_links_updated_df = pd.concat(fries_links_list, ignore_index=True)
    fries_links_updated_df = fries_links_updated_df.drop_duplicates(subset=["url"]).reset_index(drop=True)
else:
    fries_links_updated_df = pd.DataFrame(columns=["source_name", "query", "title_text", "url"])

print("\nUpdated fries links finished.")
print("Unique fries links found:", len(fries_links_updated_df))

display(fries_links_updated_df.head(30))

Testing updated BBC links for fries

Query: french fries
Source: bbc_good_food
Search URL: https://www.bbcgoodfood.com/search?q=french+fries
Status code: 200
HTML length: 978393
Refined recipe links found: 27


,source_name,query,title_text,url
0,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/air-fryer-...
1,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/french-fries
2,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/korean-fri...
3,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/egg-fried-...
4,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/french-oni...
5,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/beef-strog...
6,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/next-level...
7,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/meatball-m...
8,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/greek-load...
9,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/french-toast



Query: loaded fries
Source: bbc_good_food
Search URL: https://www.bbcgoodfood.com/search?q=loaded+fries
Status code: 200
HTML length: 975665
Refined recipe links found: 25


,source_name,query,title_text,url
0,bbc_good_food,loaded fries,,https://www.bbcgoodfood.com/recipes/loaded-fries
1,bbc_good_food,loaded fries,,https://www.bbcgoodfood.com/recipes/greek-load...
2,bbc_good_food,loaded fries,,https://www.bbcgoodfood.com/recipes/korean-fri...
3,bbc_good_food,loaded fries,,https://www.bbcgoodfood.com/recipes/egg-fried-...
4,bbc_good_food,loaded fries,,https://www.bbcgoodfood.com/recipes/next-level...
5,bbc_good_food,loaded fries,,https://www.bbcgoodfood.com/recipes/meatball-m...
6,bbc_good_food,loaded fries,,https://www.bbcgoodfood.com/recipes/prawn-frie...
7,bbc_good_food,loaded fries,,https://www.bbcgoodfood.com/recipes/chorizo-eg...
8,bbc_good_food,loaded fries,,https://www.bbcgoodfood.com/recipes/pineapple-...
9,bbc_good_food,loaded fries,,https://www.bbcgoodfood.com/recipes/chicken-gi...



Query: sweet potato fries
Source: bbc_good_food
Search URL: https://www.bbcgoodfood.com/search?q=sweet+potato+fries
Status code: 200
HTML length: 1014743
Refined recipe links found: 27


...

,source_name,query,title_text,url
0,bbc_good_food,oven chips,,https://www.bbcgoodfood.com/recipes/chunky-ove...
1,bbc_good_food,oven chips,,https://www.bbcgoodfood.com/recipes/indian-ove...
2,bbc_good_food,oven chips,,https://www.bbcgoodfood.com/recipes/ultimate-o...
3,bbc_good_food,oven chips,,https://www.bbcgoodfood.com/recipes/oven-roast...
4,bbc_good_food,oven chips,,https://www.bbcgoodfood.com/recipes/celeriac-o...
5,bbc_good_food,oven chips,,https://www.bbcgoodfood.com/recipes/quick-roas...
6,bbc_good_food,oven chips,,https://www.bbcgoodfood.com/recipes/vintage-ch...
7,bbc_good_food,oven chips,,https://www.bbcgoodfood.com/recipes/lighter-la...
8,bbc_good_food,oven chips,,https://www.bbcgoodfood.com/recipes/oven-baked...
9,bbc_good_food,oven chips,,https://www.bbcgoodfood.com/recipes/brilliant-...



Query: skinny fries
Source: bbc_good_food
Search URL: https://www.bbcgoodfood.com/search?q=skinny+fries
Status code: 200
HTML length: 974361
Refined recipe links found: 24


,source_name,query,title_text,url
0,bbc_good_food,skinny fries,,https://www.bbcgoodfood.com/recipes/baked-skin...
1,bbc_good_food,skinny fries,,https://www.bbcgoodfood.com/recipes/korean-fri...
2,bbc_good_food,skinny fries,,https://www.bbcgoodfood.com/recipes/egg-fried-...
3,bbc_good_food,skinny fries,,https://www.bbcgoodfood.com/recipes/next-level...
4,bbc_good_food,skinny fries,,https://www.bbcgoodfood.com/recipes/prawn-frie...
5,bbc_good_food,skinny fries,,https://www.bbcgoodfood.com/recipes/chorizo-eg...
6,bbc_good_food,skinny fries,,https://www.bbcgoodfood.com/recipes/pineapple-...
7,bbc_good_food,skinny fries,,https://www.bbcgoodfood.com/recipes/chicken-gi...
8,bbc_good_food,skinny fries,,https://www.bbcgoodfood.com/recipes/chicken-ch...
9,bbc_good_food,skinny fries,,https://www.bbcgoodfood.com/recipes/pan-fried-...



Updated fries links finished.
Unique fries links found: 151


,source_name,query,title_text,url
0,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/air-fryer-...
1,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/french-fries
2,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/korean-fri...
3,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/egg-fried-...
4,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/french-oni...
5,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/beef-strog...
6,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/next-level...
7,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/meatball-m...
8,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/greek-load...
9,bbc_good_food,french fries,,https://www.bbcgoodfood.com/recipes/french-toast


In [15]:
# Cell 14 — Test scraping relevant fries records only

fries_test_records = []
fries_test_rejected = []
fries_test_failed = []

for idx, row in fries_links_updated_df.iterrows():
    if len(fries_test_records) >= 10:
        break

    url = row["url"]
    query = row["query"]

    print("\nScraping URL:")
    print(url)

    record = scrape_bbc_good_food_recipe(
        url=url,
        food_class="fries",
        query=query
    )

    if record is None:
        fries_test_failed.append(url)
        print("Result: failed")
        continue

    if is_relevant_record_v2(record):
        fries_test_records.append(record)
        print("Result: relevant")
        print("Recipe:", record["recipe_name"])
        print("Calories:", record["calories"])
    else:
        fries_test_rejected.append(record)
        print("Result: rejected")
        print("Recipe:", record["recipe_name"])

fries_test_df = pd.DataFrame(fries_test_records)
fries_rejected_df = pd.DataFrame(fries_test_rejected)

print("\nFries relevance scraping test finished.")
print("Relevant fries records:", len(fries_test_df))
print("Rejected fries records:", len(fries_rejected_df))
print("Failed URLs:", len(fries_test_failed))

if not fries_test_df.empty:
    display(fries_test_df[[
        "food_class",
        "query",
        "recipe_name",
        "calories",
        "source_url"
    ]])

if not fries_rejected_df.empty:
    print("\nRejected examples:")
    display(fries_rejected_df[[
        "food_class",
        "query",
        "recipe_name",
        "calories",
        "source_url"
    ]].head(10))


Scraping URL:
https://www.bbcgoodfood.com/recipes/air-fryer-french-fries
Result: relevant
Recipe: Air fryer French fries
Calories: 204.0

Scraping URL:
https://www.bbcgoodfood.com/recipes/french-fries
Result: relevant
Recipe: French fries
Calories: 317.0

Scraping URL:
https://www.bbcgoodfood.com/recipes/korean-fried-chicken
Result: rejected
Recipe: Korean fried chicken

Scraping URL:
https://www.bbcgoodfood.com/recipes/egg-fried-rice
Result: rejected
Recipe: Easy egg-fried rice

Scraping URL:
https://www.bbcgoodfood.com/recipes/french-onion-soup
Result: rejected
Recipe: French onion soup

Scraping URL:
https://www.bbcgoodfood.com/recipes/beef-stroganoff
Result: rejected
Recipe: Beef stroganoff

Scraping URL:
https://www.bbcgoodfood.com/recipes/next-level-fried-chicken
Result: rejected
Recipe: Next level fried chicken

Scraping URL:
https://www.bbcgoodfood.com/recipes/meatball-marinara-fries
Result: relevant
Recipe: Meatball marinara fries
Calories: 399.0

Scraping URL:
https://www.bb

,food_class,query,recipe_name,calories,source_url
0,fries,french fries,Air fryer French fries,204.0,https://www.bbcgoodfood.com/recipes/air-fryer-...
1,fries,french fries,French fries,317.0,https://www.bbcgoodfood.com/recipes/french-fries
2,fries,french fries,Meatball marinara fries,399.0,https://www.bbcgoodfood.com/recipes/meatball-m...
3,fries,french fries,"Feta, tomato & olive loaded fries",290.0,https://www.bbcgoodfood.com/recipes/greek-load...
4,fries,french fries,Loaded fries,415.0,https://www.bbcgoodfood.com/recipes/loaded-fries
5,fries,french fries,Vegan bean chilli fries,428.0,https://www.bbcgoodfood.com/recipes/vegan-bean...
6,fries,french fries,Sweet potato fries,120.0,https://www.bbcgoodfood.com/recipes/sweet-pota...
7,fries,french fries,Air fryer sweet potato fries,244.0,https://www.bbcgoodfood.com/recipes/air-fryer-...
8,fries,loaded fries,Loaded chips ideas,409.0,https://www.bbcgoodfood.com/recipes/best-ever-...
9,fries,sweet potato fries,Steaks with goulash sauce & sweet potato fries,452.0,https://www.bbcgoodfood.com/recipes/steaks-gou...



Rejected examples:


,food_class,query,recipe_name,calories,source_url
0,fries,french fries,Korean fried chicken,487.0,https://www.bbcgoodfood.com/recipes/korean-fri...
1,fries,french fries,Easy egg-fried rice,387.0,https://www.bbcgoodfood.com/recipes/egg-fried-...
2,fries,french fries,French onion soup,618.0,https://www.bbcgoodfood.com/recipes/french-oni...
3,fries,french fries,Beef stroganoff,438.0,https://www.bbcgoodfood.com/recipes/beef-strog...
4,fries,french fries,Next level fried chicken,556.0,https://www.bbcgoodfood.com/recipes/next-level...
5,fries,french fries,French toast,401.0,https://www.bbcgoodfood.com/recipes/french-toast
6,fries,french fries,Prawn fried rice,418.0,https://www.bbcgoodfood.com/recipes/prawn-frie...
7,fries,french fries,Chorizo egg-fried rice,325.0,https://www.bbcgoodfood.com/recipes/chorizo-eg...
8,fries,french fries,Pineapple fried rice,301.0,https://www.bbcgoodfood.com/recipes/pineapple-...
9,fries,french fries,Chicken & ginger fried rice,535.0,https://www.bbcgoodfood.com/recipes/chicken-gi...


In [16]:
# Cell 15 — Final BBC scraped dataset collection

# Replace fries links with the improved fries links
bbc_all_class_links["fries"] = fries_links_updated_df

def scrape_final_bbc_records_for_class(food_class, links_df, target_records=50):
    """
    Scrape final relevant BBC Good Food records for one class.
    Keeps only records that pass relevance filtering.
    """
    records = []
    failed_urls = []
    rejected_records = []
    seen_urls = set()

    print("Final scraping class:", food_class)
    print("Available links:", len(links_df))
    print("Target records:", target_records)

    for idx, row in links_df.iterrows():
        if len(records) >= target_records:
            break

        url = row["url"]
        query = row["query"]

        if url in seen_urls:
            continue

        seen_urls.add(url)

        print("\nScraping URL:")
        print(url)

        record = scrape_bbc_good_food_recipe(
            url=url,
            food_class=food_class,
            query=query
        )

        if record is None:
            failed_urls.append({
                "food_class": food_class,
                "query": query,
                "url": url,
                "reason": "scrape_failed"
            })
            print("Result: failed")
            continue

        ingredients_text = record.get("ingredients_text")
        calories = record.get("calories")

        has_required_fields = (
            record.get("recipe_name") is not None
            and ingredients_text is not None
            and len(ingredients_text) > 0
            and calories is not None
        )

        if not has_required_fields:
            failed_urls.append({
                "food_class": food_class,
                "query": query,
                "url": url,
                "reason": "missing_required_fields"
            })
            print("Result: missing required fields")
            continue

        if not is_relevant_record_v2(record):
            rejected_records.append(record)
            print("Result: rejected")
            print("Recipe:", record.get("recipe_name"))
            continue

        records.append(record)
        print("Result: accepted")
        print("Recipe:", record["recipe_name"])
        print("Calories:", record["calories"])
        print("Collected so far:", len(records), "/", target_records)

    return records, failed_urls, rejected_records


final_bbc_records = []
final_failed_urls = []
final_rejected_records = []

for food_class in FOOD_CLASSES:
    print("\n====================================")
    print("Final scraping for class:", food_class)
    print("====================================")

    links_df = bbc_all_class_links[food_class]

    class_records, class_failed_urls, class_rejected_records = scrape_final_bbc_records_for_class(
        food_class=food_class,
        links_df=links_df,
        target_records=TARGET_RECORDS_PER_CLASS
    )

    final_bbc_records.extend(class_records)
    final_failed_urls.extend(class_failed_urls)
    final_rejected_records.extend(class_rejected_records)

    print("\nClass finished:", food_class)
    print("Accepted records:", len(class_records))
    print("Failed URLs:", len(class_failed_urls))
    print("Rejected records:", len(class_rejected_records))

final_bbc_df = pd.DataFrame(final_bbc_records)
final_failed_df = pd.DataFrame(final_failed_urls)
final_rejected_df = pd.DataFrame(final_rejected_records)

print("\nFinal BBC scraping finished.")
print("Final dataset shape:", final_bbc_df.shape)

print("\nRecords per class:")
print(final_bbc_df["food_class"].value_counts())

print("\nFailed URLs:", len(final_failed_df))
print("Rejected records:", len(final_rejected_df))

final_raw_output_path = SCRAPED_RAW_DIR / "bbc_text_calorie_final_raw.csv"
final_failed_output_path = LOGS_DIR / "bbc_final_failed_urls.csv"
final_rejected_output_path = LOGS_DIR / "bbc_final_rejected_records.csv"

final_bbc_df.to_csv(final_raw_output_path, index=False)
final_failed_df.to_csv(final_failed_output_path, index=False)
final_rejected_df.to_csv(final_rejected_output_path, index=False)

print("\nFinal raw dataset saved to:")
print(final_raw_output_path)

print("\nFailed URLs log saved to:")
print(final_failed_output_path)

print("\nRejected records log saved to:")
print(final_rejected_output_path)

display(final_bbc_df.head(10))


Final scraping for class: burger
Final scraping class: burger
Available links: 90
Target records: 50

Scraping URL:
https://www.bbcgoodfood.com/recipes/burger-bowl
Result: accepted
Recipe: Burger bowl
Calories: 207.0
Collected so far: 1 / 50

Scraping URL:
https://www.bbcgoodfood.com/recipes/falafel-burgers-0
Result: accepted
Recipe: Falafel burgers
Calories: 175.0
Collected so far: 2 / 50

Scraping URL:
https://www.bbcgoodfood.com/recipes/15-minute-chicken-halloumi-burgers
Result: accepted
Recipe: 15-minute chicken & halloumi burgers
Calories: 737.0
Collected so far: 3 / 50

Scraping URL:
https://www.bbcgoodfood.com/recipes/the-big-double-cheeseburger-secret-sauce
Result: accepted
Recipe: The ultimate beef burger
Calories: 893.0
Collected so far: 4 / 50

Scraping URL:
https://www.bbcgoodfood.com/recipes/next-level-chicken-burgers
Result: accepted
Recipe: Next level chicken burgers
Calories: 794.0
Collected so far: 5 / 50

Scraping URL:
https://www.bbcgoodfood.com/recipes/superhealthy

,source_site,food_class,query,recipe_name,ingredients_text,calories,protein,fat,carbs,source_url,scraped_at
0,bbc_good_food,burger,burger,Burger bowl,2 large potatoes cut into rough chunks 2 tbsp ...,207.0,18.0,13.0,19.0,https://www.bbcgoodfood.com/recipes/burger-bowl,2026-05-12 00:34:19
1,bbc_good_food,burger,burger,Falafel burgers,400g can chickpeas rinsed and drained 1 small...,175.0,6.0,8.0,18.0,https://www.bbcgoodfood.com/recipes/falafel-bu...,2026-05-12 00:34:21
2,bbc_good_food,burger,burger,15-minute chicken & halloumi burgers,2 chicken breast fillets 1 tbsp oil plus extra...,737.0,39.0,42.0,49.0,https://www.bbcgoodfood.com/recipes/15-minute-...,2026-05-12 00:34:23
3,bbc_good_food,burger,burger,The ultimate beef burger,1 small onion finely chopped 4 sesame-topped ...,893.0,56.0,37.0,82.0,https://www.bbcgoodfood.com/recipes/the-big-do...,2026-05-12 00:34:25
4,bbc_good_food,burger,burger,Next level chicken burgers,2 large chicken breasts sunflower oil for deep...,794.0,33.0,33.0,90.0,https://www.bbcgoodfood.com/recipes/next-level...,2026-05-12 00:34:27
5,bbc_good_food,burger,burger,Superhealthy salmon burgers,"4 boneless, skinless salmon fillets about 550g...",292.0,29.0,17.0,7.0,https://www.bbcgoodfood.com/recipes/superhealt...,2026-05-12 00:34:28
6,bbc_good_food,burger,burger,Smash burgers,"4 burger buns sesame topped or brioche, whiche...",666.0,37.0,39.0,41.0,https://www.bbcgoodfood.com/recipes/smash-burgers,2026-05-12 00:34:30
7,bbc_good_food,burger,burger,Mexican chicken burger,1 chicken breast 1 tsp chipotle paste 1 lime j...,709.0,46.0,34.0,52.0,https://www.bbcgoodfood.com/recipes/mexican-ch...,2026-05-12 00:34:32
8,bbc_good_food,burger,burger,Soft burger buns,200ml whole milk plus extra for brushing 50g u...,210.0,6.0,5.0,35.0,https://www.bbcgoodfood.com/recipes/soft-burge...,2026-05-12 00:34:33
9,bbc_good_food,burger,burger,Spiced halloumi & pineapple burger with zingy ...,½ red cabbage grated 2 carrots grated 100g rad...,264.0,11.0,14.0,19.0,https://www.bbcgoodfood.com/recipes/spiced-hal...,2026-05-12 00:34:35


In [17]:
# Cell 16 — Raw dataset audit

print("Raw dataset shape:")
print(final_bbc_df.shape)

print("\nColumns:")
print(final_bbc_df.columns.tolist())

print("\nRecords per class:")
class_counts = final_bbc_df["food_class"].value_counts()
print(class_counts)

print("\nMissing values per column:")
print(final_bbc_df.isna().sum())

print("\nDuplicate source URLs:")
print(final_bbc_df["source_url"].duplicated().sum())

print("\nDuplicate recipe names:")
print(final_bbc_df["recipe_name"].duplicated().sum())

print("\nNutrition summary:")
display(final_bbc_df[["calories", "protein", "fat", "carbs"]].describe())

print("\nCalories summary by class:")
display(
    final_bbc_df.groupby("food_class")["calories"]
    .agg(["count", "min", "max", "mean", "median"])
    .sort_values("count", ascending=False)
)

print("\nSample records from each class:")
for food_class in FOOD_CLASSES:
    print("\n====================================")
    print("Class:", food_class)
    print("====================================")

    sample_df = final_bbc_df[final_bbc_df["food_class"] == food_class][[
        "food_class",
        "recipe_name",
        "calories",
        "protein",
        "fat",
        "carbs",
        "source_url"
    ]].head(10)

    display(sample_df)

Raw dataset shape:
(428, 11)

Columns:
['source_site', 'food_class', 'query', 'recipe_name', 'ingredients_text', 'calories', 'protein', 'fat', 'carbs', 'source_url', 'scraped_at']

Records per class:
food_class
burger      50
fries       50
salad       50
pasta       50
rice        50
chicken     50
pizza       39
sandwich    39
steak       32
sushi       18
Name: count, dtype: int64

Missing values per column:
source_site         0
food_class          0
query               0
recipe_name         0
ingredients_text    0
calories            0
protein             0
fat                 1
carbs               3
source_url          0
scraped_at          0
dtype: int64

Duplicate source URLs:
12

Duplicate recipe names:
13

Nutrition summary:


,calories,protein,fat,carbs
count,428.000000,428.000000,427.000000,425.000000
mean,488.946262,37.042991,22.975504,42.725882
std,225.768216,223.331000,15.413796,24.109913
min,49.000000,0.400000,1.000000,0.400000
25%,332.750000,15.750000,12.000000,24.000000
50%,452.000000,27.000000,19.000000,42.000000
75%,619.500000,37.000000,30.515000,59.000000
max,1394.000000,4637.000000,89.000000,129.000000



Calories summary by class:


,count,min,max,mean,median
food_class,,,,,
burger,50,61.0,1080.0,514.480000,506.0
chicken,50,239.0,857.0,432.120000,407.5
fries,50,120.0,1394.0,406.660000,393.5
pasta,50,115.0,1140.0,605.440000,568.0
rice,50,58.0,834.0,454.780000,411.5
salad,50,111.0,838.0,403.940000,376.0
pizza,39,166.0,740.0,470.435897,446.0
sandwich,39,125.0,1256.0,616.974359,583.0
steak,32,225.0,1282.0,591.781250,516.5



Sample records from each class:

Class: burger


...

,food_class,recipe_name,calories,protein,fat,carbs,source_url
339,sandwich,Coronation chickpea sandwich filler,220.0,5.0,11.0,22.0,https://www.bbcgoodfood.com/recipes/coronation...
340,sandwich,Caprese sandwich,462.0,21.0,23.0,42.0,https://www.bbcgoodfood.com/recipes/caprese-sa...
341,sandwich,Next level steak sandwich,1166.0,69.0,63.0,76.0,https://www.bbcgoodfood.com/recipes/next-level...
342,sandwich,Fried chicken waffle sandwich,989.0,56.0,26.0,129.0,https://www.bbcgoodfood.com/recipes/fried-chic...
343,sandwich,Elvis sandwich,952.0,35.0,63.0,57.0,https://www.bbcgoodfood.com/recipes/elvis-sand...
344,sandwich,Panuozzo sandwich,552.0,22.0,25.0,58.0,https://www.bbcgoodfood.com/recipes/panuozzo-s...
345,sandwich,Pastrami sandwich,867.0,32.0,57.0,51.0,https://www.bbcgoodfood.com/recipes/pastrami-s...
346,sandwich,Tuna salad sandwich,474.0,21.0,23.0,42.0,https://www.bbcgoodfood.com/recipes/tuna-salad...
347,sandwich,Reuben sandwich,682.0,36.0,37.0,47.0,https://www.bbcgoodfood.com/recipes/reuben-san...
348,sandwich,Caprese chicken sandwiches,574.0,52.0,17.0,51.0,https://www.bbcgoodfood.com/recipes/caprese-ch...



Class: sushi


,food_class,recipe_name,calories,protein,fat,carbs,source_url
378,sushi,Easy salmon sushi rice bowl,713.0,41.0,27.0,73.0,https://www.bbcgoodfood.com/recipes/japanese-s...
379,sushi,Easy salmon sushi rice bowl,713.0,41.0,27.0,73.0,https://www.bbcgoodfood.com/recipes/easy-salmo...
380,sushi,Quick sushi bowl,498.0,27.0,11.0,70.0,https://www.bbcgoodfood.com/recipes/quick-sush...
381,sushi,Sesame & ginger sushi bowls,449.0,17.0,21.0,44.0,https://www.bbcgoodfood.com/recipes/sesame-gin...
382,sushi,Salmon sushi salad,862.0,27.0,43.0,87.0,https://www.bbcgoodfood.com/recipes/salmon-sus...
383,sushi,Perfect sushi rice,432.0,6.0,1.0,99.0,https://www.bbcgoodfood.com/recipes/sushi-rice
384,sushi,Simple sushi,390.0,8.0,9.0,70.0,https://www.bbcgoodfood.com/recipes/simple-sushi
385,sushi,"Sushi rice bowl with beef, egg & chilli sauce",621.0,41.0,23.0,63.0,https://www.bbcgoodfood.com/recipes/sushi-rice...
386,sushi,Smoked salmon & avocado sushi,49.0,2.0,2.0,7.0,https://www.bbcgoodfood.com/recipes/smoked-sal...
387,sushi,Build-your-own salmon sushi burrito,441.0,19.0,26.0,30.0,https://www.bbcgoodfood.com/recipes/build-your...



Class: steak


,food_class,recipe_name,calories,protein,fat,carbs,source_url
396,steak,Easy steak pie,611.0,39.0,36.0,32.0,https://www.bbcgoodfood.com/recipes/easy-steak...
397,steak,Next level steak & ale pie,939.0,38.0,49.0,80.0,https://www.bbcgoodfood.com/recipes/steak-ale-pie
398,steak,Steak Diane,402.0,24.0,26.0,4.0,https://www.bbcgoodfood.com/recipes/next-level...
399,steak,Air fryer steak,523.0,46.0,37.0,NaN,https://www.bbcgoodfood.com/recipes/air-fryer-...
400,steak,Spanish pork shoulder steaks with beans,454.0,39.0,14.0,35.0,https://www.bbcgoodfood.com/recipes/spanish-po...
401,steak,Smoky steak with Cajun potatoes & spicy slaw,802.0,47.0,50.0,40.0,https://www.bbcgoodfood.com/recipes/smoky-stea...
402,steak,"Steak, ale & mushroom pie",1282.0,54.0,68.0,107.0,https://www.bbcgoodfood.com/recipes/proper-bee...
403,steak,One-pan sirloin steak & creamy mushroom sauce,1078.0,55.0,89.0,5.0,https://www.bbcgoodfood.com/recipes/one-pan-si...
404,steak,Steak & blue cheese pie,831.0,44.0,50.0,49.0,https://www.bbcgoodfood.com/recipes/steak-blue...
405,steak,Sesame steak & buckwheat noodle bowls,550.0,34.0,16.0,65.0,https://www.bbcgoodfood.com/recipes/sesame-ste...


In [18]:
# Cell 17 — Detailed quality issue inspection

audit_df = final_bbc_df.copy()

print("Dataset shape before cleaning:")
print(audit_df.shape)

print("\nRecords per class:")
print(audit_df["food_class"].value_counts())

print("\nRows with duplicate source_url:")
duplicate_url_df = audit_df[audit_df["source_url"].duplicated(keep=False)].sort_values("source_url")
print("Duplicate URL rows:", len(duplicate_url_df))
display(duplicate_url_df[[
    "food_class",
    "recipe_name",
    "calories",
    "protein",
    "fat",
    "carbs",
    "source_url"
]])

print("\nRows with duplicate recipe_name:")
duplicate_name_df = audit_df[audit_df["recipe_name"].duplicated(keep=False)].sort_values("recipe_name")
print("Duplicate recipe name rows:", len(duplicate_name_df))
display(duplicate_name_df[[
    "food_class",
    "recipe_name",
    "calories",
    "protein",
    "fat",
    "carbs",
    "source_url"
]])

print("\nRows with missing nutrition values:")
missing_nutrition_df = audit_df[
    audit_df[["calories", "protein", "fat", "carbs"]].isna().any(axis=1)
]
print("Missing nutrition rows:", len(missing_nutrition_df))
display(missing_nutrition_df[[
    "food_class",
    "recipe_name",
    "calories",
    "protein",
    "fat",
    "carbs",
    "source_url"
]])

print("\nNutrition outlier rows:")
outlier_df = audit_df[
    (audit_df["calories"] > 1200)
    | (audit_df["protein"] > 120)
    | (audit_df["fat"] > 100)
    | (audit_df["carbs"] > 150)
    | (audit_df["calories"] < 50)
]
print("Outlier rows:", len(outlier_df))
display(outlier_df[[
    "food_class",
    "recipe_name",
    "calories",
    "protein",
    "fat",
    "carbs",
    "source_url"
]].sort_values(["protein", "calories"], ascending=False))

SUSPICIOUS_KEYWORDS_BY_CLASS = {
    "burger": [
        "sauce", "bun", "buns", "pasta"
    ],
    "pizza": [
        "dough", "sauce", "fondue", "cookie", "dip"
    ],
    "fries": [
        "hummus", "pitta chips", "pasta chips", "fish & chips"
    ],
    "pasta": [
        "soup", "sauce for pasta"
    ],
    "rice": [
        "rice pudding", "cauliflower rice"
    ],
    "chicken": [
        "salad", "pasta"
    ],
    "salad": [
        "dressing", "wrap", "burgers"
    ],
    "sandwich": [
        "cake", "biscuit", "biscuits", "cookie", "cookies", "ice cream", "shortbread", "raspberry"
    ],
    "sushi": [
        "sweet sushi", "sushi rice"
    ],
    "steak": [
        "pork", "tuna steak", "pie", "pudding"
    ]
}

def has_suspicious_keyword(row):
    food_class = row["food_class"]
    recipe_name = normalize_text_for_filtering(row["recipe_name"])
    source_url = normalize_text_for_filtering(row["source_url"])
    combined_text = recipe_name + " " + source_url

    suspicious_keywords = SUSPICIOUS_KEYWORDS_BY_CLASS.get(food_class, [])

    for keyword in suspicious_keywords:
        if keyword in combined_text:
            return True

    return False

audit_df["is_suspicious"] = audit_df.apply(has_suspicious_keyword, axis=1)

print("\nSuspicious rows by class:")
print(audit_df.groupby("food_class")["is_suspicious"].value_counts())

print("\nSuspicious examples:")
suspicious_df = audit_df[audit_df["is_suspicious"] == True]
display(suspicious_df[[
    "food_class",
    "recipe_name",
    "calories",
    "protein",
    "fat",
    "carbs",
    "source_url"
]].sort_values(["food_class", "recipe_name"]))

print("\nClean candidate count if we remove duplicates, missing nutrition, outliers, and suspicious rows:")

clean_candidate_df = audit_df.copy()

clean_candidate_df = clean_candidate_df.drop_duplicates(subset=["source_url"])
clean_candidate_df = clean_candidate_df.drop_duplicates(subset=["recipe_name"])
clean_candidate_df = clean_candidate_df.dropna(subset=["calories", "protein", "fat", "carbs"])

clean_candidate_df = clean_candidate_df[
    (clean_candidate_df["calories"] >= 50)
    & (clean_candidate_df["calories"] <= 1200)
    & (clean_candidate_df["protein"] <= 120)
    & (clean_candidate_df["fat"] <= 100)
    & (clean_candidate_df["carbs"] <= 150)
]

clean_candidate_df = clean_candidate_df[clean_candidate_df["is_suspicious"] == False]

print(clean_candidate_df.shape)

print("\nClean candidate records per class:")
print(clean_candidate_df["food_class"].value_counts())

Dataset shape before cleaning:
(428, 11)

Records per class:
food_class
burger      50
fries       50
salad       50
pasta       50
rice        50
chicken     50
pizza       39
sandwich    39
steak       32
sushi       18
Name: count, dtype: int64

Rows with duplicate source_url:
Duplicate URL rows: 24


,food_class,recipe_name,calories,protein,fat,carbs,source_url
140,pasta,Cajun chicken pasta,555.0,28.0,20.0,63.0,https://www.bbcgoodfood.com/recipes/cajun-chic...
252,chicken,Cajun chicken pasta,555.0,28.0,20.0,63.0,https://www.bbcgoodfood.com/recipes/cajun-chic...
26,burger,Cheeseburger & chips,558.0,38.0,21.0,58.0,https://www.bbcgoodfood.com/recipes/cheeseburg...
115,fries,Cheeseburger & chips,558.0,38.0,21.0,58.0,https://www.bbcgoodfood.com/recipes/cheeseburg...
258,chicken,Chicken & bacon pasta,857.0,35.0,58.0,46.0,https://www.bbcgoodfood.com/recipes/chicken-ba...
141,pasta,Chicken & bacon pasta,857.0,35.0,58.0,46.0,https://www.bbcgoodfood.com/recipes/chicken-ba...
240,chicken,Chicken pasta bake,575.0,33.0,30.0,41.0,https://www.bbcgoodfood.com/recipes/chicken-pa...
139,pasta,Chicken pasta bake,575.0,33.0,30.0,41.0,https://www.bbcgoodfood.com/recipes/chicken-pa...
246,chicken,Chicken satay salad,353.0,38.0,10.0,24.0,https://www.bbcgoodfood.com/recipes/chicken-sa...
289,salad,Chicken satay salad,353.0,38.0,10.0,24.0,https://www.bbcgoodfood.com/recipes/chicken-sa...



Rows with duplicate recipe_name:
Duplicate recipe name rows: 25


,food_class,recipe_name,calories,protein,fat,carbs,source_url
140,pasta,Cajun chicken pasta,555.0,28.0,20.0,63.0,https://www.bbcgoodfood.com/recipes/cajun-chic...
252,chicken,Cajun chicken pasta,555.0,28.0,20.0,63.0,https://www.bbcgoodfood.com/recipes/cajun-chic...
26,burger,Cheeseburger & chips,558.0,38.0,21.0,58.0,https://www.bbcgoodfood.com/recipes/cheeseburg...
115,fries,Cheeseburger & chips,558.0,38.0,21.0,58.0,https://www.bbcgoodfood.com/recipes/cheeseburg...
141,pasta,Chicken & bacon pasta,857.0,35.0,58.0,46.0,https://www.bbcgoodfood.com/recipes/chicken-ba...
258,chicken,Chicken & bacon pasta,857.0,35.0,58.0,46.0,https://www.bbcgoodfood.com/recipes/chicken-ba...
139,pasta,Chicken pasta bake,575.0,33.0,30.0,41.0,https://www.bbcgoodfood.com/recipes/chicken-pa...
240,chicken,Chicken pasta bake,575.0,33.0,30.0,41.0,https://www.bbcgoodfood.com/recipes/chicken-pa...
246,chicken,Chicken satay salad,353.0,38.0,10.0,24.0,https://www.bbcgoodfood.com/recipes/chicken-sa...
289,salad,Chicken satay salad,353.0,38.0,10.0,24.0,https://www.bbcgoodfood.com/recipes/chicken-sa...



Rows with missing nutrition values:
Missing nutrition rows: 3


...


Nutrition outlier rows:
Outlier rows: 6


,food_class,recipe_name,calories,protein,fat,carbs,source_url
176,pasta,Chicken & broccoli pasta bake,764.0,4637.0,34.0,65.0,https://www.bbcgoodfood.com/recipes/chicken-br...
377,sandwich,Best ever Christmas leftovers sandwich,1237.0,61.0,66.0,94.0,https://www.bbcgoodfood.com/recipes/best-ever-...
376,sandwich,Ultimate turkey sandwich,1256.0,60.0,67.0,97.0,https://www.bbcgoodfood.com/recipes/ultimate-t...
106,fries,Air-fryer or oven-cooked fish & chips,1394.0,55.0,85.0,98.0,https://www.bbcgoodfood.com/recipes/air-fryer-...
402,steak,"Steak, ale & mushroom pie",1282.0,54.0,68.0,107.0,https://www.bbcgoodfood.com/recipes/proper-bee...
386,sushi,Smoked salmon & avocado sushi,49.0,2.0,2.0,7.0,https://www.bbcgoodfood.com/recipes/smoked-sal...



Suspicious rows by class:
food_class  is_suspicious
burger      False            46
            True              4
chicken     False            43
            True              7
fries       False            48
            True              2
pasta       False            47
            True              3
pizza       False            35
            True              4
rice        False            47
            True              3
salad       False            43
            True              7
sandwich    False            34
            True              5
steak       False            23
            True              9
sushi       False            13
            True              5
Name: count, dtype: int64

Suspicious examples:


,food_class,recipe_name,calories,protein,fat,carbs,source_url
18,burger,Classic burger sauce,61.0,0.4,1.0,2.0,https://www.bbcgoodfood.com/recipes/classic-bu...
24,burger,One-pot cheeseburger pasta,687.0,43.0,30.0,57.0,https://www.bbcgoodfood.com/recipes/one-pot-ch...
8,burger,Soft burger buns,210.0,6.0,5.0,35.0,https://www.bbcgoodfood.com/recipes/soft-burge...
3,burger,The ultimate beef burger,893.0,56.0,37.0,82.0,https://www.bbcgoodfood.com/recipes/the-big-do...
252,chicken,Cajun chicken pasta,555.0,28.0,20.0,63.0,https://www.bbcgoodfood.com/recipes/cajun-chic...
267,chicken,Cajun grilled chicken with lime black-eyed bea...,491.0,48.0,20.0,30.0,https://www.bbcgoodfood.com/recipes/cajun-gril...
258,chicken,Chicken & bacon pasta,857.0,35.0,58.0,46.0,https://www.bbcgoodfood.com/recipes/chicken-ba...
240,chicken,Chicken pasta bake,575.0,33.0,30.0,41.0,https://www.bbcgoodfood.com/recipes/chicken-pa...
246,chicken,Chicken satay salad,353.0,38.0,10.0,24.0,https://www.bbcgoodfood.com/recipes/chicken-sa...
282,chicken,"Chicken, spinach & bacon alfredo pasta bake",757.0,30.0,43.0,60.0,https://www.bbcgoodfood.com/recipes/chicken-sp...



Clean candidate count if we remove duplicates, missing nutrition, outliers, and suspicious rows:
(365, 12)

Clean candidate records per class:
food_class
rice        47
burger      46
pasta       46
fries       45
salad       41
chicken     41
pizza       35
sandwich    32
steak       20
sushi       12
Name: count, dtype: int64


In [19]:
# Cell 18 — Smart cleaning candidate v1

clean_work_df = final_bbc_df.copy()

clean_work_df["normalized_recipe_name"] = clean_work_df["recipe_name"].apply(normalize_text_for_filtering)
clean_work_df["normalized_source_url"] = clean_work_df["source_url"].apply(normalize_text_for_filtering)


def hard_reject_reason(row):
    """
    Return a hard rejection reason for records that are clearly not suitable
    for the intended class.
    """
    food_class = row["food_class"]
    name = row["normalized_recipe_name"]

    if food_class == "burger":
        if "burger sauce" in name:
            return "burger_sauce_only"
        if "burger buns" in name or name.endswith("buns"):
            return "burger_buns_only"
        if "cheeseburger pasta" in name or "burger pasta" in name:
            return "burger_pasta_dish"

    if food_class == "pizza":
        if "pizza sauce" in name:
            return "pizza_sauce_only"
        if "pizza dough" in name:
            return "pizza_dough_only"
        if "cookie dough pizza" in name:
            return "dessert_pizza"
        if "pizza fondue dip" in name or "pizza dip" in name:
            return "pizza_dip"

    if food_class == "fries":
        if "pitta chips" in name:
            return "pitta_chips_not_fries"
        if "pasta chips" in name:
            return "pasta_chips_not_fries"
        if "tortilla chips" in name:
            return "tortilla_chips_not_fries"

    if food_class == "pasta":
        if "pasta soup" in name:
            return "pasta_soup"
        if "sauce for pasta" in name:
            return "pasta_sauce_only"

    if food_class == "rice":
        if "rice pudding" in name:
            return "rice_dessert"
        if "cauliflower rice" in name:
            return "cauliflower_rice_not_rice"

    if food_class == "chicken":
        if "pasta" in name:
            return "chicken_pasta_belongs_to_pasta"
        if "salad" in name:
            return "chicken_salad_belongs_to_salad"

    if food_class == "salad":
        if "salad dressing" in name:
            return "salad_dressing_only"
        if "salad wrap" in name:
            return "salad_wrap_not_salad"
        if "salad burgers" in name:
            return "salad_burger_not_salad"

    if food_class == "sandwich":
        dessert_terms = [
            "cake",
            "biscuit",
            "biscuits",
            "cookie",
            "cookies",
            "ice cream",
            "shortbread",
            "raspberry",
            "custard"
        ]

        for term in dessert_terms:
            if term in name:
                return "dessert_sandwich"

    if food_class == "sushi":
        if name == "sweet sushi":
            return "sweet_sushi_dessert"
        if name == "perfect sushi rice":
            return "sushi_rice_only"

    if food_class == "steak":
        if "pork" in name:
            return "pork_steak_not_target"
        if "tuna steak" in name:
            return "tuna_steak_not_target"
        if "steak pie" in name or "steak & ale pie" in name or "steak ale pie" in name:
            return "steak_pie_not_steak_plate"
        if "steak & kidney pudding" in name or "kidney pudding" in name:
            return "steak_pudding_not_steak_plate"

    return None


def nutrition_reject_reason(row):
    """
    Return rejection reason for missing or impossible nutrition values.
    """
    required_cols = ["calories", "protein", "fat", "carbs"]

    for col in required_cols:
        if pd.isna(row[col]):
            return "missing_nutrition"

    if row["protein"] > 120:
        return "protein_outlier"

    if row["calories"] < 30:
        return "calories_too_low"

    if row["calories"] > 1600:
        return "calories_too_high"

    if row["fat"] > 120:
        return "fat_outlier"

    if row["carbs"] > 180:
        return "carbs_outlier"

    return None


def get_class_match_score(food_class, name):
    """
    Score how strongly the recipe name matches a class.
    Used only for resolving duplicates across classes.
    """
    score = 0

    if food_class == "pizza":
        if "pizza" in name or "pizzas" in name:
            score += 5

    elif food_class == "burger":
        if "burger" in name or "burgers" in name or "cheeseburger" in name or "beefburger" in name:
            score += 5

    elif food_class == "fries":
        if "fries" in name or "chips" in name or "wedges" in name:
            score += 5

    elif food_class == "pasta":
        pasta_terms = ["pasta", "spaghetti", "penne", "lasagne", "lasagna", "tagliatelle", "fettuccine", "macaroni", "orzo"]
        if any(term in name for term in pasta_terms):
            score += 5

    elif food_class == "rice":
        rice_terms = ["rice", "risotto", "paella", "pilau", "biryani", "jambalaya"]
        if any(term in name for term in rice_terms):
            score += 5

    elif food_class == "chicken":
        if "chicken" in name:
            score += 2

    elif food_class == "salad":
        if "salad" in name or "slaw" in name:
            score += 5

    elif food_class == "sandwich":
        sandwich_terms = ["sandwich", "sandwiches", "toastie", "panini", "sub sandwich"]
        if any(term in name for term in sandwich_terms):
            score += 5

    elif food_class == "sushi":
        sushi_terms = ["sushi", "maki", "onigiri", "sushi bowl", "sushi rice bowl", "sushi rolls"]
        if any(term in name for term in sushi_terms):
            score += 5

    elif food_class == "steak":
        steak_terms = ["steak", "steaks", "sirloin", "rib-eye", "rib eye", "rump steak"]
        if any(term in name for term in steak_terms):
            score += 5

    return score


CLASS_PRIORITY_FOR_DUPLICATES = {
    "burger": 1,
    "pizza": 2,
    "pasta": 3,
    "rice": 4,
    "sushi": 5,
    "steak": 6,
    "salad": 7,
    "sandwich": 8,
    "fries": 9,
    "chicken": 10
}


def build_drop_reason(row):
    reasons = []

    nutrition_reason = nutrition_reject_reason(row)
    if nutrition_reason is not None:
        reasons.append(nutrition_reason)

    hard_reason = hard_reject_reason(row)
    if hard_reason is not None:
        reasons.append(hard_reason)

    if len(reasons) == 0:
        return ""

    return "|".join(reasons)


clean_work_df["drop_reason"] = clean_work_df.apply(build_drop_reason, axis=1)

hard_removed_df = clean_work_df[clean_work_df["drop_reason"] != ""].copy()
clean_stage_df = clean_work_df[clean_work_df["drop_reason"] == ""].copy()

print("Initial raw shape:")
print(clean_work_df.shape)

print("\nRows removed by hard rules:")
print(len(hard_removed_df))

print("\nHard removal reasons:")
print(hard_removed_df["drop_reason"].value_counts())

print("\nShape after hard removal:")
print(clean_stage_df.shape)

# Score records before duplicate resolution
clean_stage_df["dedupe_match_score"] = clean_stage_df.apply(
    lambda row: get_class_match_score(row["food_class"], row["normalized_recipe_name"]),
    axis=1
)

clean_stage_df["class_priority"] = clean_stage_df["food_class"].map(CLASS_PRIORITY_FOR_DUPLICATES)

# Resolve duplicate source URLs
before_url_dedup = len(clean_stage_df)

clean_stage_df = clean_stage_df.sort_values(
    by=["source_url", "dedupe_match_score", "class_priority"],
    ascending=[True, False, True]
)

clean_stage_df = clean_stage_df.drop_duplicates(subset=["source_url"], keep="first")

after_url_dedup = len(clean_stage_df)

# Resolve duplicate recipe names
before_name_dedup = len(clean_stage_df)

clean_stage_df = clean_stage_df.sort_values(
    by=["recipe_name", "dedupe_match_score", "class_priority"],
    ascending=[True, False, True]
)

clean_stage_df = clean_stage_df.drop_duplicates(subset=["recipe_name"], keep="first")

after_name_dedup = len(clean_stage_df)

# Remove helper columns from final cleaned candidate
cleaned_candidate_v1_df = clean_stage_df.drop(
    columns=[
        "normalized_recipe_name",
        "normalized_source_url",
        "drop_reason",
        "dedupe_match_score",
        "class_priority"
    ],
    errors="ignore"
).reset_index(drop=True)

print("\nDuplicate source_url rows removed:")
print(before_url_dedup - after_url_dedup)

print("\nDuplicate recipe_name rows removed:")
print(before_name_dedup - after_name_dedup)

print("\nCleaned candidate v1 shape:")
print(cleaned_candidate_v1_df.shape)

print("\nCleaned candidate v1 records per class:")
print(cleaned_candidate_v1_df["food_class"].value_counts())

print("\nMissing values after cleaning:")
print(cleaned_candidate_v1_df.isna().sum())

print("\nNutrition summary after cleaning:")
display(cleaned_candidate_v1_df[["calories", "protein", "fat", "carbs"]].describe())

print("\nCalories summary by class after cleaning:")
display(
    cleaned_candidate_v1_df.groupby("food_class")["calories"]
    .agg(["count", "min", "max", "mean", "median"])
    .sort_values("count", ascending=False)
)

# Save cleaning audit files
cleaned_candidate_v1_path = CLEAN_DIR / "bbc_text_calorie_cleaned_candidate_v1.csv"
hard_removed_v1_path = LOGS_DIR / "bbc_cleaning_v1_hard_removed.csv"

cleaned_candidate_v1_df.to_csv(cleaned_candidate_v1_path, index=False)
hard_removed_df.to_csv(hard_removed_v1_path, index=False)

print("\nCleaned candidate v1 saved to:")
print(cleaned_candidate_v1_path)

print("\nHard removed rows saved to:")
print(hard_removed_v1_path)

Initial raw shape:
(428, 14)

Rows removed by hard rules:
41

Hard removal reasons:
drop_reason
dessert_sandwich                           5
chicken_pasta_belongs_to_pasta             4
chicken_salad_belongs_to_salad             3
rice_dessert                               2
pork_steak_not_target                      2
steak_pie_not_steak_plate                  2
salad_dressing_only                        2
missing_nutrition                          2
pasta_soup                                 2
burger_buns_only                           1
protein_outlier                            1
tortilla_chips_not_fries                   1
pitta_chips_not_fries                      1
pasta_chips_not_fries                      1
dessert_pizza                              1
pizza_dip                                  1
burger_pasta_dish                          1
burger_sauce_only                          1
salad_burger_not_salad                     1
salad_wrap_not_salad                       1
past

,calories,protein,fat,carbs
count,379.000000,379.00000,379.000000,379.000000
mean,493.424802,26.70343,23.244169,42.713720
std,222.134443,13.87972,15.463211,24.138651
min,49.000000,1.00000,1.000000,0.400000
25%,338.500000,17.00000,12.000000,24.000000
50%,455.000000,27.00000,19.000000,42.000000
75%,622.500000,36.00000,31.500000,58.000000
max,1394.000000,69.00000,89.000000,129.000000



Calories summary by class after cleaning:


,count,min,max,mean,median
food_class,,,,,
burger,47,175.0,1080.0,526.936170,508.0
rice,47,164.0,834.0,470.404255,418.0
pasta,46,371.0,1140.0,625.173913,587.5
salad,45,111.0,838.0,404.444444,374.0
fries,44,120.0,1394.0,403.590909,379.0
chicken,41,239.0,707.0,409.243902,368.0
pizza,37,266.0,740.0,479.702703,462.0
sandwich,33,125.0,1256.0,632.909091,619.0
steak,25,253.0,1282.0,599.080000,510.0



Cleaned candidate v1 saved to:
/content/drive/MyDrive/Calorify/phase2_text_calorie/data/cleaned/bbc_text_calorie_cleaned_candidate_v1.csv

Hard removed rows saved to:
/content/drive/MyDrive/Calorify/phase2_text_calorie/logs/bbc_cleaning_v1_hard_removed.csv


In [20]:
# Cell 19 — Test one Skinnytaste recipe page scraper

def scrape_skinnytaste_recipe(url, food_class=None, query=None):
    """
    Scrape one Skinnytaste recipe page using JSON-LD when available.
    Returns a structured recipe record.
    """
    response = safe_get(url, sleep_min=1, sleep_max=2)

    if response is None:
        return None

    if response.status_code != 200:
        print("Failed URL:", url)
        print("Status code:", response.status_code)
        return None

    soup = BeautifulSoup(response.text, "html.parser")
    recipe_json = find_recipe_json_ld(soup)

    if recipe_json is None:
        print("No Recipe JSON-LD found for:", url)
        return None

    recipe_name = recipe_json.get("name")

    ingredients = recipe_json.get("recipeIngredient", [])
    if isinstance(ingredients, list):
        ingredients_text = " ".join([str(item) for item in ingredients])
    else:
        ingredients_text = str(ingredients)

    nutrition = recipe_json.get("nutrition", {})
    if not isinstance(nutrition, dict):
        nutrition = {}

    calories = extract_number(nutrition.get("calories"))
    protein = extract_number(nutrition.get("proteinContent"))
    fat = extract_number(nutrition.get("fatContent"))
    carbs = extract_number(nutrition.get("carbohydrateContent"))

    record = {
        "source_site": "skinnytaste",
        "food_class": food_class,
        "query": query,
        "recipe_name": recipe_name,
        "ingredients_text": ingredients_text,
        "calories": calories,
        "protein": protein,
        "fat": fat,
        "carbs": carbs,
        "source_url": url,
        "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return record


# Use one known Skinnytaste burger link from the earlier refined search
test_skinnytaste_url = skinnytaste_burger_refined_df.iloc[0]["url"]

print("Testing Skinnytaste recipe URL:")
print(test_skinnytaste_url)

test_skinnytaste_record = scrape_skinnytaste_recipe(
    url=test_skinnytaste_url,
    food_class="burger",
    query="burger"
)

print("\nScraped Skinnytaste record:")
print(test_skinnytaste_record)

if test_skinnytaste_record is not None:
    print("\nIngredients length:", len(test_skinnytaste_record["ingredients_text"]))
    print("Calories:", test_skinnytaste_record["calories"])
    print("Protein:", test_skinnytaste_record["protein"])
    print("Fat:", test_skinnytaste_record["fat"])
    print("Carbs:", test_skinnytaste_record["carbs"])

Testing Skinnytaste recipe URL:
https://www.skinnytaste.com/spicy-black-bean-burgers-with-chipotle/

Scraped Skinnytaste record:
{'source_site': 'skinnytaste', 'food_class': 'burger', 'query': 'burger', 'recipe_name': 'Spicy Black Bean Burgers with Chipotle Mayonnaise', 'ingredients_text': "3 1/2 tbsp light mayonnaise (Hellman's) 1 tbsp canned chipotle in adobo sauce 16 oz can black beans (rinsed and drained) 1/2  red bell pepper (cut into 2 inch pieces) 1/2 cup chopped scallions 3 tbsp chopped cilantro 3 cloves garlic (peeled) 1  jumbo egg 1 tbsp cumin 1/4 to 1/2 tsp kosher salt 1 tsp hot sauce 1/2 cup quick oats (use gf oats for gluten free) 4  whole wheat 100 calorie buns (Martin's*) 1  small hass avocado (sliced thin)", 'calories': 362.5, 'protein': 18.0, 'fat': 14.0, 'carbs': 50.0, 'source_url': 'https://www.skinnytaste.com/spicy-black-bean-burgers-with-chipotle/', 'scraped_at': '2026-05-12 01:52:01'}

Ingredients length: 439
Calories: 362.5
Protein: 18.0
Fat: 14.0
Carbs: 50.0


In [21]:
# Cell 20 — Extract Skinnytaste links for classes that need supplementation

SUPPLEMENT_CLASSES = ["pizza", "sandwich", "steak", "sushi"]

CURRENT_CLEAN_COUNTS = cleaned_candidate_v1_df["food_class"].value_counts().to_dict()

SUPPLEMENT_TARGETS = {
    food_class: max(0, TARGET_RECORDS_PER_CLASS - CURRENT_CLEAN_COUNTS.get(food_class, 0))
    for food_class in SUPPLEMENT_CLASSES
}

SKINNYTASTE_SEARCH_QUERIES = {
    "pizza": [
        "pizza",
        "pepperoni pizza",
        "margherita pizza",
        "flatbread pizza",
        "pizza recipe"
    ],
    "sandwich": [
        "sandwich",
        "chicken sandwich",
        "turkey sandwich",
        "grilled cheese",
        "wrap"
    ],
    "steak": [
        "steak",
        "beef steak",
        "sirloin steak",
        "grilled steak",
        "steak recipe"
    ],
    "sushi": [
        "sushi",
        "sushi bowl",
        "sushi roll",
        "poke bowl",
        "salmon bowl",
        "tuna bowl"
    ]
}

print("Current clean counts:")
for food_class in SUPPLEMENT_CLASSES:
    print(food_class, "current:", CURRENT_CLEAN_COUNTS.get(food_class, 0), "needed:", SUPPLEMENT_TARGETS[food_class])

skinnytaste_supplement_links = {}

for food_class in SUPPLEMENT_CLASSES:
    print("\n====================================")
    print("Extracting Skinnytaste links for:", food_class)
    print("====================================")

    class_links = []

    for query in SKINNYTASTE_SEARCH_QUERIES[food_class]:
        print("\nQuery:", query)

        links_df = extract_refined_recipe_links_from_search(
            source_name="skinnytaste",
            query=query,
            max_links=20
        )

        if links_df is not None and not links_df.empty:
            class_links.append(links_df)

        time.sleep(1)

    if class_links:
        class_links_df = pd.concat(class_links, ignore_index=True)
        class_links_df = class_links_df.drop_duplicates(subset=["url"]).reset_index(drop=True)
    else:
        class_links_df = pd.DataFrame(columns=["source_name", "query", "title_text", "url"])

    skinnytaste_supplement_links[food_class] = class_links_df

    print("\nFinished:", food_class)
    print("Unique Skinnytaste links found:", len(class_links_df))

    if not class_links_df.empty:
        display(class_links_df.head(20))

print("\nSkinnytaste supplement link summary:")
for food_class, links_df in skinnytaste_supplement_links.items():
    print(food_class, "links:", len(links_df), "needed:", SUPPLEMENT_TARGETS[food_class])

Current clean counts:
pizza current: 37 needed: 13
sandwich current: 33 needed: 17
steak current: 25 needed: 25
sushi current: 14 needed: 36

Extracting Skinnytaste links for: pizza

Query: pizza
Source: skinnytaste
Search URL: https://www.skinnytaste.com/?s=pizza
Status code: 200
HTML length: 577514
Refined recipe links found: 10


,source_name,query,title_text,url
0,skinnytaste,pizza,Chicken Crust Pizza,https://www.skinnytaste.com/chicken-crust-pizza/
1,skinnytaste,pizza,Breakfast Pizza (Packed With Protein!),https://www.skinnytaste.com/breakfast-pizza/
2,skinnytaste,pizza,Fruit Pizza,https://www.skinnytaste.com/fruit-pizza/
3,skinnytaste,pizza,The 4 Best Pizza Stones of 2024,https://www.skinnytaste.com/best-pizza-stones/
4,skinnytaste,pizza,Spicy Salmon Sushi Pizza,https://www.skinnytaste.com/spicy-salmon-sushi...
5,skinnytaste,pizza,Pizza Sausage Rolls,https://www.skinnytaste.com/pizza-sausage-rolls/
6,skinnytaste,pizza,Pepperoni Pizza Bites,https://www.skinnytaste.com/pepperoni-pizza-bi...
7,skinnytaste,pizza,Spaghetti Squash Crust Pizza,https://www.skinnytaste.com/spaghetti-squash-c...
8,skinnytaste,pizza,Margherita Pizza,https://www.skinnytaste.com/margherita-pizza/
9,skinnytaste,pizza,Thin Tortilla Pizza,https://www.skinnytaste.com/cast-iron-thin-cru...



Query: pepperoni pizza
Source: skinnytaste
Search URL: https://www.skinnytaste.com/?s=pepperoni+pizza
Status code: 200
HTML length: 513851
Refined recipe links found: 1


,source_name,query,title_text,url
0,skinnytaste,pepperoni pizza,Pepperoni Pizza Bites,https://www.skinnytaste.com/pepperoni-pizza-bi...



Query: margherita pizza
Source: skinnytaste
Search URL: https://www.skinnytaste.com/?s=margherita+pizza
Status code: 200
HTML length: 514350
Refined recipe links found: 1


...


Query: tuna bowl
Source: skinnytaste
Search URL: https://www.skinnytaste.com/?s=tuna+bowl
Status code: 200
HTML length: 558505
Refined recipe links found: 6


,source_name,query,title_text,url
0,skinnytaste,tuna bowl,Super Bowl Recipes,https://www.skinnytaste.com/holiday-recipes/su...
1,skinnytaste,tuna bowl,Spicy Tuna Poke Bowl,https://www.skinnytaste.com/spicy-tuna-poke-bo...
2,skinnytaste,tuna bowl,Tuna Sub-in-a-Tub,https://www.skinnytaste.com/tuna-sub-in-a-tub/
3,skinnytaste,tuna bowl,Shoyu Ahi Tuna Poke,https://www.skinnytaste.com/shoyu-aji-poke/
4,skinnytaste,tuna bowl,Tofu Poke Bowl,https://www.skinnytaste.com/tofu-poke-bowl/
5,skinnytaste,tuna bowl,Ahi Poke Bowl with Mango,https://www.skinnytaste.com/ahi-poke-bowl/



Finished: sushi
Unique Skinnytaste links found: 17


,source_name,query,title_text,url
0,skinnytaste,sushi,Spicy Salmon Sushi Pizza,https://www.skinnytaste.com/spicy-salmon-sushi...
1,skinnytaste,sushi bowl,Super Bowl Recipes,https://www.skinnytaste.com/holiday-recipes/su...
2,skinnytaste,sushi bowl,Spicy Canned Salmon Rice Bowl,https://www.skinnytaste.com/spicy-canned-salmo...
3,skinnytaste,sushi roll,California Roll Cucumber Salad,https://www.skinnytaste.com/california-roll-cu...
4,skinnytaste,poke bowl,Tofu Poke Bowl,https://www.skinnytaste.com/tofu-poke-bowl/
5,skinnytaste,poke bowl,Spicy Tuna Poke Bowl,https://www.skinnytaste.com/spicy-tuna-poke-bo...
6,skinnytaste,poke bowl,Ahi Poke Bowl with Mango,https://www.skinnytaste.com/ahi-poke-bowl/
7,skinnytaste,poke bowl,Shoyu Ahi Tuna Poke,https://www.skinnytaste.com/shoyu-aji-poke/
8,skinnytaste,salmon bowl,Teriyaki Salmon Bowl (Air Fryer or Oven),https://www.skinnytaste.com/teriyaki-salmon-bo...
9,skinnytaste,salmon bowl,Asian Salmon Bowl,https://www.skinnytaste.com/seattle-asian-salm...



Skinnytaste supplement link summary:
pizza links: 42 needed: 13
sandwich links: 48 needed: 17
steak links: 47 needed: 25
sushi links: 17 needed: 36


In [22]:
# Cell 21 — Scrape Skinnytaste supplement records for missing classes

def is_valid_skinnytaste_recipe_candidate_url(url):
    """
    Remove obvious non-recipe Skinnytaste URLs before scraping.
    """
    if url is None:
        return False

    parsed = urlparse(url)
    path = parsed.path.strip("/").lower()

    if not path:
        return False

    blocked_prefixes = [
        "holiday-recipes",
        "main-ingredient",
        "recipe-index",
        "category",
        "meal-plans",
        "shopping",
        "cookbook",
        "cookbooks",
        "how-to",
        "about",
        "contact",
        "privacy",
        "saved-recipes",
        "grilled-recipes",
        "one-pot",
    ]

    for prefix in blocked_prefixes:
        if path == prefix or path.startswith(prefix + "/"):
            return False

    blocked_exact_or_contains = [
        "best-pizza-stones",
        "how-to-view-your-saved-recipes",
    ]

    for blocked in blocked_exact_or_contains:
        if blocked in path:
            return False

    return True


def supplement_hard_reject_reason(row):
    """
    Extra hard rejection rules for Skinnytaste supplement records.
    This extends the earlier hard_reject_reason logic.
    """
    base_reason = hard_reject_reason(row)
    if base_reason is not None:
        return base_reason

    food_class = row["food_class"]
    name = row["normalized_recipe_name"]

    if food_class == "pizza":
        if "pizza stone" in name or "best pizza stones" in name:
            return "pizza_equipment_not_recipe"
        if "fruit pizza" in name:
            return "dessert_pizza"

    if food_class == "sandwich":
        if "bread" in name and "sandwich" not in name:
            return "bread_not_sandwich"
        if "recipes" in name:
            return "category_page_not_recipe"
        if "salad" in name and "sandwich" not in name and "wrap" not in name:
            return "salad_not_sandwich"
        if "quesadilla" in name:
            return "quesadilla_not_sandwich"
        if "fajitas" in name:
            return "fajita_not_sandwich"
        if "dip" in name:
            return "dip_not_sandwich"

    if food_class == "steak":
        if "recipes" in name:
            return "category_page_not_recipe"
        if "brussels sprouts" in name:
            return "vegetable_not_steak"
        if "pasta" in name:
            return "steak_pasta_dish"

    if food_class == "sushi":
        if "super bowl" in name:
            return "super_bowl_category_not_sushi"
        if "chowder" in name:
            return "chowder_not_sushi"
        if "salmon cakes" in name:
            return "salmon_cakes_not_sushi"
        if "salmon bites" in name:
            return "salmon_bites_not_sushi"
        if "tuna sub" in name:
            return "tuna_sub_not_sushi"
        if "sushi pizza" in name:
            return "sushi_pizza_hybrid"
        if "california roll cucumber salad" in name:
            return "sushi_style_salad_not_sushi"

    return None


def build_supplement_drop_reason(row):
    """
    Apply nutrition and supplement-specific hard rejection rules.
    """
    reasons = []

    nutrition_reason = nutrition_reject_reason(row)
    if nutrition_reason is not None:
        reasons.append(nutrition_reason)

    hard_reason = supplement_hard_reject_reason(row)
    if hard_reason is not None:
        reasons.append(hard_reason)

    if len(reasons) == 0:
        return ""

    return "|".join(reasons)


def scrape_skinnytaste_supplement_for_class(food_class, links_df, needed_records, existing_urls, existing_names):
    """
    Scrape Skinnytaste supplement records for one class.
    Keeps only clean, non-duplicate records.
    """
    accepted_records = []
    rejected_records = []
    failed_urls = []

    print("Supplement scraping class:", food_class)
    print("Needed records:", needed_records)
    print("Candidate links before URL filtering:", len(links_df))

    if needed_records <= 0:
        return accepted_records, rejected_records, failed_urls

    filtered_links_df = links_df.copy()
    filtered_links_df = filtered_links_df[
        filtered_links_df["url"].apply(is_valid_skinnytaste_recipe_candidate_url)
    ].drop_duplicates(subset=["url"]).reset_index(drop=True)

    print("Candidate links after URL filtering:", len(filtered_links_df))

    for idx, row in filtered_links_df.iterrows():
        if len(accepted_records) >= needed_records:
            break

        url = row["url"]
        query = row["query"]

        if url in existing_urls:
            print("\nSkipping existing URL:")
            print(url)
            continue

        print("\nScraping Skinnytaste URL:")
        print(url)

        record = scrape_skinnytaste_recipe(
            url=url,
            food_class=food_class,
            query=query
        )

        if record is None:
            failed_urls.append({
                "food_class": food_class,
                "query": query,
                "url": url,
                "reason": "scrape_failed"
            })
            print("Result: failed")
            continue

        record["normalized_recipe_name"] = normalize_text_for_filtering(record["recipe_name"])
        record["normalized_source_url"] = normalize_text_for_filtering(record["source_url"])

        normalized_name = record["normalized_recipe_name"]

        if normalized_name in existing_names:
            rejected_records.append({
                **record,
                "drop_reason": "duplicate_recipe_name_existing_dataset"
            })
            print("Result: rejected duplicate name")
            print("Recipe:", record["recipe_name"])
            continue

        drop_reason = build_supplement_drop_reason(record)

        if drop_reason != "":
            rejected_records.append({
                **record,
                "drop_reason": drop_reason
            })
            print("Result: rejected")
            print("Recipe:", record["recipe_name"])
            print("Reason:", drop_reason)
            continue

        accepted_clean_record = {
            "source_site": record["source_site"],
            "food_class": record["food_class"],
            "query": record["query"],
            "recipe_name": record["recipe_name"],
            "ingredients_text": record["ingredients_text"],
            "calories": record["calories"],
            "protein": record["protein"],
            "fat": record["fat"],
            "carbs": record["carbs"],
            "source_url": record["source_url"],
            "scraped_at": record["scraped_at"]
        }

        accepted_records.append(accepted_clean_record)
        existing_urls.add(url)
        existing_names.add(normalized_name)

        print("Result: accepted")
        print("Recipe:", record["recipe_name"])
        print("Calories:", record["calories"])
        print("Collected:", len(accepted_records), "/", needed_records)

    return accepted_records, rejected_records, failed_urls


existing_urls = set(cleaned_candidate_v1_df["source_url"].tolist())
existing_names = set(cleaned_candidate_v1_df["recipe_name"].apply(normalize_text_for_filtering).tolist())

skinnytaste_accepted_records = []
skinnytaste_rejected_records = []
skinnytaste_failed_urls = []

for food_class in SUPPLEMENT_CLASSES:
    print("\n====================================")
    print("Skinnytaste supplement for:", food_class)
    print("====================================")

    links_df = skinnytaste_supplement_links[food_class]
    needed_records = SUPPLEMENT_TARGETS[food_class]

    class_accepted, class_rejected, class_failed = scrape_skinnytaste_supplement_for_class(
        food_class=food_class,
        links_df=links_df,
        needed_records=needed_records,
        existing_urls=existing_urls,
        existing_names=existing_names
    )

    skinnytaste_accepted_records.extend(class_accepted)
    skinnytaste_rejected_records.extend(class_rejected)
    skinnytaste_failed_urls.extend(class_failed)

    print("\nFinished supplement class:", food_class)
    print("Accepted:", len(class_accepted))
    print("Rejected:", len(class_rejected))
    print("Failed:", len(class_failed))

skinnytaste_supplement_df = pd.DataFrame(skinnytaste_accepted_records)
skinnytaste_supplement_rejected_df = pd.DataFrame(skinnytaste_rejected_records)
skinnytaste_supplement_failed_df = pd.DataFrame(skinnytaste_failed_urls)

print("\nSkinnytaste supplement scraping finished.")
print("Accepted supplement shape:", skinnytaste_supplement_df.shape)

if not skinnytaste_supplement_df.empty:
    print("\nAccepted supplement records per class:")
    print(skinnytaste_supplement_df["food_class"].value_counts())

print("\nRejected supplement records:", len(skinnytaste_supplement_rejected_df))
print("Failed supplement URLs:", len(skinnytaste_supplement_failed_df))

skinnytaste_supplement_path = SCRAPED_RAW_DIR / "skinnytaste_text_calorie_supplement_raw.csv"
skinnytaste_rejected_path = LOGS_DIR / "skinnytaste_supplement_rejected_records.csv"
skinnytaste_failed_path = LOGS_DIR / "skinnytaste_supplement_failed_urls.csv"

skinnytaste_supplement_df.to_csv(skinnytaste_supplement_path, index=False)
skinnytaste_supplement_rejected_df.to_csv(skinnytaste_rejected_path, index=False)
skinnytaste_supplement_failed_df.to_csv(skinnytaste_failed_path, index=False)

print("\nSkinnytaste supplement saved to:")
print(skinnytaste_supplement_path)

print("\nSkinnytaste rejected log saved to:")
print(skinnytaste_rejected_path)

print("\nSkinnytaste failed URL log saved to:")
print(skinnytaste_failed_path)

if not skinnytaste_supplement_df.empty:
    display(skinnytaste_supplement_df[[
        "source_site",
        "food_class",
        "query",
        "recipe_name",
        "calories",
        "protein",
        "fat",
        "carbs",
        "source_url"
    ]])


Skinnytaste supplement for: pizza
Supplement scraping class: pizza
Needed records: 13
Candidate links before URL filtering: 42
Candidate links after URL filtering: 14

Scraping Skinnytaste URL:
https://www.skinnytaste.com/chicken-crust-pizza/
Result: accepted
Recipe: Chicken Crust Pizza with Veggies
Calories: 287.0
Collected: 1 / 13

Scraping Skinnytaste URL:
https://www.skinnytaste.com/breakfast-pizza/
Result: accepted
Recipe: Breakfast Pizza (Packed With Protein!)
Calories: 271.0
Collected: 2 / 13

Scraping Skinnytaste URL:
https://www.skinnytaste.com/fruit-pizza/
Result: rejected
Recipe: Fruit Pizza
Reason: dessert_pizza

Scraping Skinnytaste URL:
https://www.skinnytaste.com/spicy-salmon-sushi-pizza/
Result: accepted
Recipe: Spicy Salmon Sushi Pizza
Calories: 403.0
Collected: 3 / 13

Scraping Skinnytaste URL:
https://www.skinnytaste.com/pizza-sausage-rolls/
Result: accepted
Recipe: Pizza Sausage Rolls
Calories: 302.0
Collected: 4 / 13

Scraping Skinnytaste URL:
https://www.skinnyta

,source_site,food_class,query,recipe_name,calories,protein,fat,carbs,source_url
0,skinnytaste,pizza,pizza,Chicken Crust Pizza with Veggies,287.0,34.0,14.50,7.0,https://www.skinnytaste.com/chicken-crust-pizza/
1,skinnytaste,pizza,pizza,Breakfast Pizza (Packed With Protein!),271.0,20.5,9.00,27.0,https://www.skinnytaste.com/breakfast-pizza/
2,skinnytaste,pizza,pizza,Spicy Salmon Sushi Pizza,403.0,24.5,26.50,17.0,https://www.skinnytaste.com/spicy-salmon-sushi...
3,skinnytaste,pizza,pizza,Pizza Sausage Rolls,302.0,22.0,10.00,30.0,https://www.skinnytaste.com/pizza-sausage-rolls/
4,skinnytaste,pizza,pizza,Pepperoni Pizza Bites,286.0,18.0,9.50,31.0,https://www.skinnytaste.com/pepperoni-pizza-bi...
5,skinnytaste,pizza,pizza,Spaghetti Squash Crust Pizza,303.0,28.0,17.00,18.0,https://www.skinnytaste.com/spaghetti-squash-c...
6,skinnytaste,pizza,pizza,Margarita Pizza,236.0,15.0,6.50,27.0,https://www.skinnytaste.com/margherita-pizza/
7,skinnytaste,pizza,pizza,Tortilla Pizza Recipe,118.0,4.5,5.00,15.5,https://www.skinnytaste.com/cast-iron-thin-cru...
8,skinnytaste,pizza,flatbread pizza,Lavash Flatbread Pizzas,192.0,12.0,5.00,24.0,https://www.skinnytaste.com/lavash-flatbread-p...
9,skinnytaste,pizza,flatbread pizza,Smoked Salmon Breakfast Flatbread,247.0,15.5,8.00,28.5,https://www.skinnytaste.com/smoked-salmon-brea...


In [23]:
# Cell 22 — Audit Skinnytaste supplement before merging

supplement_audit_df = skinnytaste_supplement_df.copy()

print("Skinnytaste supplement shape:")
print(supplement_audit_df.shape)

print("\nRecords per class:")
print(supplement_audit_df["food_class"].value_counts())

print("\nMissing values:")
print(supplement_audit_df.isna().sum())

print("\nDuplicate source URLs:")
print(supplement_audit_df["source_url"].duplicated().sum())

print("\nDuplicate recipe names:")
print(supplement_audit_df["recipe_name"].duplicated().sum())

print("\nNutrition summary:")
display(supplement_audit_df[["calories", "protein", "fat", "carbs"]].describe())

SUPPLEMENT_REVIEW_KEYWORDS = {
    "pizza": [
        "sushi pizza",
        "fruit pizza"
    ],
    "sandwich": [
        "bread",
        "burger",
        "burgers",
        "meatballs",
        "roll ups"
    ],
    "steak": [
        "baked potato",
        "pasta",
        "salad"
    ],
    "sushi": [
        "poke",
        "salmon bowl",
        "rice bowl",
        "salad rice bowl"
    ]
}

def is_supplement_review_needed(row):
    food_class = row["food_class"]
    name = normalize_text_for_filtering(row["recipe_name"])
    url = normalize_text_for_filtering(row["source_url"])
    combined = name + " " + url

    keywords = SUPPLEMENT_REVIEW_KEYWORDS.get(food_class, [])

    for keyword in keywords:
        if keyword in combined:
            return True

    return False

supplement_audit_df["needs_manual_review"] = supplement_audit_df.apply(
    is_supplement_review_needed,
    axis=1
)

print("\nManual review summary:")
print(supplement_audit_df.groupby("food_class")["needs_manual_review"].value_counts())

print("\nRecords needing manual review:")
review_df = supplement_audit_df[supplement_audit_df["needs_manual_review"] == True]

display(review_df[[
    "source_site",
    "food_class",
    "query",
    "recipe_name",
    "calories",
    "protein",
    "fat",
    "carbs",
    "source_url"
]].sort_values(["food_class", "recipe_name"]))

print("\nClean-looking supplement records:")
clean_looking_supplement_df = supplement_audit_df[
    supplement_audit_df["needs_manual_review"] == False
].copy()

display(clean_looking_supplement_df[[
    "source_site",
    "food_class",
    "query",
    "recipe_name",
    "calories",
    "protein",
    "fat",
    "carbs",
    "source_url"
]].sort_values(["food_class", "recipe_name"]))

Skinnytaste supplement shape:
(57, 11)

Records per class:
food_class
steak       18
sandwich    17
pizza       12
sushi       10
Name: count, dtype: int64

Missing values:
source_site         0
food_class          0
query               0
recipe_name         0
ingredients_text    0
calories            0
protein             0
fat                 0
carbs               0
source_url          0
scraped_at          0
dtype: int64

Duplicate source URLs:
0

Duplicate recipe names:
0

Nutrition summary:


,calories,protein,fat,carbs
count,57.000000,57.000000,57.000000,57.000000
mean,316.385965,27.231579,12.290351,24.482456
std,104.727368,9.115022,5.139582,15.333776
min,118.000000,4.500000,3.000000,0.500000
25%,247.000000,22.000000,9.000000,12.000000
50%,286.000000,27.000000,11.500000,27.000000
75%,394.500000,33.000000,15.000000,32.000000
max,533.000000,48.500000,26.500000,62.500000



Manual review summary:
food_class  needs_manual_review
pizza       False                  11
            True                    1
sandwich    False                  13
            True                    4
steak       False                  15
            True                    3
sushi       True                   10
Name: count, dtype: int64

Records needing manual review:


,source_site,food_class,query,recipe_name,calories,protein,fat,carbs,source_url
2,skinnytaste,pizza,pizza,Spicy Salmon Sushi Pizza,403.0,24.5,26.5,17.0,https://www.skinnytaste.com/spicy-salmon-sushi...
17,skinnytaste,sandwich,sandwich,High Protein Bread (Oat Sandwich Rolls),135.5,10.5,3.0,18.0,https://www.skinnytaste.com/high-protein-bread...
23,skinnytaste,sandwich,chicken sandwich,Pickle Ham and Swiss Chicken Roll Ups (Cuban C...,280.0,36.0,9.0,12.5,https://www.skinnytaste.com/cubano-chicken-pic...
26,skinnytaste,sandwich,turkey sandwich,Turkey Burgers with Zucchini,161.0,18.0,7.0,4.5,https://www.skinnytaste.com/turkey-burgers-wit...
28,skinnytaste,sandwich,turkey sandwich,Turkey Meatballs,280.0,25.5,11.5,25.0,https://www.skinnytaste.com/skinny-italian-mea...
40,skinnytaste,steak,beef steak,Carne Asada Salad,390.0,38.0,21.5,12.0,https://www.skinnytaste.com/carne-asada-steak-...
29,skinnytaste,steak,steak,Fall Steak Salad with Sweet Potatoes,394.5,33.5,17.5,26.0,https://www.skinnytaste.com/fall-steak-salad-w...
43,skinnytaste,steak,beef steak,Loaded Philly Cheesesteak Baked Potato,390.0,26.0,13.0,44.0,https://www.skinnytaste.com/loaded-philly-chee...
56,skinnytaste,sushi,salmon bowl,5-Minute Microwave Salmon Rice Bowl,533.0,48.5,18.0,51.0,https://www.skinnytaste.com/microwave-salmon-r...
50,skinnytaste,sushi,poke bowl,Ahi Poke Bowl with Mango,527.0,34.0,19.0,60.0,https://www.skinnytaste.com/ahi-poke-bowl/



Clean-looking supplement records:


,source_site,food_class,query,recipe_name,calories,protein,fat,carbs,source_url
1,skinnytaste,pizza,pizza,Breakfast Pizza (Packed With Protein!),271.0,20.5,9.00,27.0,https://www.skinnytaste.com/breakfast-pizza/
0,skinnytaste,pizza,pizza,Chicken Crust Pizza with Veggies,287.0,34.0,14.50,7.0,https://www.skinnytaste.com/chicken-crust-pizza/
10,skinnytaste,pizza,pizza recipe,Eggs Pizzaiola,266.0,19.0,15.00,13.0,https://www.skinnytaste.com/eggs-pizzaiola/
11,skinnytaste,pizza,pizza recipe,French Bread Pizza Mummies,239.0,10.0,8.00,32.0,https://www.skinnytaste.com/french-bread-pizza...
8,skinnytaste,pizza,flatbread pizza,Lavash Flatbread Pizzas,192.0,12.0,5.00,24.0,https://www.skinnytaste.com/lavash-flatbread-p...
6,skinnytaste,pizza,pizza,Margarita Pizza,236.0,15.0,6.50,27.0,https://www.skinnytaste.com/margherita-pizza/
4,skinnytaste,pizza,pizza,Pepperoni Pizza Bites,286.0,18.0,9.50,31.0,https://www.skinnytaste.com/pepperoni-pizza-bi...
3,skinnytaste,pizza,pizza,Pizza Sausage Rolls,302.0,22.0,10.00,30.0,https://www.skinnytaste.com/pizza-sausage-rolls/
9,skinnytaste,pizza,flatbread pizza,Smoked Salmon Breakfast Flatbread,247.0,15.5,8.00,28.5,https://www.skinnytaste.com/smoked-salmon-brea...
5,skinnytaste,pizza,pizza,Spaghetti Squash Crust Pizza,303.0,28.0,17.00,18.0,https://www.skinnytaste.com/spaghetti-squash-c...


In [24]:
# Cell 23 — Clean Skinnytaste supplement and merge with BBC cleaned candidate

supplement_clean_work_df = skinnytaste_supplement_df.copy()

SUPPLEMENT_REMOVE_RECIPE_NAMES = [
    # Pizza hybrid / weak pizza
    "Spicy Salmon Sushi Pizza",

    # Sandwich weak/non-sandwich items
    "High Protein Bread (Oat Sandwich Rolls)",
    "Pickle Ham and Swiss Chicken Roll Ups (Cuban Chicken)",
    "Turkey Burgers with Zucchini",
    "Turkey Meatballs",

    # Steak weak/non-steak-plate items
    "Fall Steak Salad with Sweet Potatoes",
    "Carne Asada Salad",
    "Loaded Philly Cheesesteak Baked Potato",
    "Steak & Caramelized Onions with Arugula and Penne",
]

# Remove all Skinnytaste sushi supplement for now because it is mostly poke/salmon/rice bowls, not direct sushi.
remove_by_name_mask = supplement_clean_work_df["recipe_name"].isin(SUPPLEMENT_REMOVE_RECIPE_NAMES)
remove_sushi_mask = supplement_clean_work_df["food_class"] == "sushi"

supplement_removed_manual_df = supplement_clean_work_df[
    remove_by_name_mask | remove_sushi_mask
].copy()

skinnytaste_supplement_clean_v1_df = supplement_clean_work_df[
    ~(remove_by_name_mask | remove_sushi_mask)
].copy().reset_index(drop=True)

print("Original Skinnytaste supplement shape:")
print(supplement_clean_work_df.shape)

print("\nManually removed supplement records:")
print(supplement_removed_manual_df.shape)

print("\nManual removal records per class:")
if not supplement_removed_manual_df.empty:
    print(supplement_removed_manual_df["food_class"].value_counts())

display(supplement_removed_manual_df[[
    "source_site",
    "food_class",
    "query",
    "recipe_name",
    "calories",
    "protein",
    "fat",
    "carbs",
    "source_url"
]].sort_values(["food_class", "recipe_name"]))

print("\nClean Skinnytaste supplement v1 shape:")
print(skinnytaste_supplement_clean_v1_df.shape)

print("\nClean Skinnytaste supplement v1 records per class:")
print(skinnytaste_supplement_clean_v1_df["food_class"].value_counts())

# Merge BBC cleaned candidate with Skinnytaste cleaned supplement
merged_text_calorie_df = pd.concat(
    [cleaned_candidate_v1_df, skinnytaste_supplement_clean_v1_df],
    ignore_index=True
)

# Safety duplicate removal after merge
merged_text_calorie_df["normalized_recipe_name"] = merged_text_calorie_df["recipe_name"].apply(normalize_text_for_filtering)

before_merge_dedup = len(merged_text_calorie_df)

merged_text_calorie_df = merged_text_calorie_df.drop_duplicates(subset=["source_url"], keep="first")
merged_text_calorie_df = merged_text_calorie_df.drop_duplicates(subset=["normalized_recipe_name"], keep="first")

after_merge_dedup = len(merged_text_calorie_df)

merged_text_calorie_df = merged_text_calorie_df.drop(
    columns=["normalized_recipe_name"],
    errors="ignore"
).reset_index(drop=True)

print("\nRows removed by safety dedup after merge:")
print(before_merge_dedup - after_merge_dedup)

print("\nMerged dataset shape:")
print(merged_text_calorie_df.shape)

print("\nMerged records per class:")
print(merged_text_calorie_df["food_class"].value_counts())

print("\nMissing values after merge:")
print(merged_text_calorie_df.isna().sum())

print("\nDuplicate source URLs after merge:")
print(merged_text_calorie_df["source_url"].duplicated().sum())

print("\nDuplicate recipe names after merge:")
print(merged_text_calorie_df["recipe_name"].duplicated().sum())

print("\nNutrition summary after merge:")
display(merged_text_calorie_df[["calories", "protein", "fat", "carbs"]].describe())

print("\nCalories summary by class after merge:")
display(
    merged_text_calorie_df.groupby("food_class")["calories"]
    .agg(["count", "min", "max", "mean", "median"])
    .sort_values("count", ascending=False)
)

# Save files
skinnytaste_supplement_clean_v1_path = CLEAN_DIR / "skinnytaste_text_calorie_supplement_clean_v1.csv"
supplement_removed_manual_path = LOGS_DIR / "skinnytaste_supplement_manual_removed.csv"
merged_text_calorie_path = CLEAN_DIR / "text_calorie_merged_clean_candidate_v1.csv"

skinnytaste_supplement_clean_v1_df.to_csv(skinnytaste_supplement_clean_v1_path, index=False)
supplement_removed_manual_df.to_csv(supplement_removed_manual_path, index=False)
merged_text_calorie_df.to_csv(merged_text_calorie_path, index=False)

print("\nClean Skinnytaste supplement saved to:")
print(skinnytaste_supplement_clean_v1_path)

print("\nManual removed supplement log saved to:")
print(supplement_removed_manual_path)

print("\nMerged clean candidate dataset saved to:")
print(merged_text_calorie_path)

Original Skinnytaste supplement shape:
(57, 11)

Manually removed supplement records:
(19, 11)

Manual removal records per class:
food_class
sushi       10
sandwich     4
steak        4
pizza        1
Name: count, dtype: int64


,source_site,food_class,query,recipe_name,calories,protein,fat,carbs,source_url
2,skinnytaste,pizza,pizza,Spicy Salmon Sushi Pizza,403.0,24.5,26.5,17.0,https://www.skinnytaste.com/spicy-salmon-sushi...
17,skinnytaste,sandwich,sandwich,High Protein Bread (Oat Sandwich Rolls),135.5,10.5,3.0,18.0,https://www.skinnytaste.com/high-protein-bread...
23,skinnytaste,sandwich,chicken sandwich,Pickle Ham and Swiss Chicken Roll Ups (Cuban C...,280.0,36.0,9.0,12.5,https://www.skinnytaste.com/cubano-chicken-pic...
26,skinnytaste,sandwich,turkey sandwich,Turkey Burgers with Zucchini,161.0,18.0,7.0,4.5,https://www.skinnytaste.com/turkey-burgers-wit...
28,skinnytaste,sandwich,turkey sandwich,Turkey Meatballs,280.0,25.5,11.5,25.0,https://www.skinnytaste.com/skinny-italian-mea...
40,skinnytaste,steak,beef steak,Carne Asada Salad,390.0,38.0,21.5,12.0,https://www.skinnytaste.com/carne-asada-steak-...
29,skinnytaste,steak,steak,Fall Steak Salad with Sweet Potatoes,394.5,33.5,17.5,26.0,https://www.skinnytaste.com/fall-steak-salad-w...
43,skinnytaste,steak,beef steak,Loaded Philly Cheesesteak Baked Potato,390.0,26.0,13.0,44.0,https://www.skinnytaste.com/loaded-philly-chee...
46,skinnytaste,steak,grilled steak,Steak & Caramelized Onions with Arugula and Penne,384.0,30.0,15.0,29.5,https://www.skinnytaste.com/steak-caramelized-...
56,skinnytaste,sushi,salmon bowl,5-Minute Microwave Salmon Rice Bowl,533.0,48.5,18.0,51.0,https://www.skinnytaste.com/microwave-salmon-r...



Clean Skinnytaste supplement v1 shape:
(38, 11)

Clean Skinnytaste supplement v1 records per class:
food_class
steak       14
sandwich    13
pizza       11
Name: count, dtype: int64

Rows removed by safety dedup after merge:
1

Merged dataset shape:
(416, 11)

Merged records per class:
food_class
pizza       48
burger      47
rice        47
sandwich    46
pasta       45
salad       45
fries       44
chicken     41
steak       39
sushi       14
Name: count, dtype: int64

Missing values after merge:
source_site         0
food_class          0
query               0
recipe_name         0
ingredients_text    0
calories            0
protein             0
fat                 0
carbs               0
source_url          0
scraped_at          0
dtype: int64

Duplicate source URLs after merge:
0

Duplicate recipe names after merge:
0

Nutrition summary after merge:


,calories,protein,fat,carbs
count,416.000000,416.000000,416.000000,416.000000
mean,474.264423,26.607692,22.134832,40.697115
std,221.537219,13.535983,15.202219,24.178034
min,49.000000,1.000000,1.000000,0.400000
25%,317.000000,17.750000,11.000000,22.000000
50%,435.000000,26.500000,18.000000,39.000000
75%,593.000000,35.000000,30.000000,56.000000
max,1394.000000,69.000000,89.000000,129.000000



Calories summary by class after merge:


,count,min,max,mean,median
food_class,,,,,
pizza,48,118.0,740.0,427.000000,427.0
burger,47,175.0,1080.0,526.936170,508.0
rice,47,164.0,834.0,470.404255,418.0
sandwich,46,125.0,1256.0,545.913043,470.0
salad,45,111.0,838.0,404.444444,374.0
pasta,45,371.0,1140.0,626.288889,600.0
fries,44,120.0,1394.0,403.590909,379.0
chicken,41,239.0,707.0,409.243902,368.0
steak,39,188.0,1282.0,483.717949,415.0



Clean Skinnytaste supplement saved to:
/content/drive/MyDrive/Calorify/phase2_text_calorie/data/cleaned/skinnytaste_text_calorie_supplement_clean_v1.csv

Manual removed supplement log saved to:
/content/drive/MyDrive/Calorify/phase2_text_calorie/logs/skinnytaste_supplement_manual_removed.csv

Merged clean candidate dataset saved to:
/content/drive/MyDrive/Calorify/phase2_text_calorie/data/cleaned/text_calorie_merged_clean_candidate_v1.csv


In [25]:
# Cell 24 — Create final cleaned text-calorie dataset for modeling

final_text_df = merged_text_calorie_df.copy()

def clean_text_for_modeling(text):
    """
    Basic text cleaning for model input.
    Keeps meaningful ingredient words and numbers.
    """
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9\s\.\-/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


final_text_df["clean_recipe_name"] = final_text_df["recipe_name"].apply(clean_text_for_modeling)
final_text_df["clean_ingredients_text"] = final_text_df["ingredients_text"].apply(clean_text_for_modeling)

final_text_df["input_text"] = (
    "food class: "
    + final_text_df["food_class"].astype(str)
    + " recipe name: "
    + final_text_df["clean_recipe_name"]
    + " ingredients: "
    + final_text_df["clean_ingredients_text"]
)

# Create stable record_id
final_text_df = final_text_df.reset_index(drop=True)
final_text_df.insert(0, "record_id", ["TXT_" + str(i).zfill(5) for i in range(len(final_text_df))])

# Keep final modeling columns
final_modeling_df = final_text_df[[
    "record_id",
    "source_site",
    "food_class",
    "query",
    "recipe_name",
    "ingredients_text",
    "clean_recipe_name",
    "clean_ingredients_text",
    "input_text",
    "calories",
    "protein",
    "fat",
    "carbs",
    "source_url",
    "scraped_at"
]].copy()

print("Final modeling dataset shape:")
print(final_modeling_df.shape)

print("\nRecords per class:")
print(final_modeling_df["food_class"].value_counts())

print("\nMissing values:")
print(final_modeling_df.isna().sum())

print("\nDuplicate source URLs:")
print(final_modeling_df["source_url"].duplicated().sum())

print("\nDuplicate input_text:")
print(final_modeling_df["input_text"].duplicated().sum())

print("\nInput text length summary:")
final_modeling_df["input_text_length"] = final_modeling_df["input_text"].str.len()
print(final_modeling_df["input_text_length"].describe())

print("\nSample final records:")
display(final_modeling_df[[
    "record_id",
    "food_class",
    "recipe_name",
    "calories",
    "protein",
    "fat",
    "carbs",
    "input_text"
]].head(10))

# Save final cleaned dataset
cleaned_text_calorie_dataset_path = CLEAN_DIR / "cleaned_text_calorie_dataset.csv"
final_modeling_df.to_csv(cleaned_text_calorie_dataset_path, index=False)

print("\nFinal cleaned text-calorie dataset saved to:")
print(cleaned_text_calorie_dataset_path)

Final modeling dataset shape:
(416, 15)

Records per class:
food_class
pizza       48
burger      47
rice        47
sandwich    46
pasta       45
salad       45
fries       44
chicken     41
steak       39
sushi       14
Name: count, dtype: int64

Missing values:
record_id                 0
source_site               0
food_class                0
query                     0
recipe_name               0
ingredients_text          0
clean_recipe_name         0
clean_ingredients_text    0
input_text                0
calories                  0
protein                   0
fat                       0
carbs                     0
source_url                0
scraped_at                0
dtype: int64

Duplicate source URLs:
0

Duplicate input_text:
0

Input text length summary:
count    416.000000
mean     381.918269
std      115.246820
min      109.000000
25%      300.000000
50%      375.500000
75%      440.250000
max      765.000000
Name: input_text_length, dtype: float64

Sample final records:


,record_id,food_class,recipe_name,calories,protein,fat,carbs,input_text
0,TXT_00000,chicken,'Marry me' chicken,584.0,38.0,38.0,21.0,food class: chicken recipe name: marry me chic...
1,TXT_00001,salad,10-minute couscous salad,468.0,16.0,24.0,44.0,food class: salad recipe name: 10-minute cousc...
2,TXT_00002,burger,15-minute chicken & halloumi burgers,737.0,39.0,42.0,49.0,food class: burger recipe name: 15-minute chic...
3,TXT_00003,pasta,15-minute chicken pasta,531.0,43.0,11.0,70.0,food class: pasta recipe name: 15-minute chick...
4,TXT_00004,fries,Air fryer French fries,204.0,3.0,7.0,30.0,food class: fries recipe name: air fryer frenc...
5,TXT_00005,sandwich,Air fryer cheese & ham toastie,627.0,27.0,38.0,43.0,food class: sandwich recipe name: air fryer ch...
6,TXT_00006,fries,Air fryer potato wedges,202.0,3.0,6.0,32.0,food class: fries recipe name: air fryer potat...
7,TXT_00007,fries,Air fryer sweet potato fries,244.0,3.0,3.0,48.0,food class: fries recipe name: air fryer sweet...
8,TXT_00008,fries,Air-fryer chips,140.0,3.0,3.0,24.0,food class: fries recipe name: air-fryer chips...
9,TXT_00009,fries,Air-fryer fish & chips,654.0,26.0,34.0,59.0,food class: fries recipe name: air-fryer fish ...



Final cleaned text-calorie dataset saved to:
/content/drive/MyDrive/Calorify/phase2_text_calorie/data/cleaned/cleaned_text_calorie_dataset.csv


In [26]:
# Cell 25A — Test additional candidate sources for sushi augmentation

ADDITIONAL_SOURCE_CANDIDATES = {
    "eatingwell": {
        "base_url": "https://www.eatingwell.com",
        "search_url_template": "https://www.eatingwell.com/search?q={query}"
    },
    "recipetineats": {
        "base_url": "https://www.recipetineats.com",
        "search_url_template": "https://www.recipetineats.com/?s={query}"
    },
    "the_spruce_eats": {
        "base_url": "https://www.thespruceeats.com",
        "search_url_template": "https://www.thespruceeats.com/search?q={query}"
    },
    "jamie_oliver": {
        "base_url": "https://www.jamieoliver.com",
        "search_url_template": "https://www.jamieoliver.com/search/?s={query}"
    },
    "healthy_food_guide": {
        "base_url": "https://www.healthyfood.com",
        "search_url_template": "https://www.healthyfood.com/?s={query}"
    }
}

test_queries = ["sushi", "sushi roll", "maki", "salmon sushi"]

candidate_source_rows = []

for source_name, config in ADDITIONAL_SOURCE_CANDIDATES.items():
    for query in test_queries:
        encoded_query = quote_plus(query)
        search_url = config["search_url_template"].format(query=encoded_query)

        print("\nTesting source:", source_name)
        print("Query:", query)
        print("URL:", search_url)

        response = safe_get(search_url, sleep_min=1, sleep_max=2, timeout=25)

        if response is None:
            row = {
                "source_name": source_name,
                "query": query,
                "status_code": None,
                "html_length": 0,
                "final_url": None,
                "works": False
            }
            print("No response.")
        else:
            row = {
                "source_name": source_name,
                "query": query,
                "status_code": response.status_code,
                "html_length": len(response.text),
                "final_url": response.url,
                "works": response.status_code == 200 and len(response.text) > 5000
            }

            print("Status code:", response.status_code)
            print("HTML length:", len(response.text))
            print("Final URL:", response.url)
            print("Works:", row["works"])

        candidate_source_rows.append(row)

candidate_source_test_df = pd.DataFrame(candidate_source_rows)

print("\nCandidate source test summary:")
display(candidate_source_test_df)

print("\nWorking sources summary:")
display(candidate_source_test_df[candidate_source_test_df["works"] == True])


Testing source: eatingwell
Query: sushi
URL: https://www.eatingwell.com/search?q=sushi
Status code: 402
HTML length: 612
Final URL: https://www.eatingwell.com/search?q=sushi
Works: False

Testing source: eatingwell
Query: sushi roll
URL: https://www.eatingwell.com/search?q=sushi+roll
Status code: 402
HTML length: 612
Final URL: https://www.eatingwell.com/search?q=sushi+roll
Works: False

Testing source: eatingwell
Query: maki
URL: https://www.eatingwell.com/search?q=maki
Status code: 402
HTML length: 612
Final URL: https://www.eatingwell.com/search?q=maki
Works: False

Testing source: eatingwell
Query: salmon sushi
URL: https://www.eatingwell.com/search?q=salmon+sushi
Status code: 402
HTML length: 612
Final URL: https://www.eatingwell.com/search?q=salmon+sushi
Works: False

Testing source: recipetineats
Query: sushi
URL: https://www.recipetineats.com/?s=sushi
Status code: 200
HTML length: 342953
Final URL: https://www.recipetineats.com/?s=sushi
Works: True

Testing source: recipetinea

,source_name,query,status_code,html_length,final_url,works
0,eatingwell,sushi,402,612,https://www.eatingwell.com/search?q=sushi,False
1,eatingwell,sushi roll,402,612,https://www.eatingwell.com/search?q=sushi+roll,False
2,eatingwell,maki,402,612,https://www.eatingwell.com/search?q=maki,False
3,eatingwell,salmon sushi,402,612,https://www.eatingwell.com/search?q=salmon+sushi,False
4,recipetineats,sushi,200,342953,https://www.recipetineats.com/?s=sushi,True
5,recipetineats,sushi roll,200,342434,https://www.recipetineats.com/?s=sushi+roll,True
6,recipetineats,maki,200,341958,https://www.recipetineats.com/?s=maki,True
7,recipetineats,salmon sushi,200,336297,https://www.recipetineats.com/?s=salmon+sushi,True
8,the_spruce_eats,sushi,402,612,https://www.thespruceeats.com/search?q=sushi,False
9,the_spruce_eats,sushi roll,402,612,https://www.thespruceeats.com/search?q=sushi+roll,False



Working sources summary:


,source_name,query,status_code,html_length,final_url,works
4,recipetineats,sushi,200,342953,https://www.recipetineats.com/?s=sushi,True
5,recipetineats,sushi roll,200,342434,https://www.recipetineats.com/?s=sushi+roll,True
6,recipetineats,maki,200,341958,https://www.recipetineats.com/?s=maki,True
7,recipetineats,salmon sushi,200,336297,https://www.recipetineats.com/?s=salmon+sushi,True
16,healthy_food_guide,sushi,200,245295,https://www.healthyfood.com/?s=sushi,True
17,healthy_food_guide,sushi roll,200,263352,https://www.healthyfood.com/?s=sushi+roll,True
18,healthy_food_guide,maki,200,217312,https://www.healthyfood.com/?s=maki,True
19,healthy_food_guide,salmon sushi,200,271262,https://www.healthyfood.com/?s=salmon+sushi,True


In [27]:
# Cell 25B — Extract sushi recipe links from RecipeTinEats and Healthy Food Guide

from urllib.parse import urlparse

WORKING_SUSHI_SOURCES = {
    "recipetineats": ADDITIONAL_SOURCE_CANDIDATES["recipetineats"],
    "healthy_food_guide": ADDITIONAL_SOURCE_CANDIDATES["healthy_food_guide"]
}

SUSHI_AUGMENT_QUERIES = [
    "sushi",
    "sushi roll",
    "sushi rolls",
    "maki",
    "california roll",
    "salmon sushi",
    "tuna sushi",
    "avocado sushi",
    "sushi rice"
]


def is_valid_recipetineats_recipe_url(url):
    """
    RecipeTinEats recipe URLs are usually simple post URLs.
    This filter removes category/tag/page/search URLs.
    """
    parsed = urlparse(url)
    netloc = parsed.netloc.lower()
    path = parsed.path.strip("/").lower()

    if "recipetineats.com" not in netloc:
        return False

    if not path:
        return False

    blocked_parts = [
        "category",
        "tag",
        "recipes",
        "recipe-index",
        "about",
        "contact",
        "search",
        "privacy",
        "newsletter",
        "cookbook",
        "round-ups",
        "collections"
    ]

    for blocked in blocked_parts:
        if path == blocked or path.startswith(blocked + "/"):
            return False

    if "?" in url:
        return False

    return True


def is_valid_healthy_food_guide_recipe_url(url):
    """
    Healthy Food Guide recipe URLs usually contain /recipe/.
    """
    parsed = urlparse(url)
    netloc = parsed.netloc.lower()
    path = parsed.path.strip("/").lower()

    if "healthyfood.com" not in netloc:
        return False

    if "/recipe/" in parsed.path.lower() or path.startswith("recipe/"):
        return True

    return False


def extract_sushi_links_from_new_source(source_name, query, max_links=30):
    """
    Extract candidate sushi recipe links from a search page.
    """
    config = WORKING_SUSHI_SOURCES[source_name]
    encoded_query = quote_plus(query)
    search_url = config["search_url_template"].format(query=encoded_query)

    response = safe_get(search_url, sleep_min=1, sleep_max=2, timeout=25)

    if response is None:
        print("No response returned.")
        return pd.DataFrame(columns=["source_name", "query", "title_text", "url"])

    print("\nSource:", source_name)
    print("Query:", query)
    print("Search URL:", search_url)
    print("Status code:", response.status_code)
    print("HTML length:", len(response.text))

    soup = BeautifulSoup(response.text, "html.parser")
    rows = []

    for a_tag in soup.find_all("a", href=True):
        href = a_tag.get("href")
        title_text = a_tag.get_text(" ", strip=True)

        if not href:
            continue

        full_url = urljoin(config["base_url"], href)

        if source_name == "recipetineats":
            is_valid = is_valid_recipetineats_recipe_url(full_url)

        elif source_name == "healthy_food_guide":
            is_valid = is_valid_healthy_food_guide_recipe_url(full_url)

        else:
            is_valid = False

        if not is_valid:
            continue

        title_clean = normalize_text_for_filtering(title_text)
        url_clean = normalize_text_for_filtering(full_url)
        combined = title_clean + " " + url_clean

        sushi_terms = [
            "sushi",
            "maki",
            "california roll",
            "salmon roll",
            "tuna roll",
            "avocado roll",
            "sushi rice"
        ]

        if not any(term in combined for term in sushi_terms):
            continue

        rows.append({
            "source_name": source_name,
            "query": query,
            "title_text": title_text,
            "url": full_url
        })

    links_df = pd.DataFrame(rows)

    if not links_df.empty:
        links_df = links_df.drop_duplicates(subset=["url"]).reset_index(drop=True)

    print("Candidate sushi links found:", len(links_df))

    if not links_df.empty:
        display(links_df.head(max_links))

    return links_df


all_new_sushi_link_dfs = []

for source_name in WORKING_SUSHI_SOURCES.keys():
    print("\n====================================")
    print("Extracting sushi links from:", source_name)
    print("====================================")

    for query in SUSHI_AUGMENT_QUERIES:
        links_df = extract_sushi_links_from_new_source(
            source_name=source_name,
            query=query,
            max_links=20
        )

        if links_df is not None and not links_df.empty:
            all_new_sushi_link_dfs.append(links_df)

        time.sleep(1)

if all_new_sushi_link_dfs:
    new_sushi_links_df = pd.concat(all_new_sushi_link_dfs, ignore_index=True)
    new_sushi_links_df = new_sushi_links_df.drop_duplicates(subset=["url"]).reset_index(drop=True)
else:
    new_sushi_links_df = pd.DataFrame(columns=["source_name", "query", "title_text", "url"])

print("\nNew sushi link extraction finished.")
print("Total unique sushi candidate links:", len(new_sushi_links_df))

print("\nLinks per source:")
if not new_sushi_links_df.empty:
    print(new_sushi_links_df["source_name"].value_counts())
    display(new_sushi_links_df)
else:
    print("No links found.")


Extracting sushi links from: recipetineats

Source: recipetineats
Query: sushi
Search URL: https://www.recipetineats.com/?s=sushi
Status code: 200
HTML length: 342953
Candidate sushi links found: 0

Source: recipetineats
Query: sushi roll
Search URL: https://www.recipetineats.com/?s=sushi+roll
Status code: 200
HTML length: 342434
Candidate sushi links found: 0

Source: recipetineats
Query: sushi rolls
Search URL: https://www.recipetineats.com/?s=sushi+rolls
Status code: 200
HTML length: 332514
Candidate sushi links found: 0

Source: recipetineats
Query: maki
Search URL: https://www.recipetineats.com/?s=maki
Status code: 200
HTML length: 341958
Candidate sushi links found: 2


,source_name,query,title_text,url
0,recipetineats,maki,,https://www.recipetineats.com/making-of-a-cook...
1,recipetineats,maki,,https://www.recipetineats.com/the-making-of-a-...



Source: recipetineats
Query: california roll
Search URL: https://www.recipetineats.com/?s=california+roll
Status code: 200
HTML length: 316025
Candidate sushi links found: 0

Source: recipetineats
Query: salmon sushi
Search URL: https://www.recipetineats.com/?s=salmon+sushi
Status code: 200
HTML length: 336297
Candidate sushi links found: 0

Source: recipetineats
Query: tuna sushi
Search URL: https://www.recipetineats.com/?s=tuna+sushi
Status code: 200
HTML length: 342189
Candidate sushi links found: 0

Source: recipetineats
Query: avocado sushi
Search URL: https://www.recipetineats.com/?s=avocado+sushi
Status code: 200
HTML length: 326894
Candidate sushi links found: 0

Source: recipetineats
Query: sushi rice
Search URL: https://www.recipetineats.com/?s=sushi+rice
Status code: 200
HTML length: 344442
Candidate sushi links found: 0

Extracting sushi links from: healthy_food_guide

Source: healthy_food_guide
Query: sushi
Search URL: https://www.healthyfood.com/?s=sushi
Status code: 200

,source_name,query,title_text,url
0,recipetineats,maki,,https://www.recipetineats.com/making-of-a-cook...
1,recipetineats,maki,,https://www.recipetineats.com/the-making-of-a-...


In [28]:
# Cell 25C — Test specialized sushi/Japanese recipe sources

SPECIALIZED_SUSHI_SOURCE_CANDIDATES = {
    "just_one_cookbook": {
        "base_url": "https://www.justonecookbook.com",
        "search_url_template": "https://www.justonecookbook.com/?s={query}"
    },
    "pickled_plum": {
        "base_url": "https://pickledplum.com",
        "search_url_template": "https://pickledplum.com/?s={query}"
    },
    "chopstick_chronicles": {
        "base_url": "https://www.chopstickchronicles.com",
        "search_url_template": "https://www.chopstickchronicles.com/?s={query}"
    },
    "okonomi_kitchen": {
        "base_url": "https://www.okonomikitchen.com",
        "search_url_template": "https://www.okonomikitchen.com/?s={query}"
    },
    "no_recipes": {
        "base_url": "https://norecipes.com",
        "search_url_template": "https://norecipes.com/?s={query}"
    }
}

specialized_test_queries = [
    "sushi",
    "sushi roll",
    "maki",
    "california roll",
    "salmon sushi",
    "sushi rice"
]

specialized_source_rows = []

for source_name, config in SPECIALIZED_SUSHI_SOURCE_CANDIDATES.items():
    for query in specialized_test_queries:
        encoded_query = quote_plus(query)
        search_url = config["search_url_template"].format(query=encoded_query)

        print("\nTesting source:", source_name)
        print("Query:", query)
        print("URL:", search_url)

        response = safe_get(search_url, sleep_min=1, sleep_max=2, timeout=25)

        if response is None:
            row = {
                "source_name": source_name,
                "query": query,
                "status_code": None,
                "html_length": 0,
                "final_url": None,
                "works": False
            }
            print("No response.")
        else:
            row = {
                "source_name": source_name,
                "query": query,
                "status_code": response.status_code,
                "html_length": len(response.text),
                "final_url": response.url,
                "works": response.status_code == 200 and len(response.text) > 5000
            }

            print("Status code:", response.status_code)
            print("HTML length:", len(response.text))
            print("Final URL:", response.url)
            print("Works:", row["works"])

        specialized_source_rows.append(row)

specialized_source_test_df = pd.DataFrame(specialized_source_rows)

print("\nSpecialized source test summary:")
display(specialized_source_test_df)

print("\nWorking specialized sources:")
display(specialized_source_test_df[specialized_source_test_df["works"] == True])


Testing source: just_one_cookbook
Query: sushi
URL: https://www.justonecookbook.com/?s=sushi
Status code: 403
HTML length: 5054
Final URL: https://www.justonecookbook.com/?s=sushi
Works: False

Testing source: just_one_cookbook
Query: sushi roll
URL: https://www.justonecookbook.com/?s=sushi+roll
Status code: 403
HTML length: 5054
Final URL: https://www.justonecookbook.com/?s=sushi+roll
Works: False

Testing source: just_one_cookbook
Query: maki
URL: https://www.justonecookbook.com/?s=maki
Status code: 403
HTML length: 5054
Final URL: https://www.justonecookbook.com/?s=maki
Works: False

Testing source: just_one_cookbook
Query: california roll
URL: https://www.justonecookbook.com/?s=california+roll
Status code: 403
HTML length: 5054
Final URL: https://www.justonecookbook.com/?s=california+roll
Works: False

Testing source: just_one_cookbook
Query: salmon sushi
URL: https://www.justonecookbook.com/?s=salmon+sushi
Status code: 403
HTML length: 5054
Final URL: https://www.justonecookbook.

,source_name,query,status_code,html_length,final_url,works
0,just_one_cookbook,sushi,403,5054,https://www.justonecookbook.com/?s=sushi,False
1,just_one_cookbook,sushi roll,403,5054,https://www.justonecookbook.com/?s=sushi+roll,False
2,just_one_cookbook,maki,403,5054,https://www.justonecookbook.com/?s=maki,False
3,just_one_cookbook,california roll,403,5054,https://www.justonecookbook.com/?s=california+...,False
4,just_one_cookbook,salmon sushi,403,5054,https://www.justonecookbook.com/?s=salmon+sushi,False
5,just_one_cookbook,sushi rice,403,5054,https://www.justonecookbook.com/?s=sushi+rice,False
6,pickled_plum,sushi,200,328900,https://pickledplum.com/?s=sushi,True
7,pickled_plum,sushi roll,200,326288,https://pickledplum.com/?s=sushi+roll,True
8,pickled_plum,maki,200,324724,https://pickledplum.com/?s=maki,True
9,pickled_plum,california roll,200,310729,https://pickledplum.com/?s=california+roll,True



Working specialized sources:


,source_name,query,status_code,html_length,final_url,works
6,pickled_plum,sushi,200,328900,https://pickledplum.com/?s=sushi,True
7,pickled_plum,sushi roll,200,326288,https://pickledplum.com/?s=sushi+roll,True
8,pickled_plum,maki,200,324724,https://pickledplum.com/?s=maki,True
9,pickled_plum,california roll,200,310729,https://pickledplum.com/?s=california+roll,True
10,pickled_plum,salmon sushi,200,306242,https://pickledplum.com/?s=salmon+sushi,True
11,pickled_plum,sushi rice,200,326657,https://pickledplum.com/?s=sushi+rice,True
12,chopstick_chronicles,sushi,200,371379,https://www.chopstickchronicles.com/?s=sushi,True
13,chopstick_chronicles,sushi roll,200,372690,https://www.chopstickchronicles.com/?s=sushi+roll,True
14,chopstick_chronicles,maki,200,376528,https://www.chopstickchronicles.com/?s=maki,True
15,chopstick_chronicles,california roll,200,320942,https://www.chopstickchronicles.com/?s=califor...,True


In [29]:
# Cell 25D — Extract sushi recipe links from specialized sources

SPECIALIZED_WORKING_SUSHI_SOURCES = {
    "pickled_plum": SPECIALIZED_SUSHI_SOURCE_CANDIDATES["pickled_plum"],
    "chopstick_chronicles": SPECIALIZED_SUSHI_SOURCE_CANDIDATES["chopstick_chronicles"],
    "okonomi_kitchen": SPECIALIZED_SUSHI_SOURCE_CANDIDATES["okonomi_kitchen"],
    "no_recipes": SPECIALIZED_SUSHI_SOURCE_CANDIDATES["no_recipes"]
}

SPECIALIZED_SUSHI_QUERIES = [
    "sushi",
    "sushi roll",
    "sushi rolls",
    "maki",
    "california roll",
    "salmon sushi",
    "tuna sushi",
    "avocado sushi",
    "sushi rice",
    "spicy tuna roll",
    "salmon roll"
]


def is_valid_specialized_recipe_url(source_name, url):
    """
    Filter obvious non-recipe URLs from specialized recipe sites.
    """
    if url is None:
        return False

    parsed = urlparse(url)
    netloc = parsed.netloc.lower()
    path = parsed.path.strip("/").lower()

    if not path:
        return False

    expected_domains = {
        "pickled_plum": "pickledplum.com",
        "chopstick_chronicles": "chopstickchronicles.com",
        "okonomi_kitchen": "okonomikitchen.com",
        "no_recipes": "norecipes.com"
    }

    expected_domain = expected_domains.get(source_name)

    if expected_domain is None or expected_domain not in netloc:
        return False

    blocked_parts = [
        "category",
        "tag",
        "author",
        "page",
        "about",
        "contact",
        "privacy",
        "terms",
        "shop",
        "store",
        "newsletter",
        "cookbook",
        "recipe-index",
        "recipes",
        "search",
        "wp-content",
        "comments",
        "resources"
    ]

    for blocked in blocked_parts:
        if path == blocked or path.startswith(blocked + "/") or ("/" + blocked + "/") in ("/" + path + "/"):
            return False

    blocked_contains = [
        "making-of-a-cookbook",
        "the-making-of-a",
        "best-",
        "roundup",
        "round-up",
        "collection"
    ]

    for blocked in blocked_contains:
        if blocked in path:
            return False

    return True


def looks_like_sushi_recipe(title_text, url, query):
    """
    Decide whether a candidate URL/title is sushi-related.
    """
    title_clean = normalize_text_for_filtering(title_text)
    url_clean = normalize_text_for_filtering(url)
    query_clean = normalize_text_for_filtering(query)

    combined = " ".join([title_clean, url_clean, query_clean])

    strong_sushi_terms = [
        "sushi",
        "maki",
        "nigiri",
        "sashimi",
        "temaki",
        "futomaki",
        "uramaki",
        "california roll",
        "spicy tuna roll",
        "salmon roll",
        "tuna roll",
        "avocado roll",
        "sushi rice"
    ]

    if any(term in combined for term in strong_sushi_terms):
        return True

    return False


def extract_specialized_sushi_links(source_name, query, max_links=30):
    """
    Extract candidate sushi recipe links from one specialized source search page.
    """
    config = SPECIALIZED_WORKING_SUSHI_SOURCES[source_name]
    encoded_query = quote_plus(query)
    search_url = config["search_url_template"].format(query=encoded_query)

    response = safe_get(search_url, sleep_min=1, sleep_max=2, timeout=25)

    if response is None:
        print("No response returned.")
        return pd.DataFrame(columns=["source_name", "query", "title_text", "url"])

    print("\nSource:", source_name)
    print("Query:", query)
    print("Search URL:", search_url)
    print("Status code:", response.status_code)
    print("HTML length:", len(response.text))

    soup = BeautifulSoup(response.text, "html.parser")
    rows = []

    for a_tag in soup.find_all("a", href=True):
        href = a_tag.get("href")
        title_text = a_tag.get_text(" ", strip=True)

        if not href:
            continue

        full_url = urljoin(config["base_url"], href)

        if not is_valid_specialized_recipe_url(source_name, full_url):
            continue

        if not looks_like_sushi_recipe(title_text, full_url, query):
            continue

        rows.append({
            "source_name": source_name,
            "query": query,
            "title_text": title_text,
            "url": full_url
        })

    links_df = pd.DataFrame(rows)

    if not links_df.empty:
        links_df = links_df.drop_duplicates(subset=["url"]).reset_index(drop=True)

    print("Candidate sushi links found:", len(links_df))

    if not links_df.empty:
        display(links_df.head(max_links))

    return links_df


specialized_sushi_link_dfs = []

for source_name in SPECIALIZED_WORKING_SUSHI_SOURCES.keys():
    print("\n====================================")
    print("Extracting sushi links from:", source_name)
    print("====================================")

    for query in SPECIALIZED_SUSHI_QUERIES:
        links_df = extract_specialized_sushi_links(
            source_name=source_name,
            query=query,
            max_links=20
        )

        if links_df is not None and not links_df.empty:
            specialized_sushi_link_dfs.append(links_df)

        time.sleep(1)

if specialized_sushi_link_dfs:
    specialized_sushi_links_df = pd.concat(specialized_sushi_link_dfs, ignore_index=True)
    specialized_sushi_links_df = specialized_sushi_links_df.drop_duplicates(subset=["url"]).reset_index(drop=True)
else:
    specialized_sushi_links_df = pd.DataFrame(columns=["source_name", "query", "title_text", "url"])

print("\nSpecialized sushi link extraction finished.")
print("Total unique specialized sushi candidate links:", len(specialized_sushi_links_df))

print("\nLinks per source:")
if not specialized_sushi_links_df.empty:
    print(specialized_sushi_links_df["source_name"].value_counts())
    display(specialized_sushi_links_df)
else:
    print("No links found.")


Extracting sushi links from: pickled_plum

Source: pickled_plum
Query: sushi
Search URL: https://pickledplum.com/?s=sushi
Status code: 200
HTML length: 328900
Candidate sushi links found: 98


,source_name,query,title_text,url
0,pickled_plum,sushi,All Recipes,https://pickledplum.com/recipe-filter/
1,pickled_plum,sushi,Accessibility,https://pickledplum.com/accessibility/
2,pickled_plum,sushi,Collaborate,https://pickledplum.com/collaborate/
3,pickled_plum,sushi,Content Guidelines,https://pickledplum.com/sharing-content-images...
4,pickled_plum,sushi,No AI,https://pickledplum.com/no-ai/
5,pickled_plum,sushi,Privacy Policy,https://pickledplum.com/privacy-policy/
6,pickled_plum,sushi,,https://pickledplum.com/how-to-make-sushi-at-h...
7,pickled_plum,sushi,Comment,https://pickledplum.com/how-to-make-sushi-at-h...
8,pickled_plum,sushi,,https://pickledplum.com/how-to-make-sushi-rice/
9,pickled_plum,sushi,5 Comments,https://pickledplum.com/how-to-make-sushi-rice...



Source: pickled_plum
Query: sushi roll
Search URL: https://pickledplum.com/?s=sushi+roll
Status code: 200
HTML length: 326288
Candidate sushi links found: 94


,source_name,query,title_text,url
0,pickled_plum,sushi roll,All Recipes,https://pickledplum.com/recipe-filter/
1,pickled_plum,sushi roll,Accessibility,https://pickledplum.com/accessibility/
2,pickled_plum,sushi roll,Collaborate,https://pickledplum.com/collaborate/
3,pickled_plum,sushi roll,Content Guidelines,https://pickledplum.com/sharing-content-images...
4,pickled_plum,sushi roll,No AI,https://pickledplum.com/no-ai/
5,pickled_plum,sushi roll,Privacy Policy,https://pickledplum.com/privacy-policy/
6,pickled_plum,sushi roll,,https://pickledplum.com/maki-sushi/
7,pickled_plum,sushi roll,Comment,https://pickledplum.com/maki-sushi/#commentform
8,pickled_plum,sushi roll,,https://pickledplum.com/how-to-make-sushi-at-h...
9,pickled_plum,sushi roll,Comment,https://pickledplum.com/how-to-make-sushi-at-h...



Source: pickled_plum
Query: sushi rolls
Search URL: https://pickledplum.com/?s=sushi+rolls
Status code: 200
HTML length: 325087
Candidate sushi links found: 94


...

,source_name,query,title_text,url
0,no_recipes,spicy tuna roll,Start Here,https://norecipes.com/start-here/
1,no_recipes,spicy tuna roll,Ingredients,https://norecipes.com/articles/ingredient/
2,no_recipes,spicy tuna roll,subscribe,https://norecipes.com/subscribe/
3,no_recipes,spicy tuna roll,Spicy Tuna Roll,https://norecipes.com/spicy-tuna-roll/
4,no_recipes,spicy tuna roll,Oshinko Roll (Pickled Vegetable Sushi),https://norecipes.com/oshinko-roll-maki-sushi/
5,no_recipes,spicy tuna roll,California Roll,https://norecipes.com/california-roll-recipe/
6,no_recipes,spicy tuna roll,Kappa Maki (Cucumber Sushi Rolls),https://norecipes.com/kappa-maki-cucumber-sush...
7,no_recipes,spicy tuna roll,Shrimp Tempura Roll,https://norecipes.com/shrimp-tempura-sushi-roll/
8,no_recipes,spicy tuna roll,Spicy Salmon Temaki Sushi,https://norecipes.com/spicy-salmon-sushi-hand-...
9,no_recipes,spicy tuna roll,California Roll Sushi Bowl,https://norecipes.com/california-roll-sushi-bowl/



Source: no_recipes
Query: salmon roll
Search URL: https://norecipes.com/?s=salmon+roll
Status code: 200
HTML length: 436782
Candidate sushi links found: 54


,source_name,query,title_text,url
0,no_recipes,salmon roll,Start Here,https://norecipes.com/start-here/
1,no_recipes,salmon roll,Ingredients,https://norecipes.com/articles/ingredient/
2,no_recipes,salmon roll,subscribe,https://norecipes.com/subscribe/
3,no_recipes,salmon roll,Salmon Onigiri,https://norecipes.com/salmon-onigiri-rice-balls/
4,no_recipes,salmon roll,Spicy Salmon Temaki Sushi,https://norecipes.com/spicy-salmon-sushi-hand-...
5,no_recipes,salmon roll,Japanese Breakfast Salmon,https://norecipes.com/japanese-breakfast-salmon/
6,no_recipes,salmon roll,Temaki Sushi Hand Rolls,https://norecipes.com/temaki-sushi/
7,no_recipes,salmon roll,Caterpillar Roll,https://norecipes.com/caterpillar-roll-recipe/
8,no_recipes,salmon roll,Dashimaki Tamago,https://norecipes.com/dashimaki-tamago/
9,no_recipes,salmon roll,Perfect Sushi Rice,https://norecipes.com/sushi-rice-recipe/



Specialized sushi link extraction finished.
Total unique specialized sushi candidate links: 714

Links per source:
source_name
pickled_plum            314
chopstick_chronicles    232
no_recipes              120
okonomi_kitchen          48
Name: count, dtype: int64


,source_name,query,title_text,url
0,pickled_plum,sushi,All Recipes,https://pickledplum.com/recipe-filter/
1,pickled_plum,sushi,Accessibility,https://pickledplum.com/accessibility/
2,pickled_plum,sushi,Collaborate,https://pickledplum.com/collaborate/
3,pickled_plum,sushi,Content Guidelines,https://pickledplum.com/sharing-content-images...
4,pickled_plum,sushi,No AI,https://pickledplum.com/no-ai/
...,...,...,...,...
709,no_recipes,salmon roll,Nasu Dengaku (Miso Glazed Eggplant),https://norecipes.com/miso-glazed-eggplant-nas...
710,no_recipes,salmon roll,Mushroom Doria (Mushroom Rice Casserole),https://norecipes.com/mushroom-doria-rice-cass...
711,no_recipes,salmon roll,Cabbage & Chicken Stew,https://norecipes.com/cabbage-chicken-stew/
712,no_recipes,salmon roll,Better Chocolate Chip Cookies,https://norecipes.com/better-chocolate-chip-co...


In [30]:
# Cell 25E — Refine specialized sushi candidate links before scraping

def clean_candidate_url(url):
    """
    Remove fragments and query strings from URLs.
    """
    if pd.isna(url):
        return ""

    parsed = urlparse(str(url))
    clean_url = f"{parsed.scheme}://{parsed.netloc}{parsed.path}"

    if clean_url.endswith("#"):
        clean_url = clean_url[:-1]

    return clean_url


def is_refined_sushi_recipe_link(row):
    """
    Keep only direct sushi dish recipe links.
    Remove navigation pages, comment links, policy pages, and side/helper recipes.
    """
    source_name = row["source_name"]
    title = normalize_text_for_filtering(row.get("title_text", ""))
    url = clean_candidate_url(row["url"])
    path = urlparse(url).path.strip("/").lower()

    combined = normalize_text_for_filtering(title + " " + path)

    if not path:
        return False

    blocked_terms = [
        "accessibility",
        "privacy",
        "policy",
        "terms",
        "subscribe",
        "start-here",
        "ingredient",
        "ingredients",
        "recipe-filter",
        "recipe-index",
        "all-recipes",
        "collaborate",
        "content-guidelines",
        "no-ai",
        "comment",
        "comments",
        "category",
        "tag",
        "author",
        "shop",
        "cookbook",
        "newsletter",
        "about",
        "contact",
        "quick",
        "main-dish",
        "dessert",
        "kitchen-tools",
        "pantry",
        "appetizer",
        "beverage",
        "bread",
        "breakfast",
        "condiment",
        "salad",
        "side-dish",
        "soup-stew",
        "bento",
        "curry",
        "donburi",
        "dumpling",
        "hot-pot"
    ]

    for term in blocked_terms:
        if term in combined:
            return False

    # Remove helper/side items, not direct sushi dishes
    helper_terms = [
        "sushi-rice",
        "sushi rice",
        "seasoned-sushi-vinegar",
        "sushi vinegar",
        "sushi-ginger",
        "pickled sushi ginger",
        "gari",
        "seaweed-salad",
        "kani-salad",
        "ikura",
        "how-to",
        "what-is",
        "guide",
        "leftover",
        "miso-soup",
        "fried-rice",
        "omelette",
        "tamago",
        "ramen",
        "teriyaki",
        "spring-roll",
        "beef-negimaki",
        "sandwich",
        "onigiri",
        "poke",
        "bowl"
    ]

    for term in helper_terms:
        if term in combined:
            return False

    # Strong direct sushi dish terms
    direct_sushi_terms = [
        "sushi",
        "maki",
        "roll",
        "temaki",
        "futomaki",
        "uramaki",
        "nigiri",
        "chirashi",
        "inari"
    ]

    has_direct_sushi_term = any(term in combined for term in direct_sushi_terms)

    if not has_direct_sushi_term:
        return False

    # Extra source-specific cleanup
    if source_name == "okonomi_kitchen":
        # This source had many weak/non-sushi results in the search output.
        # Keep only if the URL/title is very explicit.
        okonomi_strong_terms = [
            "sushi",
            "maki",
            "roll",
            "temaki",
            "inari"
        ]
        if not any(term in combined for term in okonomi_strong_terms):
            return False

    return True


refined_specialized_sushi_links_df = specialized_sushi_links_df.copy()

refined_specialized_sushi_links_df["url"] = refined_specialized_sushi_links_df["url"].apply(clean_candidate_url)

refined_specialized_sushi_links_df = refined_specialized_sushi_links_df.drop_duplicates(
    subset=["url"]
).reset_index(drop=True)

refined_specialized_sushi_links_df["is_refined_sushi_recipe"] = refined_specialized_sushi_links_df.apply(
    is_refined_sushi_recipe_link,
    axis=1
)

rejected_specialized_sushi_links_df = refined_specialized_sushi_links_df[
    refined_specialized_sushi_links_df["is_refined_sushi_recipe"] == False
].copy()

refined_specialized_sushi_links_df = refined_specialized_sushi_links_df[
    refined_specialized_sushi_links_df["is_refined_sushi_recipe"] == True
].copy()

refined_specialized_sushi_links_df = refined_specialized_sushi_links_df.drop(
    columns=["is_refined_sushi_recipe"],
    errors="ignore"
).reset_index(drop=True)

print("Original specialized sushi links:")
print(len(specialized_sushi_links_df))

print("\nUnique links after URL cleaning:")
print(len(refined_specialized_sushi_links_df) + len(rejected_specialized_sushi_links_df))

print("\nRefined specialized sushi recipe links:")
print(len(refined_specialized_sushi_links_df))

print("\nRejected specialized sushi links:")
print(len(rejected_specialized_sushi_links_df))

print("\nRefined links per source:")
if not refined_specialized_sushi_links_df.empty:
    print(refined_specialized_sushi_links_df["source_name"].value_counts())
    display(refined_specialized_sushi_links_df)
else:
    print("No refined links found.")

print("\nRejected examples:")
display(rejected_specialized_sushi_links_df[[
    "source_name",
    "query",
    "title_text",
    "url"
]].head(30))

Original specialized sushi links:
714

Unique links after URL cleaning:
469

Refined specialized sushi recipe links:
47

Rejected specialized sushi links:
422

Refined links per source:
source_name
chopstick_chronicles    24
no_recipes              12
pickled_plum             8
okonomi_kitchen          3
Name: count, dtype: int64


,source_name,query,title_text,url
0,pickled_plum,sushi,,https://pickledplum.com/maki-sushi/
1,pickled_plum,sushi,,https://pickledplum.com/temari-sushi/
2,pickled_plum,sushi,,https://pickledplum.com/inari-sushi-recipe/
3,pickled_plum,sushi,,https://pickledplum.com/nigiri-vs-sushi-vs-mus...
4,pickled_plum,sushi,,https://pickledplum.com/vegetarian-chirashi-su...
5,pickled_plum,sushi,,https://pickledplum.com/philadelphia-roll/
6,pickled_plum,sushi,,https://pickledplum.com/spicy-tuna-roll-recipe/
7,pickled_plum,sushi roll,,https://pickledplum.com/california-roll-spicy-...
8,chopstick_chronicles,sushi,Sushi,https://www.chopstickchronicles.com/sushi/
9,chopstick_chronicles,sushi,,https://www.chopstickchronicles.com/temari-sushi/



Rejected examples:


,source_name,query,title_text,url
0,pickled_plum,sushi,All Recipes,https://pickledplum.com/recipe-filter/
1,pickled_plum,sushi,Accessibility,https://pickledplum.com/accessibility/
2,pickled_plum,sushi,Collaborate,https://pickledplum.com/collaborate/
3,pickled_plum,sushi,Content Guidelines,https://pickledplum.com/sharing-content-images...
4,pickled_plum,sushi,No AI,https://pickledplum.com/no-ai/
5,pickled_plum,sushi,Privacy Policy,https://pickledplum.com/privacy-policy/
6,pickled_plum,sushi,,https://pickledplum.com/how-to-make-sushi-at-h...
7,pickled_plum,sushi,,https://pickledplum.com/how-to-make-sushi-rice/
13,pickled_plum,sushi,,https://pickledplum.com/what-to-do-with-leftov...
14,pickled_plum,sushi,,https://pickledplum.com/daikon-miso-soup/


In [31]:
# Cell 25F — Strict sushi link refinement v2

def is_strict_direct_sushi_link(row):
    """
    Keep only direct sushi dish recipe links.
    This removes guides, comparison articles, helper recipes, desserts, and weak hybrids.
    """
    source_name = row["source_name"]
    title = normalize_text_for_filtering(row.get("title_text", ""))
    url = clean_candidate_url(row["url"])
    path = urlparse(url).path.strip("/").lower()

    combined = normalize_text_for_filtering(title + " " + path)

    if not path:
        return False

    hard_reject_terms = [
        "accessibility",
        "privacy",
        "policy",
        "subscribe",
        "start-here",
        "ingredient",
        "recipe-filter",
        "tips",
        "guide",
        "vs",
        "how-to",
        "making",
        "crepes",
        "cake",
        "christmas",
        "donut",
        "donuts",
        "wild-rice",
        "harumaki",
        "nama-harumaki",
        "datemaki",
        "soba",
        "inari-age",
        "take-away",
        "take away",
        "sushi-rice",
        "sushi rice",
        "seasoned-sushi-vinegar",
        "vinegar",
        "ginger",
        "gari",
        "seaweed-salad",
        "kani-salad",
        "ikura",
        "onigiri",
        "bowl",
        "donburi",
        "sashimi-donburi"
    ]

    for term in hard_reject_terms:
        if term in combined:
            return False

    direct_keep_terms = [
        "maki-sushi",
        "maki sushi",
        "temari-sushi",
        "temari sushi",
        "inari-sushi",
        "inari sushi",
        "chirashi-sushi",
        "chirashi sushi",
        "seafood-chirashi-sushi",
        "seafood chirashi sushi",
        "unagi-sushi",
        "unagi sushi",
        "temaki-sushi",
        "temaki sushi",
        "kappa-maki",
        "kappa maki",
        "oshinko-roll",
        "oshinko roll",
        "california-roll",
        "california roll",
        "pressed-sushi",
        "pressed sushi",
        "futomaki",
        "uramaki",
        "hosomaki",
        "gunkan-sushi",
        "gunkan sushi",
        "shrimp-tempura-sushi-roll",
        "shrimp tempura roll",
        "spicy-tuna-roll",
        "spicy tuna roll",
        "spicy-salmon-sushi-hand-rolls",
        "spicy salmon temaki sushi",
        "caterpillar-roll",
        "caterpillar roll",
        "philadelphia-roll",
        "philadelphia roll"
    ]

    if any(term in combined for term in direct_keep_terms):
        return True

    # Keep very explicit sushi pages only, but avoid broad generic pages from weak sources.
    if source_name in ["no_recipes", "pickled_plum", "chopstick_chronicles"]:
        if combined in ["sushi", "sushi sushi"]:
            return True

    return False


strict_sushi_links_v2_df = refined_specialized_sushi_links_df.copy()

strict_sushi_links_v2_df["url"] = strict_sushi_links_v2_df["url"].apply(clean_candidate_url)

strict_sushi_links_v2_df["is_strict_direct_sushi"] = strict_sushi_links_v2_df.apply(
    is_strict_direct_sushi_link,
    axis=1
)

strict_rejected_sushi_links_v2_df = strict_sushi_links_v2_df[
    strict_sushi_links_v2_df["is_strict_direct_sushi"] == False
].copy()

strict_sushi_links_v2_df = strict_sushi_links_v2_df[
    strict_sushi_links_v2_df["is_strict_direct_sushi"] == True
].copy()

strict_sushi_links_v2_df = strict_sushi_links_v2_df.drop(
    columns=["is_strict_direct_sushi"],
    errors="ignore"
).drop_duplicates(subset=["url"]).reset_index(drop=True)

print("Refined v1 sushi links:")
print(len(refined_specialized_sushi_links_df))

print("\nStrict sushi links v2:")
print(len(strict_sushi_links_v2_df))

print("\nRejected by strict v2:")
print(len(strict_rejected_sushi_links_v2_df))

print("\nStrict sushi links per source:")
if not strict_sushi_links_v2_df.empty:
    print(strict_sushi_links_v2_df["source_name"].value_counts())
    display(strict_sushi_links_v2_df)
else:
    print("No strict links found.")

print("\nRejected examples from v2:")
display(strict_rejected_sushi_links_v2_df[[
    "source_name",
    "query",
    "title_text",
    "url"
]].head(40))

Refined v1 sushi links:
47

Strict sushi links v2:
31

Rejected by strict v2:
16

Strict sushi links per source:
source_name
chopstick_chronicles    14
no_recipes              10
pickled_plum             7
Name: count, dtype: int64


,source_name,query,title_text,url
0,pickled_plum,sushi,,https://pickledplum.com/maki-sushi/
1,pickled_plum,sushi,,https://pickledplum.com/temari-sushi/
2,pickled_plum,sushi,,https://pickledplum.com/inari-sushi-recipe/
3,pickled_plum,sushi,,https://pickledplum.com/vegetarian-chirashi-su...
4,pickled_plum,sushi,,https://pickledplum.com/philadelphia-roll/
5,pickled_plum,sushi,,https://pickledplum.com/spicy-tuna-roll-recipe/
6,pickled_plum,sushi roll,,https://pickledplum.com/california-roll-spicy-...
7,chopstick_chronicles,sushi,Sushi,https://www.chopstickchronicles.com/sushi/
8,chopstick_chronicles,sushi,,https://www.chopstickchronicles.com/temari-sushi/
9,chopstick_chronicles,sushi,,https://www.chopstickchronicles.com/gunkan-sus...



Rejected examples from v2:


,source_name,query,title_text,url
3,pickled_plum,sushi,,https://pickledplum.com/nigiri-vs-sushi-vs-mus...
16,chopstick_chronicles,sushi,,https://www.chopstickchronicles.com/sushi-cake/
19,chopstick_chronicles,sushi,,https://www.chopstickchronicles.com/christmas-...
21,chopstick_chronicles,sushi,,https://www.chopstickchronicles.com/ehoumaki/
22,chopstick_chronicles,sushi,,https://www.chopstickchronicles.com/sushi-ball...
23,chopstick_chronicles,sushi,,https://www.chopstickchronicles.com/sushi-donuts/
24,chopstick_chronicles,sushi,,https://www.chopstickchronicles.com/chirashizu...
27,chopstick_chronicles,sushi roll,,https://www.chopstickchronicles.com/datemaki/
29,chopstick_chronicles,sushi roll,,https://www.chopstickchronicles.com/wild-rice-...
30,chopstick_chronicles,maki,,https://www.chopstickchronicles.com/nama-harum...


In [32]:
# Cell 25G — Test scraping one sushi recipe page from each specialized source

def scrape_specialized_sushi_recipe(url, source_site, food_class="sushi", query=None):
    """
    Scrape one specialized sushi recipe page using Recipe JSON-LD when available.
    Returns a structured recipe record if nutrition exists.
    """
    response = safe_get(url, sleep_min=1, sleep_max=2, timeout=25)

    if response is None:
        print("No response for:", url)
        return None

    if response.status_code != 200:
        print("Failed URL:", url)
        print("Status code:", response.status_code)
        return None

    soup = BeautifulSoup(response.text, "html.parser")
    recipe_json = find_recipe_json_ld(soup)

    if recipe_json is None:
        print("No Recipe JSON-LD found for:", url)
        return None

    recipe_name = recipe_json.get("name")

    ingredients = recipe_json.get("recipeIngredient", [])
    if isinstance(ingredients, list):
        ingredients_text = " ".join([str(item) for item in ingredients])
    else:
        ingredients_text = str(ingredients)

    nutrition = recipe_json.get("nutrition", {})
    if not isinstance(nutrition, dict):
        nutrition = {}

    calories = extract_number(nutrition.get("calories"))
    protein = extract_number(nutrition.get("proteinContent"))
    fat = extract_number(nutrition.get("fatContent"))
    carbs = extract_number(nutrition.get("carbohydrateContent"))

    record = {
        "source_site": source_site,
        "food_class": food_class,
        "query": query,
        "recipe_name": recipe_name,
        "ingredients_text": ingredients_text,
        "calories": calories,
        "protein": protein,
        "fat": fat,
        "carbs": carbs,
        "source_url": url,
        "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    return record


# Test one URL from each source
test_source_records = []

for source_name in strict_sushi_links_v2_df["source_name"].unique():
    print("\n====================================")
    print("Testing source:", source_name)
    print("====================================")

    sample_row = strict_sushi_links_v2_df[
        strict_sushi_links_v2_df["source_name"] == source_name
    ].iloc[0]

    test_url = sample_row["url"]
    test_query = sample_row["query"]

    print("Test URL:")
    print(test_url)

    test_record = scrape_specialized_sushi_recipe(
        url=test_url,
        source_site=source_name,
        food_class="sushi",
        query=test_query
    )

    print("\nScraped record:")
    print(test_record)

    if test_record is not None:
        print("\nIngredients length:", len(test_record["ingredients_text"]) if test_record["ingredients_text"] else 0)
        print("Calories:", test_record["calories"])
        print("Protein:", test_record["protein"])
        print("Fat:", test_record["fat"])
        print("Carbs:", test_record["carbs"])

        test_source_records.append(test_record)

test_specialized_sushi_df = pd.DataFrame(test_source_records)

print("\nTest scraping summary:")
if not test_specialized_sushi_df.empty:
    display(test_specialized_sushi_df[[
        "source_site",
        "recipe_name",
        "calories",
        "protein",
        "fat",
        "carbs",
        "source_url"
    ]])
else:
    print("No successful test records.")


Testing source: pickled_plum
Test URL:
https://pickledplum.com/maki-sushi/

Scraped record:
{'source_site': 'pickled_plum', 'food_class': 'sushi', 'query': 'sushi', 'recipe_name': 'Maki Sushi ', 'ingredients_text': '4 cups sushi rice &#8211; visit my post on how to make sushi rice Persian or Kirby cucumber, cut into 1/2-inch long strips Takuan (pickled daikon radish), cut into 1/2-inch long strips Avocado, pitted, cut into 1/2-inch long strips Cooked sweet potato (don&#8217;t overcook it to the point where it&#8217;s mushy), cut into 1/2-inch long strips. Natto (1 box) Sashimi grade tuna, salmon, or yellowtail (about 5 oz for 8 rolls), cut into 1/2-inch long strips 4 nori sheets Soy sauce, for dipping Wasabi (optional) Pickled ginger (optional)', 'calories': 150.0, 'protein': 2.5, 'fat': 1.5, 'carbs': 30.8, 'source_url': 'https://pickledplum.com/maki-sushi/', 'scraped_at': '2026-05-12 03:08:36'}

Ingredients length: 538
Calories: 150.0
Protein: 2.5
Fat: 1.5
Carbs: 30.8

Testing source

,source_site,recipe_name,calories,protein,fat,carbs,source_url
0,pickled_plum,Maki Sushi,150.0,2.5,1.5,30.8,https://pickledplum.com/maki-sushi/
1,no_recipes,Inari Sushi (Fried Tofu Pocket Sushi),376.0,9.0,6.0,67.0,https://norecipes.com/inari-sushi-recipe/


In [33]:
# Cell 25H — Scrape strict sushi supplement records from specialized sources

def sushi_specialized_reject_reason(record):
    """
    Reject weak sushi records after scraping.
    Keeps direct sushi dishes only.
    """
    recipe_name = normalize_text_for_filtering(record.get("recipe_name"))
    source_url = normalize_text_for_filtering(record.get("source_url"))
    combined = recipe_name + " " + source_url

    required_cols = ["calories", "protein", "fat", "carbs"]

    for col in required_cols:
        if record.get(col) is None or pd.isna(record.get(col)):
            return "missing_nutrition"

    if record["calories"] < 30:
        return "calories_too_low"

    if record["calories"] > 1600:
        return "calories_too_high"

    if record["protein"] > 120:
        return "protein_outlier"

    if record["fat"] > 120:
        return "fat_outlier"

    if record["carbs"] > 180:
        return "carbs_outlier"

    weak_terms = [
        "sushi rice",
        "perfect sushi rice",
        "seasoned sushi vinegar",
        "sushi vinegar",
        "sushi ginger",
        "pickled sushi ginger",
        "gari",
        "sushi cake",
        "sushi donuts",
        "sushi donut",
        "sushi pizza",
        "sushi bowl",
        "california roll sushi bowl",
        "datemaki",
        "crepes",
        "tips",
        "guide",
        "how to",
        "vs"
    ]

    for term in weak_terms:
        if term in combined:
            return "weak_or_helper_sushi_record"

    direct_terms = [
        "sushi",
        "maki",
        "roll",
        "temaki",
        "futomaki",
        "uramaki",
        "hosomaki",
        "gunkan",
        "chirashi",
        "inari",
        "california roll",
        "spicy tuna",
        "shrimp tempura",
        "caterpillar"
    ]

    if not any(term in combined for term in direct_terms):
        return "not_direct_sushi"

    return ""


existing_urls_for_sushi = set(merged_text_calorie_df["source_url"].tolist())
existing_names_for_sushi = set(
    merged_text_calorie_df["recipe_name"].apply(normalize_text_for_filtering).tolist()
)

sushi_specialized_accepted_records = []
sushi_specialized_rejected_records = []
sushi_specialized_failed_urls = []

print("Strict sushi candidate links:")
print(len(strict_sushi_links_v2_df))

for idx, row in strict_sushi_links_v2_df.iterrows():
    source_name = row["source_name"]
    url = clean_candidate_url(row["url"])
    query = row["query"]

    print("\n====================================")
    print("Index:", idx)
    print("Source:", source_name)
    print("URL:", url)
    print("====================================")

    if url in existing_urls_for_sushi:
        sushi_specialized_rejected_records.append({
            "source_site": source_name,
            "food_class": "sushi",
            "query": query,
            "recipe_name": None,
            "ingredients_text": None,
            "calories": None,
            "protein": None,
            "fat": None,
            "carbs": None,
            "source_url": url,
            "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "drop_reason": "duplicate_url_existing_dataset"
        })
        print("Result: rejected duplicate URL")
        continue

    record = scrape_specialized_sushi_recipe(
        url=url,
        source_site=source_name,
        food_class="sushi",
        query=query
    )

    if record is None:
        sushi_specialized_failed_urls.append({
            "source_site": source_name,
            "query": query,
            "url": url,
            "reason": "scrape_failed_or_no_json_ld"
        })
        print("Result: failed")
        continue

    normalized_name = normalize_text_for_filtering(record.get("recipe_name"))

    if normalized_name in existing_names_for_sushi:
        rejected_record = record.copy()
        rejected_record["drop_reason"] = "duplicate_recipe_name_existing_dataset"
        sushi_specialized_rejected_records.append(rejected_record)
        print("Result: rejected duplicate recipe name")
        print("Recipe:", record.get("recipe_name"))
        continue

    drop_reason = sushi_specialized_reject_reason(record)

    if drop_reason != "":
        rejected_record = record.copy()
        rejected_record["drop_reason"] = drop_reason
        sushi_specialized_rejected_records.append(rejected_record)
        print("Result: rejected")
        print("Recipe:", record.get("recipe_name"))
        print("Reason:", drop_reason)
        continue

    sushi_specialized_accepted_records.append(record)
    existing_urls_for_sushi.add(url)
    existing_names_for_sushi.add(normalized_name)

    print("Result: accepted")
    print("Recipe:", record["recipe_name"])
    print("Calories:", record["calories"])
    print("Accepted so far:", len(sushi_specialized_accepted_records))

sushi_specialized_supplement_df = pd.DataFrame(sushi_specialized_accepted_records)
sushi_specialized_rejected_df = pd.DataFrame(sushi_specialized_rejected_records)
sushi_specialized_failed_df = pd.DataFrame(sushi_specialized_failed_urls)

print("\nSpecialized sushi scraping finished.")
print("Accepted sushi supplement shape:", sushi_specialized_supplement_df.shape)

if not sushi_specialized_supplement_df.empty:
    print("\nAccepted records per source:")
    print(sushi_specialized_supplement_df["source_site"].value_counts())

print("\nRejected sushi records:", len(sushi_specialized_rejected_df))
print("Failed sushi URLs:", len(sushi_specialized_failed_df))

if not sushi_specialized_supplement_df.empty:
    display(sushi_specialized_supplement_df[[
        "source_site",
        "food_class",
        "query",
        "recipe_name",
        "calories",
        "protein",
        "fat",
        "carbs",
        "source_url"
    ]])

if not sushi_specialized_rejected_df.empty:
    print("\nRejected examples:")
    display(sushi_specialized_rejected_df[[
        "source_site",
        "food_class",
        "query",
        "recipe_name",
        "calories",
        "protein",
        "fat",
        "carbs",
        "source_url",
        "drop_reason"
    ]].head(20))

if not sushi_specialized_failed_df.empty:
    print("\nFailed examples:")
    display(sushi_specialized_failed_df.head(20))

# Save outputs
sushi_specialized_supplement_path = SCRAPED_RAW_DIR / "specialized_sushi_supplement_raw.csv"
sushi_specialized_rejected_path = LOGS_DIR / "specialized_sushi_rejected_records.csv"
sushi_specialized_failed_path = LOGS_DIR / "specialized_sushi_failed_urls.csv"

sushi_specialized_supplement_df.to_csv(sushi_specialized_supplement_path, index=False)
sushi_specialized_rejected_df.to_csv(sushi_specialized_rejected_path, index=False)
sushi_specialized_failed_df.to_csv(sushi_specialized_failed_path, index=False)

print("\nSpecialized sushi supplement saved to:")
print(sushi_specialized_supplement_path)

print("\nSpecialized sushi rejected log saved to:")
print(sushi_specialized_rejected_path)

print("\nSpecialized sushi failed URL log saved to:")
print(sushi_specialized_failed_path)

Strict sushi candidate links:
31

Index: 0
Source: pickled_plum
URL: https://pickledplum.com/maki-sushi/
Result: accepted
Recipe: Maki Sushi 
Calories: 150.0
Accepted so far: 1

Index: 1
Source: pickled_plum
URL: https://pickledplum.com/temari-sushi/
Result: accepted
Recipe: Temari Sushi
Calories: 64.0
Accepted so far: 2

Index: 2
Source: pickled_plum
URL: https://pickledplum.com/inari-sushi-recipe/
Result: accepted
Recipe: Inari Sushi (Inarizushi - いなり寿司)
Calories: 81.0
Accepted so far: 3

Index: 3
Source: pickled_plum
URL: https://pickledplum.com/vegetarian-chirashi-sushi-recipe/
Result: accepted
Recipe: Vegetarian Chirashi Sushi
Calories: 604.0
Accepted so far: 4

Index: 4
Source: pickled_plum
URL: https://pickledplum.com/philadelphia-roll/
Result: accepted
Recipe: Philadelphia Roll
Calories: 575.0
Accepted so far: 5

Index: 5
Source: pickled_plum
URL: https://pickledplum.com/spicy-tuna-roll-recipe/
Result: accepted
Recipe: Spicy Tuna Roll (Poor Man's)
Calories: 106.0
Accepted so fa

,source_site,food_class,query,recipe_name,calories,protein,fat,carbs,source_url
0,pickled_plum,sushi,sushi,Maki Sushi,150.0,2.5,1.5,30.8,https://pickledplum.com/maki-sushi/
1,pickled_plum,sushi,sushi,Temari Sushi,64.0,2.7,0.3,12.2,https://pickledplum.com/temari-sushi/
2,pickled_plum,sushi,sushi,Inari Sushi (Inarizushi - いなり寿司),81.0,0.3,2.1,3.7,https://pickledplum.com/inari-sushi-recipe/
3,pickled_plum,sushi,sushi,Vegetarian Chirashi Sushi,604.0,11.3,17.4,96.4,https://pickledplum.com/vegetarian-chirashi-su...
4,pickled_plum,sushi,sushi,Philadelphia Roll,575.0,25.6,13.5,86.0,https://pickledplum.com/philadelphia-roll/
5,pickled_plum,sushi,sushi,Spicy Tuna Roll (Poor Man's),106.0,6.8,2.0,14.3,https://pickledplum.com/spicy-tuna-roll-recipe/
6,pickled_plum,sushi,sushi roll,California Roll,394.0,8.2,15.5,56.2,https://pickledplum.com/california-roll-spicy-...
7,chopstick_chronicles,sushi,sushi,Gunkan Sushi (Battleship Sushi),121.0,3.0,0.3,27.0,https://www.chopstickchronicles.com/gunkan-sus...
8,chopstick_chronicles,sushi,sushi,Chirashi Sushi ちらし寿司,430.0,7.0,3.0,92.0,https://www.chopstickchronicles.com/chirashi-s...
9,chopstick_chronicles,sushi,sushi,Unagi Sushi 鰻寿司,341.0,13.0,2.0,148.0,https://www.chopstickchronicles.com/unagi-sushi/



Rejected examples:


,source_site,food_class,query,recipe_name,calories,protein,fat,carbs,source_url,drop_reason
0,chopstick_chronicles,sushi,sushi,Temari Sushi 手毬寿司,189.0,6.0,3.0,33.0,https://www.chopstickchronicles.com/temari-sushi/,duplicate_recipe_name_existing_dataset
1,chopstick_chronicles,sushi,sushi,California Roll カリフォルニア巻き,416.0,9.0,10.0,71.0,https://www.chopstickchronicles.com/california...,duplicate_recipe_name_existing_dataset
2,chopstick_chronicles,sushi,sushi,Aunt Keiko's Inarisushi いなり寿司,200.0,4.0,NaN,42.0,https://www.chopstickchronicles.com/inari-sushi/,missing_nutrition



Failed examples:


,source_site,query,url,reason
0,chopstick_chronicles,sushi,https://www.chopstickchronicles.com/sushi/,scrape_failed_or_no_json_ld



Specialized sushi supplement saved to:
/content/drive/MyDrive/Calorify/phase2_text_calorie/data/scraped_raw/specialized_sushi_supplement_raw.csv

Specialized sushi rejected log saved to:
/content/drive/MyDrive/Calorify/phase2_text_calorie/logs/specialized_sushi_rejected_records.csv

Specialized sushi failed URL log saved to:
/content/drive/MyDrive/Calorify/phase2_text_calorie/logs/specialized_sushi_failed_urls.csv


In [34]:
# Cell 25I — Merge specialized sushi supplement and create cleaned dataset v2

print("Base merged dataset shape before sushi augmentation:")
print(merged_text_calorie_df.shape)

print("\nBase records per class:")
print(merged_text_calorie_df["food_class"].value_counts())

print("\nSpecialized sushi supplement shape:")
print(sushi_specialized_supplement_df.shape)

# Merge old clean candidate with new sushi supplement
merged_text_calorie_v2_df = pd.concat(
    [merged_text_calorie_df, sushi_specialized_supplement_df],
    ignore_index=True
)

# Safety normalization for duplicate detection
merged_text_calorie_v2_df["normalized_recipe_name"] = merged_text_calorie_v2_df["recipe_name"].apply(
    normalize_text_for_filtering
)

before_dedup = len(merged_text_calorie_v2_df)

merged_text_calorie_v2_df = merged_text_calorie_v2_df.drop_duplicates(
    subset=["source_url"],
    keep="first"
)

merged_text_calorie_v2_df = merged_text_calorie_v2_df.drop_duplicates(
    subset=["normalized_recipe_name"],
    keep="first"
)

after_dedup = len(merged_text_calorie_v2_df)

merged_text_calorie_v2_df = merged_text_calorie_v2_df.drop(
    columns=["normalized_recipe_name"],
    errors="ignore"
).reset_index(drop=True)

print("\nRows removed by safety dedup after sushi merge:")
print(before_dedup - after_dedup)

print("\nMerged v2 raw-clean shape:")
print(merged_text_calorie_v2_df.shape)

print("\nMerged v2 records per class:")
print(merged_text_calorie_v2_df["food_class"].value_counts())

print("\nMissing values after v2 merge:")
print(merged_text_calorie_v2_df.isna().sum())

print("\nDuplicate source URLs after v2 merge:")
print(merged_text_calorie_v2_df["source_url"].duplicated().sum())

print("\nDuplicate recipe names after v2 merge:")
print(merged_text_calorie_v2_df["recipe_name"].duplicated().sum())

print("\nNutrition summary after v2 merge:")
display(merged_text_calorie_v2_df[["calories", "protein", "fat", "carbs"]].describe())

print("\nCalories summary by class after v2 merge:")
display(
    merged_text_calorie_v2_df.groupby("food_class")["calories"]
    .agg(["count", "min", "max", "mean", "median"])
    .sort_values("count", ascending=False)
)

# Rebuild modeling-ready dataset v2
final_text_v2_df = merged_text_calorie_v2_df.copy()

final_text_v2_df["clean_recipe_name"] = final_text_v2_df["recipe_name"].apply(clean_text_for_modeling)
final_text_v2_df["clean_ingredients_text"] = final_text_v2_df["ingredients_text"].apply(clean_text_for_modeling)

final_text_v2_df["input_text"] = (
    "food class: "
    + final_text_v2_df["food_class"].astype(str)
    + " recipe name: "
    + final_text_v2_df["clean_recipe_name"]
    + " ingredients: "
    + final_text_v2_df["clean_ingredients_text"]
)

final_text_v2_df = final_text_v2_df.reset_index(drop=True)
final_text_v2_df.insert(0, "record_id", ["TXT_" + str(i).zfill(5) for i in range(len(final_text_v2_df))])

final_modeling_v2_df = final_text_v2_df[[
    "record_id",
    "source_site",
    "food_class",
    "query",
    "recipe_name",
    "ingredients_text",
    "clean_recipe_name",
    "clean_ingredients_text",
    "input_text",
    "calories",
    "protein",
    "fat",
    "carbs",
    "source_url",
    "scraped_at"
]].copy()

print("\nFinal modeling dataset v2 shape:")
print(final_modeling_v2_df.shape)

print("\nFinal modeling v2 records per class:")
print(final_modeling_v2_df["food_class"].value_counts())

print("\nMissing values in final modeling v2:")
print(final_modeling_v2_df.isna().sum())

print("\nDuplicate source URLs in final modeling v2:")
print(final_modeling_v2_df["source_url"].duplicated().sum())

print("\nDuplicate input_text in final modeling v2:")
print(final_modeling_v2_df["input_text"].duplicated().sum())

print("\nInput text length summary v2:")
final_modeling_v2_df["input_text_length"] = final_modeling_v2_df["input_text"].str.len()
print(final_modeling_v2_df["input_text_length"].describe())

print("\nSample sushi records after v2:")
display(
    final_modeling_v2_df[final_modeling_v2_df["food_class"] == "sushi"][[
        "record_id",
        "source_site",
        "food_class",
        "recipe_name",
        "calories",
        "protein",
        "fat",
        "carbs",
        "source_url"
    ]].head(20)
)

# Save v2 files
merged_text_calorie_v2_path = CLEAN_DIR / "text_calorie_merged_clean_candidate_v2.csv"
cleaned_text_calorie_dataset_v2_path = CLEAN_DIR / "cleaned_text_calorie_dataset_v2.csv"

merged_text_calorie_v2_df.to_csv(merged_text_calorie_v2_path, index=False)
final_modeling_v2_df.drop(columns=["input_text_length"], errors="ignore").to_csv(
    cleaned_text_calorie_dataset_v2_path,
    index=False
)

print("\nMerged clean candidate v2 saved to:")
print(merged_text_calorie_v2_path)

print("\nFinal cleaned text-calorie dataset v2 saved to:")
print(cleaned_text_calorie_dataset_v2_path)

Base merged dataset shape before sushi augmentation:
(416, 11)

Base records per class:
food_class
pizza       48
burger      47
rice        47
sandwich    46
pasta       45
salad       45
fries       44
chicken     41
steak       39
sushi       14
Name: count, dtype: int64

Specialized sushi supplement shape:
(27, 11)

Rows removed by safety dedup after sushi merge:
0

Merged v2 raw-clean shape:
(443, 11)

Merged v2 records per class:
food_class
pizza       48
burger      47
rice        47
sandwich    46
pasta       45
salad       45
fries       44
chicken     41
sushi       41
steak       39
Name: count, dtype: int64

Missing values after v2 merge:
source_site         0
food_class          0
query               0
recipe_name         0
ingredients_text    0
calories            0
protein             0
fat                 0
carbs               0
source_url          0
scraped_at          0
dtype: int64

Duplicate source URLs after v2 merge:
0

Duplicate recipe names after v2 merge:
0

Nu

,calories,protein,fat,carbs
count,443.000000,443.000000,443.000000,443.000000
mean,463.027088,25.609932,21.128646,41.330926
std,223.423556,13.923844,15.321915,24.960789
min,45.000000,0.300000,0.300000,0.400000
25%,303.500000,15.000000,10.000000,22.500000
50%,428.000000,25.200000,17.000000,39.000000
75%,580.500000,35.000000,28.000000,57.500000
max,1394.000000,69.000000,89.000000,148.000000



Calories summary by class after v2 merge:


,count,min,max,mean,median
food_class,,,,,
pizza,48,118.0,740.0,427.000000,427.0
burger,47,175.0,1080.0,526.936170,508.0
rice,47,164.0,834.0,470.404255,418.0
sandwich,46,125.0,1256.0,545.913043,470.0
salad,45,111.0,838.0,404.444444,374.0
pasta,45,371.0,1140.0,626.288889,600.0
fries,44,120.0,1394.0,403.590909,379.0
chicken,41,239.0,707.0,409.243902,368.0
sushi,41,45.0,862.0,313.487805,313.0



Final modeling dataset v2 shape:
(443, 15)

Final modeling v2 records per class:
food_class
pizza       48
burger      47
rice        47
sandwich    46
pasta       45
salad       45
fries       44
chicken     41
sushi       41
steak       39
Name: count, dtype: int64

Missing values in final modeling v2:
record_id                 0
source_site               0
food_class                0
query                     0
recipe_name               0
ingredients_text          0
clean_recipe_name         0
clean_ingredients_text    0
input_text                0
calories                  0
protein                   0
fat                       0
carbs                     0
source_url                0
scraped_at                0
dtype: int64

Duplicate source URLs in final modeling v2:
0

Duplicate input_text in final modeling v2:
0

Input text length summary v2:
count    443.000000
mean     379.318284
std      116.412554
min      109.000000
25%      298.500000
50%      372.000000
75%      439.500

,record_id,source_site,food_class,recipe_name,calories,protein,fat,carbs,source_url
32,TXT_00032,bbc_good_food,sushi,Build-your-own salmon sushi burrito,441.0,19.0,26.0,30.0,https://www.bbcgoodfood.com/recipes/build-your...
75,TXT_00075,bbc_good_food,sushi,Chicken onigiri,303.0,11.0,16.0,29.0,https://www.bbcgoodfood.com/recipes/chicken-on...
128,TXT_00128,bbc_good_food,sushi,Easy salmon sushi,216.0,12.0,2.0,41.0,https://www.bbcgoodfood.com/recipes/easy-salmo...
184,TXT_00184,bbc_good_food,sushi,Kelp and smoked salmon sushi rolls,277.0,12.0,12.0,29.0,https://www.bbcgoodfood.com/recipes/kelp-smoke...
274,TXT_00274,bbc_good_food,sushi,Quick sushi bowl,498.0,27.0,11.0,70.0,https://www.bbcgoodfood.com/recipes/quick-sush...
280,TXT_00280,bbc_good_food,sushi,Rice & quinoa prawn sushi bowl,439.0,22.0,10.0,61.0,https://www.bbcgoodfood.com/recipes/rice-quino...
288,TXT_00288,bbc_good_food,sushi,Salmon & cucumber sushi rolls,59.0,3.0,1.0,10.0,https://www.bbcgoodfood.com/recipes/salmon-cuc...
291,TXT_00291,bbc_good_food,sushi,Salmon sushi salad,862.0,27.0,43.0,87.0,https://www.bbcgoodfood.com/recipes/salmon-sus...
295,TXT_00295,bbc_good_food,sushi,Sesame & ginger sushi bowls,449.0,17.0,21.0,44.0,https://www.bbcgoodfood.com/recipes/sesame-gin...
299,TXT_00299,bbc_good_food,sushi,Simple sushi,390.0,8.0,9.0,70.0,https://www.bbcgoodfood.com/recipes/simple-sushi



Merged clean candidate v2 saved to:
/content/drive/MyDrive/Calorify/phase2_text_calorie/data/cleaned/text_calorie_merged_clean_candidate_v2.csv

Final cleaned text-calorie dataset v2 saved to:
/content/drive/MyDrive/Calorify/phase2_text_calorie/data/cleaned/cleaned_text_calorie_dataset_v2.csv


In [35]:
# Cell 25J — Train / validation / test split for cleaned dataset v2

from sklearn.model_selection import train_test_split

# Use the final v2 modeling dataset
split_v2_df = final_modeling_v2_df.drop(
    columns=["input_text_length"],
    errors="ignore"
).copy()

RANDOM_STATE = 42

# First split: 70% train, 30% temporary
train_v2_df, temp_v2_df = train_test_split(
    split_v2_df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=split_v2_df["food_class"]
)

# Second split: 15% validation, 15% test
val_v2_df, test_v2_df = train_test_split(
    temp_v2_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_v2_df["food_class"]
)

train_v2_df = train_v2_df.reset_index(drop=True)
val_v2_df = val_v2_df.reset_index(drop=True)
test_v2_df = test_v2_df.reset_index(drop=True)

print("Dataset v2 split finished.")

print("\nTrain v2 shape:")
print(train_v2_df.shape)

print("\nValidation v2 shape:")
print(val_v2_df.shape)

print("\nTest v2 shape:")
print(test_v2_df.shape)

print("\nTrain v2 class distribution:")
print(train_v2_df["food_class"].value_counts())

print("\nValidation v2 class distribution:")
print(val_v2_df["food_class"].value_counts())

print("\nTest v2 class distribution:")
print(test_v2_df["food_class"].value_counts())

# Leakage checks by source_url
train_urls = set(train_v2_df["source_url"])
val_urls = set(val_v2_df["source_url"])
test_urls = set(test_v2_df["source_url"])

print("\nURL overlap checks:")
print("Train-Val overlap:", len(train_urls.intersection(val_urls)))
print("Train-Test overlap:", len(train_urls.intersection(test_urls)))
print("Val-Test overlap:", len(val_urls.intersection(test_urls)))

# Leakage checks by record_id
train_ids = set(train_v2_df["record_id"])
val_ids = set(val_v2_df["record_id"])
test_ids = set(test_v2_df["record_id"])

print("\nRecord ID overlap checks:")
print("Train-Val overlap:", len(train_ids.intersection(val_ids)))
print("Train-Test overlap:", len(train_ids.intersection(test_ids)))
print("Val-Test overlap:", len(val_ids.intersection(test_ids)))

# Save split files
train_v2_path = CLEAN_DIR / "cleaned_text_calorie_train_v2.csv"
val_v2_path = CLEAN_DIR / "cleaned_text_calorie_val_v2.csv"
test_v2_path = CLEAN_DIR / "cleaned_text_calorie_test_v2.csv"

train_v2_df.to_csv(train_v2_path, index=False)
val_v2_df.to_csv(val_v2_path, index=False)
test_v2_df.to_csv(test_v2_path, index=False)

# Save split summary
split_v2_summary_df = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "records": [len(train_v2_df), len(val_v2_df), len(test_v2_df)]
})

split_v2_summary_path = CLEAN_DIR / "text_calorie_split_summary_v2.csv"
split_v2_summary_df.to_csv(split_v2_summary_path, index=False)

print("\nSaved train v2 split to:")
print(train_v2_path)

print("\nSaved validation v2 split to:")
print(val_v2_path)

print("\nSaved test v2 split to:")
print(test_v2_path)

print("\nSaved split v2 summary to:")
print(split_v2_summary_path)

Dataset v2 split finished.

Train v2 shape:
(310, 15)

Validation v2 shape:
(66, 15)

Test v2 shape:
(67, 15)

Train v2 class distribution:
food_class
pizza       34
rice        33
burger      33
sandwich    32
pasta       31
salad       31
fries       31
chicken     29
sushi       29
steak       27
Name: count, dtype: int64

Validation v2 class distribution:
food_class
pasta       7
rice        7
salad       7
burger      7
pizza       7
sandwich    7
fries       6
steak       6
sushi       6
chicken     6
Name: count, dtype: int64

Test v2 class distribution:
food_class
pizza       7
rice        7
sandwich    7
salad       7
fries       7
pasta       7
burger      7
sushi       6
chicken     6
steak       6
Name: count, dtype: int64

URL overlap checks:
Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0

Record ID overlap checks:
Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0

Saved train v2 split to:
/content/drive/MyDrive/Calorify/phase2_text_calorie/dat